# Etapa 1 — Padronização de nomes e preparação da base

## Finalidade

Esta etapa transforma a planilha bruta em uma base de trabalho previsível, sem modificar a cópia preservada da entrada. O objetivo não é decidir duplicidades, mas garantir que nomes, identificadores, datas e coordenadas possam ser comparados de forma consistente nas etapas seguintes.

## Entradas e premissas

- uma única planilha `.xlsx` em `01_bases`;
- cabeçalho localizado na linha esperada pelo processo de leitura;
- presença das colunas cadastrais definidas em `COL_NOME`, `COL_CANIE`, `COL_UF`, `COL_MUNICIPIO`, `COL_LONGITUDE`, `COL_LATITUDE`, `COL_DATA` e `COL_NIVEL`;
- execução dentro de `00_SCRIPT` ou em um diretório a partir do qual essa pasta possa ser localizada.

## Processamento

O notebook identifica os diretórios do projeto, lê a planilha, preserva `df_original` e cria `df` como cópia de trabalho. Em seguida, normaliza texto, gera chaves com diferentes níveis de redução, converte coordenadas e datas, extrai o nível numérico de validação e atribui `id_linha` a cada registro.

## Produtos e responsabilidade metodológica

O produto principal é `df`, enriquecido com colunas auxiliares. Essas colunas não substituem os valores originais: servem exclusivamente para comparação, agrupamento e auditoria. Valores ausentes são convertidos em texto vazio nas chaves, evitando que a string `"nan"` seja interpretada como coincidência real.

## Como ler a etapa

Os tópicos 1.1 a 1.5 preparam ambiente, caminhos e esquema. Os tópicos 1.6 a 1.10 definem funções de normalização. Os tópicos 1.11 a 1.16 aplicam essas funções e criam os campos usados pelas Etapas 2 e 3.

**Natureza desta célula.** Esta abertura é um mapa técnico da etapa; não executa código. A execução começa em 1.1.


## 1.1 — Importação das bibliotecas

Neste primeiro bloco são importadas as bibliotecas utilizadas na preparação inicial da base.

A biblioteca `pandas` é usada para leitura e manipulação tabular da planilha.
A biblioteca `numpy` é usada para representar valores nulos e apoiar operações numéricas.
A biblioteca `re` permite aplicar expressões regulares na limpeza de textos.
A biblioteca `unicodedata` é usada para remover acentos e sinais diacríticos.
A classe `Path`, do módulo `pathlib`, organiza os caminhos de arquivos e pastas de forma mais segura e legível.

Essas bibliotecas são a base para a leitura da planilha, padronização textual e criação dos campos auxiliares usados no restante do script.

**Contrato técnico com o código.** A execução disponibiliza `pd`, `np`, `re`, `unicodedata`, `Path` no ambiente do notebook. Nenhuma linha da base é transformada aqui; o efeito é somente tornar as dependências acessíveis às células seguintes.


In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path


## 1.2 — Definição dos caminhos e parâmetros iniciais

Neste bloco são definidos e validados os principais parâmetros de entrada e saída, procurando a pasta `00_SCRIPT` a partir do diretório atual e de seus diretórios ancestrais. Assim, o Jupyter pode ser iniciado em `00_SCRIPT` ou na raiz do projeto.

A variável `PASTA_ENTRADA` aponta para `01_bases`. O notebook procura nessa pasta exatamente uma planilha `.xlsx`, ignorando arquivos temporários do Excel. O arquivo encontrado é atribuído a `CAMINHO_ENTRADA`.
A variável `PASTA_SAIDA` aponta para `02_outputs`, onde serão armazenados todos os arquivos gerados pelo tratamento, como bases consolidadas, relatórios e produtos espaciais.

Se a estrutura de pastas estiver incorreta, se nenhuma planilha for encontrada ou se houver mais de uma planilha candidata, a execução é interrompida com uma mensagem explicativa.

Também é criado o seletor `USAR_PADRONIZACAO_NOMES`.

Quando esse seletor está como `True`, o script cria versões normalizadas dos nomes das cavidades, facilitando a identificação de possíveis duplicidades causadas por variações de grafia, acentuação, pontuação ou conectores.

Quando está como `False`, o script usa os nomes originais como base de comparação, sem aplicar limpeza textual agressiva.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


## 1.2.1 — Localização segura de `00_SCRIPT`

A função procura a pasta do notebook no diretório atual e nos ancestrais. Ela exige a presença de `01_bases`, evitando aceitar diretórios de mesmo nome sem a estrutura do projeto. Se nada for encontrado, interrompe o fluxo antes de qualquer leitura ou gravação.

**Contrato técnico com o código.** Esta célula define a função `localizar_pasta_script()`. Ela recebe nenhum argumento posicional e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def localizar_pasta_script():
    diretorio_atual = Path.cwd().resolve()

    for diretorio in [diretorio_atual, *diretorio_atual.parents]:
        if (
            diretorio.name == "00_SCRIPT"
            and (diretorio / "01_bases").is_dir()
        ):
            return diretorio

        pasta_script = diretorio / "00_SCRIPT"
        if (
            pasta_script.is_dir()
            and (pasta_script / "01_bases").is_dir()
        ):
            return pasta_script

    raise FileNotFoundError(
        "Não foi possível localizar a pasta 00_SCRIPT dentro da árvore do projeto. "
        "Abra o projeto ou inicie o Jupyter dentro da árvore do projeto."
    )


## 1.2.2 — Derivação dos diretórios do fluxo

A partir de `PASTA_SCRIPT`, são derivados `PASTA_ENTRADA` e `PASTA_SAIDA`. Essa relação garante que inputs e outputs permaneçam vinculados ao projeto independentemente do diretório usado para iniciar o Jupyter.

**Entradas e resultado.** `localizar_pasta_script()` fornece a raiz validada do fluxo. A partir dela, `01_bases` é reservada exclusivamente à planilha de entrada e `02_outputs` concentra os produtos gerados. Nenhum arquivo é lido ou escrito nesta célula; ela apenas estabelece os caminhos usados por todas as etapas posteriores.

**Relação com o fluxo.** Centralizar esses diretórios evita caminhos absolutos espalhados pelo notebook e torna as exportações das Etapas 8, 9 e 10 dependentes da mesma raiz identificada na Etapa 1.

**Contrato técnico com o código.** O bloco cria ou atualiza `PASTA_SCRIPT`, `PASTA_ENTRADA`, `PASTA_SAIDA`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
PASTA_SCRIPT = localizar_pasta_script()
PASTA_ENTRADA = PASTA_SCRIPT / "01_bases"
PASTA_SAIDA = PASTA_SCRIPT / "02_outputs"


## 1.2.3 — Descoberta e validação do arquivo de entrada

Arquivos temporários do Excel são ignorados. Exigir exatamente um `.xlsx` evita escolher silenciosamente uma versão antiga ou ambígua. Depois da validação, a pasta de saída é criada e os caminhos efetivos são exibidos.

**Como a validação funciona.** A busca considera somente arquivos com extensão `.xlsx` diretamente em `01_bases` e exclui nomes iniciados por `~$`, que representam arquivos de bloqueio temporário do Excel. Zero arquivos interrompe o processamento por falta de entrada; mais de um interrompe por ambiguidade. Assim, o notebook nunca escolhe uma base por ordem alfabética sem consentimento.

**Saídas desta célula.** `CAMINHO_ENTRADA` passa a apontar para a única planilha válida. `PASTA_SAIDA.mkdir(..., exist_ok=True)` garante a existência do diretório de produtos sem apagar seu conteúdo anterior. Os caminhos impressos servem como conferência antes da leitura da base.

**Contrato técnico com o código.** O bloco cria ou atualiza `arquivos_entrada`, `CAMINHO_ENTRADA`, `USAR_PADRONIZACAO_NOMES`, `nomes_encontrados`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
arquivos_entrada = sorted(
    caminho for caminho in PASTA_ENTRADA.glob("*.xlsx")
    if not caminho.name.startswith("~$")
)

if len(arquivos_entrada) != 1:
    nomes_encontrados = [caminho.name for caminho in arquivos_entrada]
    raise RuntimeError(
        "A pasta 01_bases deve conter exatamente uma planilha .xlsx. "
        f"Foram encontradas {len(arquivos_entrada)}: {nomes_encontrados}"
    )

CAMINHO_ENTRADA = arquivos_entrada[0]
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print(f"Planilha de entrada: {CAMINHO_ENTRADA}")
print(f"Pasta de saída: {PASTA_SAIDA}")

USAR_PADRONIZACAO_NOMES = True


## 1.3 — Leitura da planilha e preservação da base original

A planilha é lida com `header=8` porque as oito primeiras linhas contêm metadados ou informações institucionais, e o cabeçalho real da tabela começa na linha 9.

Após a leitura, o script cria duas cópias:

- `df_original`: cópia integral e preservada da planilha original;
- `df`: base de trabalho que receberá colunas auxiliares e transformações.

Essa separação é importante para auditoria. A base original permanece intacta, enquanto a base de trabalho pode ser modificada ao longo do processo.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_raw`, `df_original`, `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df_raw = pd.read_excel(CAMINHO_ENTRADA, header=8)

df_original = df_raw.copy(deep=True)

df = df_raw.copy(deep=True)


## 1.4 — Definição das colunas esperadas

Neste bloco são definidos os nomes das colunas essenciais da planilha.

Essas constantes evitam repetir textos longos ao longo do código e tornam o script mais fácil de manter. Se, no futuro, o nome de alguma coluna mudar na planilha de entrada, basta ajustar a constante correspondente nesta seção.

As colunas esperadas são:

- número CANIE;
- nome da cavidade;
- UF;
- município;
- longitude;
- latitude;
- nível de validação;
- data de cadastro.

Esses campos são necessários porque sustentam as principais dimensões de análise do script: identificação cadastral, comparação textual, localização espacial, validação e temporalidade.

**Contrato técnico com o código.** O bloco cria ou atualiza `COL_CANIE`, `COL_NOME`, `COL_UF`, `COL_MUNICIPIO`, `COL_LON`, `COL_LAT` e outros objetos auxiliares. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
COL_CANIE = "Número CANIE"
COL_NOME = "Nome"
COL_UF = "UF"
COL_MUNICIPIO = "Municipio"
COL_LON = "Longitude"
COL_LAT = "Latitude"
COL_NIVEL = "Nivel de validação"
COL_DATA = "Data de cadastro"


## 1.5 — Validação mínima da estrutura da planilha

Antes de continuar o processamento, o script verifica se todas as colunas obrigatórias existem na planilha.

Essa validação evita que o código avance com uma estrutura incompleta e gere erros difíceis de interpretar em etapas posteriores.

Se alguma coluna obrigatória estiver ausente, o script interrompe a execução e informa exatamente quais colunas não foram encontradas.

**Contrato técnico com o código.** O bloco cria ou atualiza `colunas_necessarias`, `faltantes`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
colunas_necessarias = [
    COL_CANIE,
    COL_NOME,
    COL_UF,
    COL_MUNICIPIO,
    COL_LON,
    COL_LAT,
    COL_NIVEL,
    COL_DATA
]

faltantes = [
    coluna for coluna in colunas_necessarias
    if coluna not in df.columns
]

if faltantes:
    raise ValueError(
        f"As seguintes colunas obrigatórias não foram encontradas na planilha: {faltantes}"
    )


## 1.6 — Função para remoção de acentos

A função `remover_acentos` elimina acentos e sinais diacríticos de textos.

Isso é necessário porque nomes de cavidades podem aparecer com pequenas variações de grafia, como:

- `Onça`;
- `Onca`.

Sem essa padronização, o script poderia interpretar nomes equivalentes como textos diferentes. A remoção de acentos melhora a comparação textual sem alterar o conteúdo semântico principal do nome.

**Contrato técnico com o código.** Esta célula define a função `remover_acentos(texto)`. Ela recebe `texto` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def remover_acentos(texto):
    """
    Remove acentos e sinais diacríticos de uma string.

    Essa função é usada para evitar que diferenças como
    'Onça' e 'Onca' impeçam a comparação entre nomes.
    """
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )

    return texto


## 1.7 — Função de limpeza textual básica

A função `limpar_texto_basico` aplica uma padronização inicial aos textos.

Ela executa cinco operações principais:

1. trata valores nulos como texto vazio;
2. remove acentos;
3. converte o texto para maiúsculas;
4. remove espaços no início e no fim;
5. substitui múltiplos espaços internos por um único espaço.

Essa função é usada como base para outras funções de padronização. Ela garante que comparações entre nomes, municípios e códigos sejam menos sensíveis a diferenças superficiais de escrita.

**Contrato técnico com o código.** Esta célula define a função `limpar_texto_basico(texto)`. Ela recebe `texto` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def limpar_texto_basico(texto):
    """
    Aplica limpeza textual básica:
    - converte para string;
    - remove acentos;
    - converte para maiúsculas;
    - remove espaços excedentes no início e no fim;
    - substitui múltiplos espaços internos por espaço simples.
    """
    if pd.isna(texto):
        return ""

    texto = remover_acentos(texto)
    texto = texto.upper()
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)

    return texto


## 1.8 — Função para padronização moderada do nome

A função `padronizar_nome` cria uma versão normalizada do nome da cavidade para comparação textual moderada.

Ela remove acentos, converte para maiúsculas, elimina separadores comuns em códigos e remove símbolos residuais. No entanto, preserva letras, números e espaços.

Essa chave é usada para identificar nomes que são praticamente iguais, mas que aparecem com diferenças pequenas de grafia ou pontuação.

Exemplos:

- `FEA-009` vira `FEA009`;
- `Gruta da Onça` vira `GRUTA DA ONCA`.

Essa versão ainda preserva boa parte da estrutura original do nome, por isso é considerada uma padronização moderada.

**Contrato técnico com o código.** Esta célula define a função `padronizar_nome(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def padronizar_nome(nome):
    """
    Cria uma versão padronizada do nome para comparação moderada.

    Tratamentos aplicados:
    - remove acentos;
    - converte para maiúsculas;
    - remove espaços duplicados;
    - remove pontuações frequentes em códigos;
    - preserva letras, números e espaços.

    Exemplo:
    'FEA-009'       -> 'FEA009'
    'Gruta da Onça' -> 'GRUTA DA ONCA'
    """
    nome = limpar_texto_basico(nome)

    # Remove separadores comuns em códigos ou nomes técnicos.
    nome = re.sub(r"[-_/\\.]", "", nome)

    # Remove símbolos residuais, preservando letras, números e espaços.
    nome = re.sub(r"[^A-Z0-9 ]", "", nome)

    # Normaliza novamente os espaços internos.
    nome = re.sub(r"\s+", " ", nome).strip()

    return nome


## 1.9 — Função para geração do nome-base

A função `gerar_nome_base` cria uma chave textual mais agressiva.

Ela remove termos genéricos e morfológicos comuns em nomes de cavidades, como:

- `GRUTA`;
- `CAVERNA`;
- `TOCA`;
- `LAPA`;
- `ABRIGO`;
- `FURNA`;
- `BURACO`;
- `SUMIDOURO`;
- `DOLINA`.

Também remove conectores como:

- `DA`;
- `DE`;
- `DO`;
- `DAS`;
- `DOS`;
- `A`;
- `O`;
- `AS`;
- `OS`.

O objetivo é comparar a parte mais distintiva do nome.

Por exemplo, nomes como `Gruta da Onça`, `Caverna Onça` e `Lapa da Onça` podem compartilhar o mesmo nome-base `ONCA`.

Essa chave não deve ser usada isoladamente para remover registros. Ela serve apenas para levantar suspeitas que serão avaliadas junto com município, localização, CANIE, data de cadastro e outras regras.

**Contrato técnico com o código.** Esta célula define a função `gerar_nome_base(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def gerar_nome_base(nome):
    """
    Cria uma chave textual mais agressiva para comparação auxiliar.

    Essa função remove termos genéricos comuns em nomes de cavidades,
    como 'GRUTA', 'CAVERNA', 'TOCA', 'LAPA' e conectores como 'DA', 'DE', 'DO'.

    Atenção:
    essa chave não deve ser usada isoladamente para excluir registros.
    Ela serve para encontrar casos suspeitos que devem ser avaliados
    em conjunto com localização, município, CANIE e data.
    """
    nome = limpar_texto_basico(nome)

    nome = re.sub(r"[-_/\\.]", " ", nome)
    nome = re.sub(r"[^A-Z0-9 ]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip()

    termos_genericos = {
        "GRUTA", "CAVERNA", "TOCA", "LAPA", "ABRIGO",
        "FURNA", "BURACO", "SUMIDOURO", "DOLINA",
        "DA", "DE", "DO", "DAS", "DOS", "A", "O", "AS", "OS"
    }

    palavras = nome.split()
    palavras = [
        palavra for palavra in palavras
        if palavra not in termos_genericos
    ]

    return " ".join(palavras)


## 1.10 — Função para geração de código textual compacto

A função `gerar_nome_codigo` cria uma chave compacta para nomes que se comportam como códigos técnicos.

Ela remove qualquer caractere que não seja letra ou número. Isso permite comparar códigos que podem aparecer com hífen, ponto, espaço ou outros separadores.

Exemplos:

- `FEA-009` vira `FEA009`;
- `S2-SF-078` vira `S2SF078`;
- `CAV. 01` vira `CAV01`.

Essa chave é especialmente útil para detectar duplicidades em registros cujo nome é, na prática, um código técnico de campo.

**Contrato técnico com o código.** Esta célula define a função `gerar_nome_codigo(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def gerar_nome_codigo(nome):
    """
    Cria uma chave compacta para nomes que se comportam como códigos técnicos.

    Tratamentos aplicados:
    - remove acentos;
    - converte para maiúsculas;
    - remove qualquer caractere que não seja letra ou número.

    Exemplo:
    'FEA-009'   -> 'FEA009'
    'S2-SF-078' -> 'S2SF078'
    'CAV. 01'   -> 'CAV01'
    """
    nome = limpar_texto_basico(nome)
    nome = re.sub(r"[^A-Z0-9]", "", nome)

    return nome


## 1.11 — Preservação do nome original e criação das chaves de comparação

Neste bloco, o script preserva explicitamente o nome original da cavidade na coluna `nome_original_preservado`.

Depois, conforme o valor de `USAR_PADRONIZACAO_NOMES`, são criadas três chaves auxiliares:

- `nome_comparacao`: nome padronizado de forma moderada;
- `nome_base`: nome reduzido à parte mais distintiva;
- `nome_codigo`: versão compacta voltada a códigos técnicos.

Se `USAR_PADRONIZACAO_NOMES` estiver como `False`, essas três colunas recebem apenas o nome original convertido para texto.

Essas chaves serão usadas nas etapas seguintes para gerar pares candidatos e classificar suspeitas de duplicidade.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`, `nome_sem_nulos`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["nome_original_preservado"] = df[COL_NOME]

if USAR_PADRONIZACAO_NOMES:
    df["nome_comparacao"] = df[COL_NOME].apply(padronizar_nome)
    df["nome_base"] = df[COL_NOME].apply(gerar_nome_base)
    df["nome_codigo"] = df[COL_NOME].apply(gerar_nome_codigo)
else:
    nome_sem_nulos = (
        df[COL_NOME]
        .astype("string")
        .fillna("")
        .str.strip()
    )
    df["nome_comparacao"] = nome_sem_nulos
    df["nome_base"] = nome_sem_nulos
    df["nome_codigo"] = nome_sem_nulos


## 1.12 — Padronização de campos cadastrais e administrativos

Neste bloco são criadas versões auxiliares de campos importantes para agrupamentos e comparações.

A coluna `canie_txt` armazena o número CANIE como texto, removendo espaços excedentes.
A coluna `uf_txt` padroniza a UF como texto em maiúsculas.
A coluna `municipio_txt` aplica a limpeza textual básica ao nome do município.

Essas padronizações evitam que diferenças simples de caixa, acentuação ou espaços prejudiquem a comparação entre registros.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["canie_txt"] = (
    df[COL_CANIE].astype("string").fillna("").str.strip()
)
df["uf_txt"] = (
    df[COL_UF].astype("string").fillna("").str.strip().str.upper()
)
df["municipio_txt"] = df[COL_MUNICIPIO].apply(limpar_texto_basico)


## 1.13 — Conversão das coordenadas para valores numéricos

As colunas originais de longitude e latitude são convertidas para valores numéricos.

A opção `errors="coerce"` transforma valores inválidos em `NaN`, permitindo que o script continue sem quebrar. Esses registros poderão ser identificados posteriormente como registros sem coordenada válida.

As colunas criadas são:

- `longitude_num`;
- `latitude_num`.

Esses campos serão usados na Etapa 2 para criar geometrias e calcular distâncias entre registros.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["longitude_num"] = pd.to_numeric(df[COL_LON], errors="coerce")
df["latitude_num"] = pd.to_numeric(df[COL_LAT], errors="coerce")


## 1.14 — Conversão da data de cadastro

A coluna de data de cadastro é convertida para formato datetime.

O parâmetro `dayfirst=True` indica que o dia aparece antes do mês, padrão comum em bases brasileiras.

Valores inválidos são convertidos para `NaT`, permitindo que o script lide com datas ausentes ou malformadas sem interromper a execução.

A coluna criada é:

- `data_cadastro_dt`.

Essa data será usada posteriormente para diferenciar possíveis cargas duplicadas, recadastros, registros antigos e registros recentes.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["data_cadastro_dt"] = pd.to_datetime(
    df[COL_DATA],
    errors="coerce",
    dayfirst=True
)


## 1.15 — Extração do nível numérico de validação

O campo original de nível de validação pode conter texto, como `Nível 3`.

Para permitir ordenação e escolha do registro principal nas etapas posteriores, o script extrai apenas o número do campo de validação.

A coluna criada é:

- `nivel_validacao_num`.

Esse campo será usado como um dos critérios para decidir qual registro deve ser mantido como principal dentro de um grupo de duplicidade.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["nivel_validacao_num"] = (
    df[COL_NIVEL]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)


## 1.16 — Criação do identificador interno da linha

A coluna `id_linha` cria um identificador sequencial interno para cada registro da base.

Esse identificador é fundamental porque as etapas seguintes trabalham com pares de registros e grupos de duplicidade. Em vez de depender apenas do número CANIE, que pode estar repetido ou inconsistente, o script usa `id_linha` como referência estável para cada linha da base de trabalho.

Essa coluna não representa um identificador oficial do CANIE. Ela é apenas um identificador operacional criado para o processamento.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["id_linha"] = range(1, len(df) + 1)


## 1.17 — Mensagens finais da Etapa 1

Ao final da etapa, o script imprime três informações:

- confirmação de conclusão da Etapa 1;
- quantidade de registros lidos;
- indicação de que a padronização de nomes está ativada ou desativada.

Essas mensagens funcionam como uma verificação rápida de que a leitura da base foi concluída corretamente e de que o parâmetro principal de padronização está configurado conforme esperado.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 1 concluída.")
print(f"Registros lidos: {len(df):,}")
print(f"Padronização de nomes ativada: {USAR_PADRONIZACAO_NOMES}")


## Síntese da Etapa 1

### Resultado produzido

`df_original` preserva a leitura da planilha, enquanto `df` contém as colunas auxiliares necessárias ao tratamento. Entre elas estão nomes normalizados, nome-base, código compacto, campos cadastrais seguros, coordenadas numéricas, data convertida, nível de validação e `id_linha`.

### Verificações esperadas

- `len(df)` deve ser igual a `len(df_original)`;
- `id_linha` deve ser único e preencher todas as linhas;
- ausências textuais não devem aparecer como `"nan"` nas chaves;
- a execução deve informar corretamente o arquivo de entrada e a pasta de saída.

### Ligação com a etapa seguinte

A Etapa 2 utiliza essas chaves e coordenadas para gerar pares candidatos. Nenhuma duplicidade foi classificada ou removida nesta etapa.

**Natureza desta célula.** Esta síntese é uma conferência conceitual e não executa código.


# Etapa 2 — Geração dos pares candidatos e cálculo das distâncias

## Finalidade

Comparar todos os registros entre si teria custo quadrático e produziria milhões de combinações pouco úteis. Esta etapa reduz o universo de comparação por meio de chaves cadastrais, textuais e espaciais e calcula a distância precisa somente para os pares plausíveis.

## Entradas

O processo recebe `df` da Etapa 1, incluindo `id_linha`, chaves textuais, CANIE padronizado e coordenadas numéricas. As coordenadas originais são interpretadas em SIRGAS 2000 geográfico (`EPSG:4674`) e reprojetadas para SIRGAS 2000 / Brazil Polyconic (`EPSG:5880`) para que distâncias euclidianas sejam expressas em metros.

## Processamento

São criadas geometrias pontuais, coordenadas métricas e chaves espaciais aproximadas de 1 m, 10 m e 50 m. A função de geração de candidatos combina registros que compartilham CANIE, nome, nome-base, código técnico ou chave espacial. Chaves textuais vazias são descartadas antes do agrupamento. Um conjunto de pares impede que a mesma combinação seja incluída repetidamente por motivos diferentes.

## Produto

`df_pares` contém os identificadores A e B, o motivo de candidatura, os atributos completos dos dois registros e `distancia_metros`. A etapa identifica relações que merecem avaliação; ela ainda não afirma que os registros são duplicados.

## Como ler a etapa

Os tópicos 2.1 a 2.10 preparam o referencial espacial. Os tópicos 2.11 a 2.20 geram candidatos. Os tópicos 2.21 a 2.23 calculam e conferem a distância definitiva.

**Natureza desta célula.** Esta abertura apresenta a arquitetura da etapa e não contém código executável. A execução começa em 2.1.


## 2.1 — Importação das bibliotecas espaciais

Neste bloco são importadas as bibliotecas necessárias para a etapa espacial.

A biblioteca `geopandas` permite trabalhar com dados geográficos em formato tabular, combinando a estrutura de um DataFrame com geometrias.

A classe `Point`, da biblioteca `shapely`, é usada para transformar longitude e latitude em pontos geográficos.

A função `combinations`, do módulo `itertools`, é usada para gerar combinações de pares dentro de grupos de registros semelhantes.

Essas bibliotecas permitem transformar a base tabular em uma estrutura espacial e criar os pares que serão avaliados quanto à distância.

**Contrato técnico com o código.** A execução disponibiliza `gpd`, `Point`, `combinations` no ambiente do notebook. Nenhuma linha da base é transformada aqui; o efeito é somente tornar as dependências acessíveis às células seguintes.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point
from itertools import combinations


## 2.2 — Definição dos sistemas de referência espacial

Nesta etapa são definidos dois sistemas de referência espacial.

O primeiro é o sistema geográfico de entrada:

`EPSG:4674`

Esse código corresponde ao SIRGAS 2000 em coordenadas geográficas, usado para representar longitude e latitude em graus decimais.

O segundo é o sistema métrico usado para cálculo de distância:

`EPSG:5880`

Esse código corresponde ao SIRGAS 2000 / Brazil Polyconic. Ele permite calcular distâncias em metros, o que é necessário para avaliar a proximidade entre registros.

A lógica é:

- usar `EPSG:4674` para representar corretamente as coordenadas originais;
- reprojetar para `EPSG:5880` antes de calcular distâncias.

**Contrato técnico com o código.** O bloco cria ou atualiza `CRS_GEOGRAFICO`, `CRS_METRICO`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
CRS_GEOGRAFICO = "EPSG:4674"

CRS_METRICO = "EPSG:5880"


## 2.3 — Criação de uma cópia da base para processamento espacial

Antes de gerar as geometrias, o script cria uma cópia da base de trabalho chamada `df_geo`.

Essa cópia é usada para montar o GeoDataFrame, preservando a estrutura principal da base `df`.

A separação ajuda a manter o processamento espacial organizado. Depois que as coordenadas métricas forem calculadas, os campos necessários serão devolvidos para `df`.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_geo`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df_geo = df.copy()


## 2.4 — Criação das geometrias pontuais

Neste bloco, cada registro com longitude e latitude válidas é transformado em um ponto geográfico.

A geometria é criada a partir da ordem:

`Point(longitude, latitude)`

Essa ordem é importante porque objetos espaciais usam o eixo X primeiro e o eixo Y depois. Em coordenadas geográficas, isso corresponde a longitude e latitude.

Registros sem longitude ou latitude válida recebem geometria nula. Eles continuam na base, mas não poderão ser usados no cálculo espacial de distância.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_geo`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df_geo["geometry"] = df_geo.apply(
    lambda row: Point(row["longitude_num"], row["latitude_num"])
    if pd.notna(row["longitude_num"]) and pd.notna(row["latitude_num"])
    else None,
    axis=1
)


## 2.5 — Criação do GeoDataFrame em coordenadas geográficas

Após criar a coluna `geometry`, o script transforma `df_geo` em um GeoDataFrame chamado `gdf`.

Esse GeoDataFrame recebe o sistema de referência `EPSG:4674`, pois as coordenadas originais estão em longitude e latitude.

Essa etapa informa ao Python como interpretar espacialmente os pontos criados. Sem essa definição de CRS, a reprojeção posterior para metros não seria confiável.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf = gpd.GeoDataFrame(
    df_geo,
    geometry="geometry",
    crs=CRS_GEOGRAFICO
)


## 2.6 — Identificação de registros com geometria válida

Neste bloco é criada a coluna `geometria_valida`.

Ela indica se cada registro possui uma geometria pontual válida.

Essa informação será usada posteriormente para avaliar a confiabilidade espacial dos registros e também para escolher o registro principal em grupos de duplicidade.

Registros com geometria nula não são descartados nesta etapa. Eles apenas são marcados como sem geometria válida.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf["geometria_valida"] = gdf.geometry.notna()


## 2.7 — Reprojeção dos pontos para sistema métrico

As coordenadas originais estão em graus decimais. Por isso, não é adequado calcular distância diretamente nelas.

Neste bloco, os pontos são reprojetados para `EPSG:5880`, um sistema projetado em metros.

Após essa reprojeção, passa a ser possível calcular distâncias lineares entre pontos usando valores métricos.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf_metrico`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf_metrico = gdf.to_crs(CRS_METRICO)


## 2.8 — Extração das coordenadas métricas

Após a reprojeção para `EPSG:5880`, o script extrai as coordenadas projetadas de cada ponto.

São criadas duas colunas:

- `x_m`: coordenada X em metros;
- `y_m`: coordenada Y em metros.

Essas coordenadas serão usadas para calcular a distância euclidiana entre registros e para criar chaves espaciais aproximadas.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf_metrico`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf_metrico["x_m"] = gdf_metrico.geometry.x
gdf_metrico["y_m"] = gdf_metrico.geometry.y


## 2.9 — Retorno das informações espaciais para a base de trabalho

Neste bloco, as coordenadas métricas e a informação de geometria válida são copiadas de volta para a base principal `df`.

A partir deste ponto, `df` passa a conter:

- `x_m`;
- `y_m`;
- `geometria_valida`.

Esses campos serão usados tanto nesta etapa quanto nas etapas seguintes, especialmente na formação de pares candidatos, no cálculo de distância e na escolha do registro principal.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["x_m"] = gdf_metrico["x_m"]
df["y_m"] = gdf_metrico["y_m"]
df["geometria_valida"] = gdf_metrico["geometria_valida"]


## 2.10 — Criação de chaves espaciais aproximadas

Neste bloco são criadas chaves espaciais simplificadas a partir das coordenadas métricas.

As chaves são:

- `coord_m_1m`;
- `coord_m_10m`;
- `coord_m_50m`.

A chave de 1 metro arredonda diretamente `x_m` e `y_m`.

A chave de 10 metros divide as coordenadas por 10, arredonda e cria uma identificação aproximada de célula espacial.

A chave de 50 metros faz o mesmo usando células aproximadas de 50 metros.

Essas chaves não são usadas para calcular a distância final. Elas servem apenas para gerar pares candidatos de forma eficiente, evitando comparar todos os registros contra todos os demais.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


**Tratamento de coordenadas ausentes.** A chave só é formada quando `x_m` e `y_m` são válidos. Se qualquer componente estiver ausente, o valor gravado é `""`; por isso, registros sem geometria não podem formar grupos artificiais pela chave `"nan_nan"`.


In [ ]:
coordenadas_metricas_validas = df["x_m"].notna() & df["y_m"].notna()

df["coord_m_1m"] = np.where(
    coordenadas_metricas_validas,
    df["x_m"].round(0).astype(str)
    + "_"
    + df["y_m"].round(0).astype(str),
    ""
)

df["coord_m_10m"] = np.where(
    coordenadas_metricas_validas,
    (df["x_m"] / 10).round(0).astype(str)
    + "_"
    + (df["y_m"] / 10).round(0).astype(str),
    ""
)

df["coord_m_50m"] = np.where(
    coordenadas_metricas_validas,
    (df["x_m"] / 50).round(0).astype(str)
    + "_"
    + (df["y_m"] / 50).round(0).astype(str),
    ""
)


## 2.11 — Inicialização das estruturas de pares candidatos

Neste bloco são criadas duas estruturas auxiliares:

- `pares`: lista que armazenará os pares candidatos encontrados;
- `pares_set`: conjunto usado para evitar repetição de pares.

O uso de `pares_set` é importante porque o mesmo par pode ser encontrado por mais de uma regra.

Por exemplo, dois registros podem ter o mesmo nome e também a mesma coordenada aproximada. Nesse caso, o par deve ser registrado apenas uma vez.

**Contrato técnico com o código.** O bloco cria ou atualiza `pares`, `pares_set`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
pares = []

pares_set = set()


## 2.12 — Função para gerar pares candidatos por chave

A função `adicionar_pares_por_chave` agrupa registros que compartilham uma ou mais chaves e gera combinações de pares dentro de cada grupo.

Ela recebe três parâmetros:

- `chaves`: lista de colunas usadas para agrupar registros;
- `motivo`: texto que registra por qual regra o par foi criado;
- `max_grupo`: limite máximo de registros dentro de um grupo.

O parâmetro `max_grupo` evita explosão combinatória. Se um grupo tiver registros demais, o script não gera todos os pares possíveis dentro dele, pois isso poderia criar milhões de combinações e travar o processamento.

A função usa o campo `id_linha`, criado na Etapa 1, para identificar cada registro de forma estável.

**Contrato técnico com o código.** Esta célula define a função `adicionar_pares_por_chave(chaves, motivo, max_grupo)`. Ela recebe `chaves`, `motivo`, `max_grupo` e atua por efeito sobre os objetos manipulados. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_pares_por_chave(chaves, motivo, max_grupo=300):
    """
    Gera pares candidatos dentro de grupos definidos por uma ou mais chaves.

    Parâmetros:
    chaves: lista de colunas usadas para agrupar registros semelhantes.
    motivo: descrição da regra que gerou o par candidato.
    max_grupo: limite máximo de registros por grupo para evitar explosão combinatória.

    Exemplo:
    se três registros compartilham o mesmo nome e município,
    serão gerados os pares A-B, A-C e B-C.
    """
    df_validos = df.copy()

    for chave in chaves:
        valores = df_validos[chave].astype("string").str.strip()
        df_validos = df_validos[valores.notna() & valores.ne("")]

    grupos = df_validos.groupby(chaves, dropna=False).groups

    for _, indices in grupos.items():
        indices = list(indices)

        if len(indices) < 2:
            continue

        if len(indices) > max_grupo:
            continue

        for i, j in combinations(indices, 2):
            id_a = int(df.loc[i, "id_linha"])
            id_b = int(df.loc[j, "id_linha"])

            par = tuple(sorted((id_a, id_b)))

            if par not in pares_set:
                pares_set.add(par)
                pares.append({
                    "id_linha_a": par[0],
                    "id_linha_b": par[1],
                    "motivo_candidatura": motivo
                })


## 2.13 — Geração de pares por mesmo número CANIE

Neste bloco, o script gera pares candidatos entre registros que possuem o mesmo número CANIE.

Esse é um critério forte de suspeita, porque o número CANIE deveria identificar uma cavidade. Se dois registros diferentes compartilham o mesmo número, eles precisam ser avaliados.

O limite `max_grupo=1000` é maior que nos demais critérios porque repetições de CANIE são especialmente relevantes para auditoria.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["canie_txt"],
    motivo="mesmo_numero_canie",
    max_grupo=1000
)


## 2.14 — Geração de pares por mesmo nome de comparação no mesmo município

Neste bloco, são gerados pares entre registros que compartilham:

- mesma UF;
- mesmo município padronizado;
- mesmo `nome_comparacao`.

Esse critério identifica cavidades com nomes praticamente iguais dentro do mesmo município.

Como `nome_comparacao` já passou por uma padronização moderada, pequenas diferenças de acentuação, caixa ou pontuação deixam de impedir a comparação.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["uf_txt", "municipio_txt", "nome_comparacao"],
    motivo="mesmo_nome_comparacao_mesmo_municipio",
    max_grupo=300
)


## 2.15 — Geração de pares por mesmo nome-base no mesmo município

Neste bloco, são gerados pares entre registros com:

- mesma UF;
- mesmo município padronizado;
- mesmo `nome_base`.

O `nome_base` é uma chave textual mais agressiva, que remove termos morfológicos e conectores.

Esse critério permite identificar registros que podem se referir à mesma cavidade mesmo quando o tipo inicial varia, por exemplo:

- `Gruta da Onça`;
- `Caverna Onça`;
- `Lapa da Onça`.

No entanto, essa regra apenas gera candidatos. A decisão sobre duplicidade será tomada posteriormente, combinando nome, distância, tipo morfológico, data e outros critérios.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["uf_txt", "municipio_txt", "nome_base"],
    motivo="mesmo_nome_base_mesmo_municipio",
    max_grupo=300
)


## 2.16 — Geração de pares por mesmo código técnico no mesmo município

Neste bloco, são gerados pares entre registros com:

- mesma UF;
- mesmo município;
- mesmo `nome_codigo`.

O campo `nome_codigo` remove todos os caracteres que não sejam letras ou números. Ele é útil para nomes que funcionam como códigos técnicos de campo.

Exemplos de equivalência:

- `FEA-009` e `FEA009`;
- `S2-SF-078` e `S2SF078`;
- `CAV. 01` e `CAV01`.

Essa regra ajuda a identificar registros que podem ter o mesmo código técnico escrito de formas diferentes.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["uf_txt", "municipio_txt", "nome_codigo"],
    motivo="mesmo_nome_codigo_mesmo_municipio",
    max_grupo=300
)


## 2.17 — Geração de pares por coordenada métrica aproximada a 1 metro

Neste bloco, são gerados pares entre registros localizados na mesma célula espacial aproximada de 1 metro.

Esse critério captura registros com coordenadas praticamente idênticas.

Ele é útil para detectar possíveis duplicidades espaciais muito fortes, principalmente quando o nome também apresenta alguma relação.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["uf_txt", "municipio_txt", "coord_m_1m"],
    motivo="coordenada_metrica_1m",
    max_grupo=300
)


## 2.18 — Geração de pares por coordenada métrica aproximada a 10 metros

Neste bloco, são gerados pares entre registros localizados na mesma célula espacial aproximada de 10 metros.

Esse critério amplia a busca espacial em relação à chave de 1 metro, permitindo identificar registros muito próximos, ainda que não estejam exatamente na mesma coordenada.

A decisão final não será baseada apenas nesta chave. A distância exata será calculada posteriormente.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["uf_txt", "municipio_txt", "coord_m_10m"],
    motivo="coordenada_metrica_10m",
    max_grupo=300
)


## 2.19 — Geração de pares por coordenada métrica aproximada a 50 metros

Neste bloco, são gerados pares entre registros localizados na mesma célula espacial aproximada de 50 metros.

Esse critério é mais amplo e serve para capturar registros próximos que poderiam não ser encontrados pelas chaves de 1 m ou 10 m.

Ele ajuda a localizar possíveis recadastros, variações de posicionamento ou registros próximos com nomes semelhantes.

Assim como nas demais chaves espaciais, essa regra apenas gera candidatos. A distância definitiva será calculada em seguida.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
adicionar_pares_por_chave(
    chaves=["uf_txt", "municipio_txt", "coord_m_50m"],
    motivo="coordenada_metrica_50m",
    max_grupo=300
)


## 2.20 — Conversão da lista de pares em DataFrame

Depois que todos os critérios de candidatura são aplicados, a lista `pares` é convertida em um DataFrame chamado `df_pares`.

Cada linha de `df_pares` representa um par candidato.

Inicialmente, essa tabela contém:

- `id_linha_a`;
- `id_linha_b`;
- `motivo_candidatura`.

Nas etapas seguintes, os atributos completos dos registros A e B serão agregados a essa tabela.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_pares`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df_pares = pd.DataFrame(pares)


## 2.21 — Função para cálculo da distância métrica

A função `calcular_distancia_metrica` calcula a distância euclidiana entre dois registros usando as coordenadas métricas `x_m` e `y_m`.

A fórmula utilizada é:

`distância = raiz quadrada de ((x1 - x2)² + (y1 - y2)²)`

Como as coordenadas estão em `EPSG:5880`, o resultado é expresso em metros.

Se algum dos registros do par não possuir coordenada métrica válida, a função retorna `NaN`.

**Contrato técnico com o código.** Esta célula define a função `calcular_distancia_metrica(row)`. Ela recebe `row` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def calcular_distancia_metrica(row):
    """
    Calcula a distância euclidiana em metros entre dois registros,
    usando coordenadas projetadas em EPSG:5880.
    """
    x1 = row["x_m_a"]
    y1 = row["y_m_a"]
    x2 = row["x_m_b"]
    y2 = row["y_m_b"]

    if pd.isna(x1) or pd.isna(y1) or pd.isna(x2) or pd.isna(y2):
        return np.nan

    return ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5


## 2.22 — Tratamento para o caso sem pares candidatos

Antes de juntar atributos e calcular distâncias, o script verifica se `df_pares` está vazio.

Se nenhum par candidato tiver sido encontrado, a etapa é encerrada com uma mensagem informativa.

Esse controle evita erros posteriores, pois não faria sentido tentar juntar atributos ou calcular distâncias em uma tabela vazia.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
if df_pares.empty:
    print("Etapa 2 concluída.")
    print("Nenhum par candidato foi encontrado.")


## 2.23 — Junção dos atributos e cálculo da distância definitiva

Quando existem pares candidatos, o script cria duas cópias da base `df`:

- `df_a`, com sufixo `_a`;
- `df_b`, com sufixo `_b`.

Essas cópias permitem juntar, na tabela de pares, os atributos completos dos dois registros comparados.

Assim, cada linha de `df_pares` passa a conter as informações do registro A e do registro B, como nome, CANIE, município, coordenadas, data de cadastro e campos auxiliares de comparação.

Essa estrutura é essencial para a Etapa 3, onde cada par será classificado conforme regras textuais, espaciais, temporais e cadastrais.

### Cálculo métrico após a junção

Após a junção dos atributos dos registros A e B, o script calcula a distância definitiva entre os dois pontos.

A distância é armazenada na coluna:

- `distancia_metros`.

Essa coluna será uma das informações centrais da Etapa 3, pois muitas regras de decisão dependem de limiares espaciais, como 1 m, 10 m, 20 m, 50 m ou 100 m.

O cálculo é feito com coordenadas métricas, portanto a unidade do resultado é metro.

### Conferência produzida pelo mesmo bloco

Ao final da etapa, o script imprime uma mensagem de conclusão.

Quando pares candidatos são encontrados, também informa:

- a quantidade total de pares candidatos gerados;
- o sistema métrico usado para calcular a distância.

Essas mensagens permitem verificar rapidamente se a geração de pares funcionou e se o cálculo espacial foi concluído.

**Contrato técnico com o código.** O bloco cria `df_a` e `df_b`, associa os atributos dos dois lados a `df_pares`, calcula `distancia_metros` e apresenta a contagem final. A célula é executada somente quando há pares candidatos; no caso vazio, o encerramento já foi tratado no tópico 2.22.


In [ ]:
if not df_pares.empty:
    # Prepara duas cópias da base para juntar os atributos dos registros A e B.
    df_a = df.add_suffix("_a")
    df_b = df.add_suffix("_b")

    df_pares = df_pares.merge(
        df_a,
        left_on="id_linha_a",
        right_on="id_linha_a",
        how="left"
    )

    df_pares = df_pares.merge(
        df_b,
        left_on="id_linha_b",
        right_on="id_linha_b",
        how="left"
    )

    # Calcula a distância definitiva em metros.
    df_pares["distancia_metros"] = df_pares.apply(
        calcular_distancia_metrica,
        axis=1
    )

    print("Etapa 2 concluída.")
    print(f"Pares candidatos gerados: {len(df_pares):,}")
    print("Distância calculada em metros usando EPSG:5880.")


## Síntese da Etapa 2

### Resultado produzido

`df` passa a conter coordenadas projetadas e chaves espaciais aproximadas. `df_pares` reúne cada combinação candidata com os atributos A/B, o motivo de geração e a distância definitiva em metros.

### Verificações esperadas

- pares devem ter `id_linha_a < id_linha_b`, evitando inversões duplicadas;
- chaves textuais vazias não devem gerar candidatos;
- `distancia_metros` deve ser nula apenas quando alguma coordenada necessária for inválida;
- a contagem informada no encerramento deve coincidir com `len(df_pares)`.

### Ligação com a etapa seguinte

A Etapa 3 recebe essa tabela e transforma evidências em decisões preliminares. A existência de um par candidato ainda não significa duplicidade.

**Natureza desta célula.** Esta síntese documenta o contrato de saída e não executa código.


# Etapa 3 — Classificação das evidências de duplicidade

## Finalidade

Esta etapa converte cada linha de `df_pares` em uma decisão preliminar explicável. O classificador não depende de uma única pontuação: combina evidências cadastrais, textuais, morfológicas, técnicas, espaciais e temporais em uma sequência explícita de regras.

## Entradas

Cada par já possui os atributos dos registros A e B e a distância calculada na Etapa 2. As funções auxiliares interpretam nomes, ordinais, códigos, datas e tipos morfológicos sem alterar os cadastros de origem.

## Modelo decisório

As regras são avaliadas por precedência. Regras protetivas aparecem antes das regras de remoção para evitar falsos positivos em séries como “Gruta I/II” ou códigos com letras finais distintas. Evidências fortes podem produzir `duplicidade_automatica`; evidências relevantes, mas insuficientes, produzem `revisar_manual`; conflitos ou baixa evidência produzem `nao_remover`.

## Produtos

- `df_pares`, acrescido de classe, decisão e evidências auxiliares;
- `df_pares_suspeitos`, contendo duplicidades automáticas e casos de revisão;
- funções pequenas e auditáveis, cada uma responsável por uma transformação ou regra.

## Como ler a etapa

Os tópicos 3.1 a 3.17 constroem as ferramentas analíticas. O tópico 3.18 prepara o contexto e aplica as 18 regras. O tópico 3.19 executa a classificação. A Etapa 3.20 registra decisões humanas sobre os casos pendentes.

**Natureza desta célula.** Esta abertura descreve o contrato do classificador; a execução começa em 3.1.


## 3.1 — Importação da função de similaridade textual

A função `SequenceMatcher`, da biblioteca `difflib`, é usada para medir a similaridade entre duas strings.

O resultado é um valor entre 0 e 1:

- valores próximos de 0 indicam textos muito diferentes;
- valores próximos de 1 indicam textos muito semelhantes ou idênticos.

Essa métrica será usada em várias regras para avaliar se dois nomes de cavidade são suficientemente parecidos para indicar possível duplicidade.

**Contrato técnico com o código.** A execução disponibiliza `SequenceMatcher` no ambiente do notebook. Nenhuma linha da base é transformada aqui; o efeito é somente tornar as dependências acessíveis às células seguintes.


In [ ]:
from difflib import SequenceMatcher


## 3.2 — Função para calcular similaridade textual

A função `calcular_similaridade_textual` compara dois textos e retorna uma pontuação entre 0 e 1.

Antes da comparação, valores nulos são convertidos para texto vazio. Caso os dois textos estejam vazios, a função retorna 0, evitando que dois campos ausentes sejam interpretados como nomes semelhantes.

Essa função é usada para avaliar a proximidade entre nomes normalizados, principalmente nos casos em que não há igualdade exata, mas existe semelhança relevante.

**Contrato técnico com o código.** Esta célula define a função `calcular_similaridade_textual(texto_a, texto_b)`. Ela recebe `texto_a`, `texto_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def calcular_similaridade_textual(texto_a, texto_b):
    """
    Calcula a similaridade textual entre duas strings.

    O resultado varia de 0 a 1:
    - 0 indica textos totalmente diferentes;
    - 1 indica textos idênticos.
    """
    texto_a = "" if pd.isna(texto_a) else str(texto_a)
    texto_b = "" if pd.isna(texto_b) else str(texto_b)

    if texto_a == "" and texto_b == "":
        return 0

    return SequenceMatcher(None, texto_a, texto_b).ratio()


## 3.3 — Função para calcular diferença entre datas de cadastro

A função `calcular_diferenca_dias` calcula a diferença absoluta, em dias, entre duas datas de cadastro.

Ela retorna `NaN` quando alguma das datas está ausente ou inválida.

Essa diferença temporal é usada para distinguir situações como:

- registros cadastrados na mesma campanha;
- registros muito próximos no tempo;
- possíveis recadastros feitos meses ou anos depois;
- casos em que datas muito distantes reforçam a hipótese de duplicidade ou recadastro.

**Contrato técnico com o código.** Esta célula define a função `calcular_diferenca_dias(data_a, data_b)`. Ela recebe `data_a`, `data_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def calcular_diferenca_dias(data_a, data_b):
    """
    Calcula a diferença absoluta, em dias, entre duas datas de cadastro.

    Retorna NaN quando alguma das datas está ausente ou inválida.
    """
    if pd.isna(data_a) or pd.isna(data_b):
        return np.nan

    return abs((data_a - data_b).days)


## 3.4 — Função para classificar a distância entre registros

A função `classificar_distancia` transforma a distância numérica em metros em uma categoria interpretável.

As classes criadas são:

- `sem_distancia`;
- `localizacao_igual`;
- `localizacao_praticamente_igual`;
- `localizacao_muito_proxima`;
- `localizacao_proxima`;
- `localizacao_revisar`;
- `localizacao_distante`;
- `localizacao_muito_distante`.

Essa classificação facilita a auditoria dos pares e ajuda a interpretar a proximidade espacial em termos qualitativos.

**Contrato técnico com o código.** Esta célula define a função `classificar_distancia(distancia)`. Ela recebe `distancia` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def classificar_distancia(distancia):
    """
    Classifica a distância entre dois pontos em classes interpretáveis
    para auditoria de duplicidade.
    """
    if pd.isna(distancia):
        return "sem_distancia"

    if distancia == 0:
        return "localizacao_igual"

    if distancia <= 1:
        return "localizacao_praticamente_igual"

    if distancia <= 10:
        return "localizacao_muito_proxima"

    if distancia <= 50:
        return "localizacao_proxima"

    if distancia <= 250:
        return "localizacao_revisar"

    if distancia <= 1000:
        return "localizacao_distante"

    return "localizacao_muito_distante"


## 3.5 — Função para normalizar algarismos romanos

A função `normalizar_romano_para_arabico` converte algarismos romanos simples para números arábicos.

Ela é usada para comparar nomes com sufixos ordinais.

Por exemplo:

- `I` vira `1`;
- `II` vira `2`;
- `III` vira `3`;
- `IV` vira `4`.

Isso permite que nomes como `Gruta Azul II` e `Gruta Azul 2` sejam interpretados de forma compatível.

**Contrato técnico com o código.** Esta célula define a função `normalizar_romano_para_arabico(token)`. Ela recebe `token` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def normalizar_romano_para_arabico(token):
    """
    Converte algarismos romanos simples para números arábicos.

    Exemplos:
    I    -> 1
    II   -> 2
    III  -> 3
    IV   -> 4
    V    -> 5
    X    -> 10
    """
    if pd.isna(token):
        return ""

    token = str(token).upper().strip()

    mapa_romanos = {
        "I": "1",
        "II": "2",
        "III": "3",
        "IV": "4",
        "V": "5",
        "VI": "6",
        "VII": "7",
        "VIII": "8",
        "IX": "9",
        "X": "10"
    }

    return mapa_romanos.get(token, token)


## 3.6 — Visão geral: extração do tipo morfológico da cavidade

A função `extrair_tipo_morfologico` identifica o primeiro termo morfológico relevante no nome da cavidade.

O script adota equivalências controladas:

Grupo equivalente 1:

- `GRUTA`;
- `CAVERNA`;
- `LAPA`;
- `FURNA`;
- `TOCA`.

Grupo equivalente 2:

- `DOLINA`;
- `SUMIDOURO`.

Tipos mantidos separados:

- `ABRIGO`;
- `ABISMO`;
- `PALEOTOCA`.

Essa distinção é importante porque nem toda diferença de tipo deve indicar cavidade diferente. Por exemplo, `Gruta` e `Caverna` podem ser variações nominais. Já `Abrigo` e `Abismo` são mantidos separados porque podem representar feições morfologicamente distintas.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 3.6.1 — Dicionário de equivalências morfológicas

A tabela explicita quais termos são aproximados e quais permanecem distintos. Separá-la da função facilita auditoria e futuras alterações sem misturar dados de configuração com lógica.

**Estrutura da configuração.** Cada chave é um termo encontrado no nome padronizado; cada valor é a categoria usada na comparação. Gruta, caverna, lapa, furna e toca compartilham uma categoria, assim como dolina e sumidouro. Abrigo, abismo e paleotoca permanecem categorias próprias porque não devem ser aproximados automaticamente aos demais tipos.

**Uso posterior.** A função da célula seguinte consulta este dicionário. Alterar uma equivalência aqui afeta as evidências morfológicas da Etapa 3 e, por consequência, pode mudar a classe e a decisão preliminar de pares candidatos já formados na Etapa 2.

**Contrato técnico com o código.** O bloco cria ou atualiza `EQUIVALENCIAS_MORFOLOGICAS`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
EQUIVALENCIAS_MORFOLOGICAS = {
    "GRUTA": "GRUTA_CAVERNA_LAPA_FURNA_TOCA",
    "CAVERNA": "GRUTA_CAVERNA_LAPA_FURNA_TOCA",
    "LAPA": "GRUTA_CAVERNA_LAPA_FURNA_TOCA",
    "FURNA": "GRUTA_CAVERNA_LAPA_FURNA_TOCA",
    "TOCA": "GRUTA_CAVERNA_LAPA_FURNA_TOCA",

    "DOLINA": "DOLINA_SUMIDOURO",
    "SUMIDOURO": "DOLINA_SUMIDOURO",

    "ABRIGO": "ABRIGO",
    "ABISMO": "ABISMO",
    "PALEOTOCA": "PALEOTOCA"
}


### 3.6.2 — Extração da categoria morfológica

Após limpar o nome, a função percorre seus termos e devolve a primeira categoria reconhecida. Texto vazio sinaliza ausência de tipo conhecido e impede equivalências artificiais.

**Processamento.** `limpar_texto_basico` coloca o valor no mesmo padrão usado pelas demais comparações. Em seguida, os termos são percorridos na ordem em que aparecem no nome; a primeira chave presente em `EQUIVALENCIAS_MORFOLOGICAS` determina a categoria retornada. Se nenhum termo for reconhecido, o retorno é `""`.

**Por que o vazio importa.** A ausência de categoria não é tratada como uma categoria comum. Isso impede que dois nomes sem informação morfológica sejam considerados equivalentes apenas porque ambos produziram vazio.

**Contrato técnico com o código.** Esta célula define a função `extrair_tipo_morfologico(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def extrair_tipo_morfologico(nome):
    """
    Extrai o primeiro termo morfológico relevante do nome da cavidade.

    Equivalências adotadas:

    Grupo 1:
    - GRUTA
    - CAVERNA
    - LAPA
    - FURNA
    - TOCA

    Grupo 2:
    - DOLINA
    - SUMIDOURO

    Tipos mantidos separados:
    - ABRIGO
    - ABISMO
    - PALEOTOCA
    """
    nome = limpar_texto_basico(nome)

    tokens = nome.split()

    for token in tokens:
        if token in EQUIVALENCIAS_MORFOLOGICAS:
            return EQUIVALENCIAS_MORFOLOGICAS[token]

    return ""


## 3.7 — Verificação de tipos morfológicos diferentes

A função `tipos_morfologicos_diferentes` compara o tipo morfológico extraído de dois nomes.

Como alguns termos são tratados como equivalentes, pares como `GRUTA x CAVERNA` ou `LAPA x TOCA` não são considerados morfologicamente diferentes.

Por outro lado, pares como `ABRIGO x GRUTA` ou `ABISMO x LAPA` são considerados diferentes.

Essa verificação é essencial para evitar remoções automáticas indevidas em casos de possível diferença morfológica real.

**Contrato técnico com o código.** Esta célula define a função `tipos_morfologicos_diferentes(nome_a, nome_b)`. Ela recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def tipos_morfologicos_diferentes(nome_a, nome_b):
    """
    Verifica se dois nomes possuem tipos morfológicos diferentes.

    Como alguns tipos são normalizados para o mesmo grupo, eles não
    serão considerados diferentes entre si.
    """
    tipo_a = extrair_tipo_morfologico(nome_a)
    tipo_b = extrair_tipo_morfologico(nome_b)

    if tipo_a == "" or tipo_b == "":
        return False

    return tipo_a != tipo_b


## 3.8 — Geração de nome morfologicamente equivalente

A função `gerar_nome_morfologico_equivalente` cria uma versão comparável do nome da cavidade, substituindo termos morfológicos equivalentes por um mesmo termo.

As equivalências aplicadas são:

- `GRUTA`, `CAVERNA`, `LAPA`, `FURNA` e `TOCA` viram `CAVIDADE`;
- `DOLINA` e `SUMIDOURO` viram `DOLINA_SUMIDOURO`.

Conectores como `DA`, `DE`, `DO`, `DAS`, `DOS`, `A`, `O`, `AS` e `OS` são removidos.

Essa função permite identificar nomes que são equivalentes apesar de usarem termos morfológicos diferentes.

**Contrato técnico com o código.** Esta célula define a função `gerar_nome_morfologico_equivalente(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def gerar_nome_morfologico_equivalente(nome):
    """
    Cria uma versão do nome com equivalência morfológica controlada.
    """
    nome = limpar_texto_basico(nome)

    nome = re.sub(r"[-_/\\.]", " ", nome)
    nome = re.sub(r"[^A-Z0-9 ]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip()

    equivalencias = {
        "GRUTA": "CAVIDADE",
        "CAVERNA": "CAVIDADE",
        "LAPA": "CAVIDADE",
        "FURNA": "CAVIDADE",
        "TOCA": "CAVIDADE",

        "DOLINA": "DOLINA_SUMIDOURO",
        "SUMIDOURO": "DOLINA_SUMIDOURO"
    }

    conectores = {
        "DA", "DE", "DO", "DAS", "DOS",
        "A", "O", "AS", "OS"
    }

    palavras = []

    for token in nome.split():
        token = equivalencias.get(token, token)

        if token in conectores:
            continue

        palavras.append(token)

    return " ".join(palavras)


## 3.9 — Extração de radical e sufixo ordinal

A função `extrair_radical_e_sufixo_ordinal` separa o nome da cavidade em duas partes:

- radical do nome;
- sufixo ordinal, quando existente.

Ela identifica sufixos numéricos, letras finais isoladas e algarismos romanos convertidos para números.

Exemplos:

- `LAPA DO JOAO I` vira radical `LAPA DO JOAO` e sufixo `1`;
- `GRUTA AZUL 02` vira radical `GRUTA AZUL` e sufixo `2`;
- `ABRIGO A` vira radical `ABRIGO` e sufixo `A`.

Essa separação é usada para evitar que cavidades sequenciais sejam confundidas com duplicidades.

**Contrato técnico com o código.** Esta célula define a função `extrair_radical_e_sufixo_ordinal(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def extrair_radical_e_sufixo_ordinal(nome):
    """
    Separa o radical do nome e um possível sufixo ordinal final.
    """
    nome = limpar_texto_basico(nome)
    nome = re.sub(r"[-_/\\.]", " ", nome)
    nome = re.sub(r"[^A-Z0-9 ]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip()

    if nome == "":
        return "", ""

    tokens = nome.split()

    if len(tokens) < 2:
        return nome, ""

    ultimo_token = tokens[-1]
    ultimo_token = normalizar_romano_para_arabico(ultimo_token)

    eh_numero = re.fullmatch(r"\d+", ultimo_token) is not None
    eh_letra = re.fullmatch(r"[A-Z]", ultimo_token) is not None

    if eh_numero:
        radical = " ".join(tokens[:-1]).strip()
        sufixo = str(int(ultimo_token))
        return radical, sufixo

    if eh_letra:
        radical = " ".join(tokens[:-1]).strip()
        sufixo = ultimo_token
        return radical, sufixo

    return nome, ""


## 3.10 — Comparação de ordinais

A função `comparar_ordinais` compara os sufixos ordinais de dois nomes.

Ela retorna uma das seguintes situações:

- `ordinal_igual`;
- `ordinal_diferente`;
- `ordinal_em_apenas_um`;
- `sem_ordinal`.

A interpretação usada no script é conservadora:

- ordinais diferentes indicam cavidades diferentes de uma série;
- ordinal presente em apenas um nome sugere comparação entre cavidade-base e cavidade numerada;
- ordinal igual ou ausência de ordinal permite seguir com a avaliação normal de duplicidade.

**Contrato técnico com o código.** Esta célula define a função `comparar_ordinais(nome_a, nome_b)`. Ela recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def comparar_ordinais(nome_a, nome_b):
    """
    Compara os sufixos ordinais dos dois nomes.
    """
    radical_a, sufixo_a = extrair_radical_e_sufixo_ordinal(nome_a)
    radical_b, sufixo_b = extrair_radical_e_sufixo_ordinal(nome_b)

    if sufixo_a == "" and sufixo_b == "":
        return "sem_ordinal"

    if sufixo_a != "" and sufixo_b != "":
        if sufixo_a == sufixo_b:
            return "ordinal_igual"

        return "ordinal_diferente"

    return "ordinal_em_apenas_um"


## 3.11 — Funções auxiliares para conflitos ordinais

As funções `tem_sufixo_ordinal_diferente` e `tem_ordinal_presente_em_apenas_um_nome` identificam situações específicas de série numerada.

A primeira verifica se dois nomes têm o mesmo radical, mas sufixos diferentes.

A segunda verifica se apenas um dos dois nomes possui sufixo ordinal ou numérico.

Essas funções ajudam a evitar a remoção de cavidades que pertencem a uma sequência, como `Gruta Azul I`, `Gruta Azul II` e `Gruta Azul III`.

**Contrato técnico com o código.** Esta célula define a função `tem_sufixo_ordinal_diferente(nome_a, nome_b)`. Ela recebe `nome_a`, `nome_b` e devolve um resultado ao chamador; `tem_ordinal_presente_em_apenas_um_nome(nome_a, nome_b)` recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def tem_sufixo_ordinal_diferente(nome_a, nome_b):
    """
    Verifica se dois nomes têm o mesmo radical, mas sufixos ordinais diferentes.

    Mantida por compatibilidade com regras auxiliares.
    """
    radical_a, sufixo_a = extrair_radical_e_sufixo_ordinal(nome_a)
    radical_b, sufixo_b = extrair_radical_e_sufixo_ordinal(nome_b)

    if radical_a == "" or radical_b == "":
        return False

    if sufixo_a == "" or sufixo_b == "":
        return False

    return radical_a == radical_b and sufixo_a != sufixo_b

def tem_ordinal_presente_em_apenas_um_nome(nome_a, nome_b):
    """
    Verifica se dois nomes têm o mesmo radical, mas apenas um deles possui
    sufixo ordinal ou numérico.
    """
    radical_a, sufixo_a = extrair_radical_e_sufixo_ordinal(nome_a)
    radical_b, sufixo_b = extrair_radical_e_sufixo_ordinal(nome_b)

    if radical_a == "" or radical_b == "":
        return False

    um_tem_sufixo = sufixo_a != "" and sufixo_b == ""
    outro_tem_sufixo = sufixo_a == "" and sufixo_b != ""

    if not (um_tem_sufixo or outro_tem_sufixo):
        return False

    return radical_a == radical_b


## 3.12 — Extração de código técnico simples

A função `extrair_codigo_tecnico` tenta identificar nomes que seguem o padrão de código técnico com prefixo e número.

Exemplos:

- `CAV-01` vira prefixo `CAV` e número `1`;
- `FEA-009` vira prefixo `FEA` e número `9`;
- `S2-SF-078` vira prefixo `S2SF` e número `78`.

Essa função permite diferenciar códigos iguais, códigos com formatação diferente e códigos de uma mesma série técnica com numeração distinta.

**Contrato técnico com o código.** Esta célula define a função `extrair_codigo_tecnico(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def extrair_codigo_tecnico(nome):
    """
    Tenta extrair um padrão de código técnico formado por prefixo e número.
    """
    nome = limpar_texto_basico(nome)
    nome = re.sub(r"[^A-Z0-9]", "", nome)

    if nome == "":
        return "", ""

    match = re.match(r"^([A-Z0-9]*?[A-Z]+[A-Z0-9]*?)(\d+)$", nome)

    if not match:
        return "", ""

    prefixo = match.group(1)
    numero = str(int(match.group(2)))

    return prefixo, numero


## 3.13 — Identificação de código técnico com numeração diferente

A função `tem_codigo_tecnico_diferente` identifica pares em que os dois nomes possuem o mesmo prefixo técnico, mas números diferentes.

Exemplos:

- `CAV-01 x CAV-02`;
- `FEA-009 x FEA-010`;
- `S2-SF-078 x S2-SF-079`.

Esses casos são interpretados como cavidades diferentes dentro de uma série técnica, e não como duplicidades.

**Contrato técnico com o código.** Esta célula define a função `tem_codigo_tecnico_diferente(nome_a, nome_b)`. Ela recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def tem_codigo_tecnico_diferente(nome_a, nome_b):
    """
    Identifica códigos técnicos com mesmo prefixo, mas numeração diferente.
    """
    prefixo_a, numero_a = extrair_codigo_tecnico(nome_a)
    prefixo_b, numero_b = extrair_codigo_tecnico(nome_b)

    if prefixo_a == "" or prefixo_b == "":
        return False

    if numero_a == "" or numero_b == "":
        return False

    return prefixo_a == prefixo_b and numero_a != numero_b


## 3.14 — Códigos técnicos com letra final

A função `extrair_codigo_com_sufixo_letra` trata códigos formados por prefixo, número e letra final.

Exemplos:

- `SM-054A`;
- `SM-054B`;
- `ICCA-0074A`;
- `LTA-0042B`.

A letra final pode indicar subfeições, variantes ou cavidades distintas de uma mesma série. Por isso, o script trata diferenças de letra final com cautela.

**Contrato técnico com o código.** Esta célula define a função `extrair_codigo_com_sufixo_letra(nome)`. Ela recebe `nome` e devolve um resultado ao chamador; `tem_mesmo_codigo_com_letra_final_diferente(nome_a, nome_b)` recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def extrair_codigo_com_sufixo_letra(nome):
    """
    Extrai códigos técnicos com número e letra final.
    """
    nome = limpar_texto_basico(nome)
    nome = re.sub(r"[^A-Z0-9]", "", nome)

    if nome == "":
        return "", "", ""

    match = re.match(r"^([A-Z]+[A-Z0-9]*?)(\d+)([A-Z])$", nome)

    if not match:
        return "", "", ""

    prefixo = match.group(1)
    numero = str(int(match.group(2)))
    letra = match.group(3)

    return prefixo, numero, letra

def tem_mesmo_codigo_com_letra_final_diferente(nome_a, nome_b):
    """
    Identifica pares com mesmo prefixo e mesmo número, mas letra final diferente.
    """
    prefixo_a, numero_a, letra_a = extrair_codigo_com_sufixo_letra(nome_a)
    prefixo_b, numero_b, letra_b = extrair_codigo_com_sufixo_letra(nome_b)

    if "" in [prefixo_a, numero_a, letra_a, prefixo_b, numero_b, letra_b]:
        return False

    return (
        prefixo_a == prefixo_b
        and numero_a == numero_b
        and letra_a != letra_b
    )


## 3.15 — Código canônico sem zeros à esquerda

A função `extrair_codigo_canonico_sem_zero` cria uma forma canônica de códigos técnicos.

Ela remove separadores e zeros à esquerda do número final.

Exemplos:

- `CAVRP3` e `CAVRP-03` viram `CAVRP3`;
- `SL_0125` e `SL_125` viram `SL125`;
- `Gruta 099` e `Gruta 99` viram `GRUTA99`.

A função `mesmo_codigo_canonico_sem_zero` compara duas formas canônicas. Quando elas são iguais, há forte indício de que os nomes representam o mesmo código escrito de formas diferentes.

**Contrato técnico com o código.** Esta célula define a função `extrair_codigo_canonico_sem_zero(nome)`. Ela recebe `nome` e devolve um resultado ao chamador; `mesmo_codigo_canonico_sem_zero(nome_a, nome_b)` recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def extrair_codigo_canonico_sem_zero(nome):
    """
    Cria uma forma canônica de códigos técnicos removendo separadores
    e zeros à esquerda do número final.
    """
    nome = limpar_texto_basico(nome)
    nome = re.sub(r"[^A-Z0-9]", "", nome)

    if nome == "":
        return ""

    match = re.match(r"^(.+?)(\d+)$", nome)

    if not match:
        return nome

    prefixo = match.group(1)
    numero = str(int(match.group(2)))

    return f"{prefixo}{numero}"

def mesmo_codigo_canonico_sem_zero(nome_a, nome_b):
    """
    Verifica se dois nomes/códigos são iguais após remoção de separadores
    e zeros à esquerda no número final.
    """
    codigo_a = extrair_codigo_canonico_sem_zero(nome_a)
    codigo_b = extrair_codigo_canonico_sem_zero(nome_b)

    if codigo_a == "" or codigo_b == "":
        return False

    return codigo_a == codigo_b


## 3.16 — Visão geral: extração de base, número e letra opcional

A função `extrair_base_numero_letra` identifica nomes formados por uma base textual, um número e uma letra final opcional.

Exemplos:

- `BRILAM 1`;
- `BRILAM 1 A`;
- `LF-168`;
- `LF-168B`.

A função `tem_mesma_base_numero_e_letra_opcional` identifica quando dois nomes compartilham a mesma base e número, mas diferem pela letra final ou pela presença/ausência dela.

Esses casos são tratados como possíveis cavidades distintas, subfeições ou séries internas.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 3.16.1 — Decomposição de códigos em base, número e letra

A função reconhece formatos compactos e separados, normaliza zeros e devolve componentes comparáveis. Falhas de reconhecimento produzem uma tupla vazia, preservando comportamento conservador.

**Etapas da decomposição.** Separadores usuais são convertidos em espaços, caracteres estranhos são removidos e uma versão compacta é avaliada por expressão regular. Quando o padrão é reconhecido, o retorno contém a base alfanumérica, o número sem zeros à esquerda e uma letra final opcional. Por exemplo, representações equivalentes de um mesmo número passam a compartilhar o mesmo componente numérico.

**Contrato da função.** O retorno tem sempre três posições. A tupla `("", "", "")` indica que o nome não pôde ser interpretado como código e permite que a comparação posterior encerre com `False`, em vez de criar uma coincidência a partir de ausências.

**Contrato técnico com o código.** Esta célula define a função `extrair_base_numero_letra(nome)`. Ela recebe `nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def extrair_base_numero_letra(nome):
    """
    Extrai base, número e letra final opcional.
    """
    nome = limpar_texto_basico(nome)
    nome = re.sub(r"[-_/\\.]", " ", nome)
    nome = re.sub(r"[^A-Z0-9 ]", " ", nome)
    nome = re.sub(r"\s+", " ", nome).strip()

    compacto = re.sub(r"[^A-Z0-9]", "", nome)

    match = re.match(r"^([A-Z]+[A-Z0-9]*?)(\d+)([A-Z]?)$", compacto)

    if match:
        base = match.group(1)
        numero = str(int(match.group(2)))
        letra = match.group(3)
        return base, numero, letra

    tokens = nome.split()

    if len(tokens) >= 2:
        ultimo = tokens[-1]
        penultimo = tokens[-2]

        if re.fullmatch(r"[A-Z]", ultimo) and re.fullmatch(r"\d+", penultimo):
            base = "".join(tokens[:-2])
            numero = str(int(penultimo))
            letra = ultimo
            return base, numero, letra

        if re.fullmatch(r"\d+", ultimo):
            base = "".join(tokens[:-1])
            numero = str(int(ultimo))
            letra = ""
            return base, numero, letra

    return "", "", ""


### 3.16.2 — Comparação de letras opcionais

Depois da decomposição, a função exige base e número iguais e válidos. Diferença na letra final é tratada como evidência de elementos distintos de uma série.

**Condições avaliadas.** Primeiro são rejeitados resultados sem base ou sem número. Depois, base e número precisam coincidir exatamente. Somente então a letra é comparada; a função retorna `True` quando ela está ausente em um nome ou é diferente entre os dois nomes.

**Efeito na classificação.** Esse indicador funciona como evidência de distinção. Ele evita que elementos numerados de uma mesma série sejam removidos apenas por compartilharem grande parte do código textual.

**Contrato técnico com o código.** Esta célula define a função `tem_mesma_base_numero_e_letra_opcional(nome_a, nome_b)`. Ela recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def tem_mesma_base_numero_e_letra_opcional(nome_a, nome_b):
    """
    Identifica casos em que dois nomes compartilham base e número,
    mas diferem pela presença, ausência ou valor da letra final.
    """
    base_a, numero_a, letra_a = extrair_base_numero_letra(nome_a)
    base_b, numero_b, letra_b = extrair_base_numero_letra(nome_b)

    if base_a == "" or base_b == "":
        return False

    if numero_a == "" or numero_b == "":
        return False

    if base_a != base_b or numero_a != numero_b:
        return False

    return letra_a != letra_b


## 3.17 — Visão geral: nome-base significativo e relação textual mínima

A função `nome_base_significativo` verifica se o nome-base possui informação suficiente para comparação.

Nomes-base muito curtos podem gerar falsos positivos. Por isso, o script exige que o nome-base tenha pelo menos seis caracteres, desconsiderando espaços.

A função `um_nome_contem_o_outro` verifica se um nome normalizado contém o outro. Isso captura casos como:

- `GRUTA BACAETAVA x GRUTA DO BACAETAVA`;
- `BAMBUZINHO II x ABRIGO DO BAMBUZINHO II`.

A função `existe_relacao_textual_minima` combina várias evidências textuais para decidir se há relação mínima entre dois nomes.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 3.17.1 — Validação da força informativa do nome-base

A função rejeita nomes-base nulos, vazios ou com menos de seis caracteres úteis. Esse limite reduz falsos positivos produzidos por radicais muito curtos ou genéricos.

Ela é usada tanto na verificação de relação textual quanto nas regras que comparam `nome_base`. O retorno é apenas booleano e não modifica os dados.

**Contrato técnico com o código.** Esta célula define a função `nome_base_significativo(nome_base)`. Ela recebe `nome_base` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def nome_base_significativo(nome_base):
    """
    Avalia se o nome-base tem informação suficiente para comparação.

    Nomes-base muito curtos ou genéricos podem gerar falsos positivos.
    """
    if pd.isna(nome_base):
        return False

    nome_base = str(nome_base).strip()

    if nome_base == "":
        return False

    nome_sem_espaco = nome_base.replace(" ", "")

    if len(nome_sem_espaco) < 6:
        return False

    return True


### 3.17.2 — Verificação de contenção entre nomes

Os dois nomes passam pela mesma limpeza de acentos, caixa, pontuação e espaços. Depois, a função verifica se um texto normalizado está contido no outro.

Nomes vazios ou muito curtos são recusados para evitar coincidências triviais. Essa evidência complementa a igualdade exata e a similaridade calculada por `SequenceMatcher`.

**Contrato técnico com o código.** Esta célula define a função `um_nome_contem_o_outro(nome_a, nome_b)`. Ela recebe `nome_a`, `nome_b` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def um_nome_contem_o_outro(nome_a, nome_b):
    """
    Verifica se um nome normalizado contém o outro.
    """
    nome_a = limpar_texto_basico(nome_a)
    nome_b = limpar_texto_basico(nome_b)

    nome_a = re.sub(r"[-_/\\.]", " ", nome_a)
    nome_b = re.sub(r"[-_/\\.]", " ", nome_b)

    nome_a = re.sub(r"[^A-Z0-9 ]", " ", nome_a)
    nome_b = re.sub(r"[^A-Z0-9 ]", " ", nome_b)

    nome_a = re.sub(r"\s+", " ", nome_a).strip()
    nome_b = re.sub(r"\s+", " ", nome_b).strip()

    if nome_a == "" or nome_b == "":
        return False

    if len(nome_a) < 6 or len(nome_b) < 6:
        return False

    return nome_a in nome_b or nome_b in nome_a


### 3.17.3 — Consolidação das evidências textuais mínimas

Esta função reúne cinco caminhos possíveis de relação: mesmo nome, mesmo código, mesmo nome-base significativo, contenção entre nomes ou similaridade igual ou superior a 0,75.

Ela funciona como uma barreira de segurança nas regras posteriores: proximidade espacial sem qualquer relação textual não deve, por si só, gerar remoção ou revisão prioritária.

**Contrato técnico com o código.** Esta célula define a função `existe_relacao_textual_minima(row, similaridade_nome)`. Ela recebe `row`, `similaridade_nome` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def existe_relacao_textual_minima(row, similaridade_nome):
    """
    Verifica se há relação textual mínima entre dois nomes.

    Essa função evita que registros com nomes totalmente diferentes sejam
    enviados para revisão manual apenas porque têm a mesma coordenada.
    """
    mesmo_nome = row["nome_comparacao_a"] == row["nome_comparacao_b"]

    mesmo_nome_base = (
        row["nome_base_a"] == row["nome_base_b"]
        and nome_base_significativo(row["nome_base_a"])
    )

    mesmo_codigo = (
        row["nome_codigo_a"] == row["nome_codigo_b"]
        and row["nome_codigo_a"] != ""
    )

    contem_nome = um_nome_contem_o_outro(
        row[f"{COL_NOME}_a"],
        row[f"{COL_NOME}_b"]
    )

    if mesmo_nome:
        return True

    if mesmo_codigo:
        return True

    if mesmo_nome_base:
        return True

    if contem_nome:
        return True

    if similaridade_nome >= 0.75:
        return True

    return False


## 3.18 — Visão geral: função principal de classificação dos pares

A função `classificar_suspeita` é o centro decisório da Etapa 3.

Ela recebe uma linha de `df_pares`, ou seja, um par de registros, e avalia esse par conforme um conjunto de regras hierarquizadas.

A função calcula variáveis auxiliares, como:

- se o CANIE é igual ou diferente;
- se o nome normalizado é igual;
- se o nome-base é igual e significativo;
- se o código técnico é igual;
- se os registros estão na mesma UF e município;
- se as datas são iguais, próximas ou distantes;
- a similaridade textual entre os nomes;
- o nome morfologicamente equivalente;
- a situação ordinal;
- conflitos de código, letra final ou série;
- existência de relação textual mínima.

A função retorna uma `pd.Series` com:

- `classe_suspeita`;
- `decisao_preliminar`;
- atributos auxiliares para auditoria.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


## 3.18.1 — Visão geral: preparação do contexto cadastral, espacial e temporal

A classificação será dividida em funções menores, mas todas precisam trabalhar com os mesmos fatos calculados para o par. Esta função cria a primeira parte desse contexto.

Ela verifica igualdade ou diferença de CANIE, nome, nome-base, código, UF, município e data; recupera a distância métrica calculada na Etapa 2; calcula a diferença entre datas; e cria indicadores para datas próximas, moderadamente próximas, distantes e muito distantes.

O resultado é um dicionário de contexto. Nenhuma decisão é tomada aqui: o bloco apenas calcula evidências que serão reutilizadas pelas regras seguintes, evitando recalcular os mesmos valores em cada função.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


## 3.18.1.1 — Comparações cadastrais

São calculadas igualdade de CANIE, nome, nome-base, código, UF, município e data. O bloco produz apenas evidências; nenhuma decisão ocorre nesta fase.

**Entrada e indicadores.** A função recebe uma linha de `df_pares`, que já contém os atributos dos registros A e B. Igualdades diretas são calculadas sobre CANIE, nome normalizado, código, UF, município e data. Para `mesmo_nome_base`, a igualdade só vale quando a chave também passa por `nome_base_significativo`, evitando coincidências entre strings vazias.

**Saída.** Um dicionário de contexto reúne a linha original e os booleanos calculados. Esse dicionário é enriquecido pelas células seguintes e depois consumido, sem recomputação, pelas regras ordenadas de classificação.

**Contrato técnico com o código.** Esta célula define a função `criar_contexto_cadastral(row)`. Ela recebe `row` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_contexto_cadastral(row):
    mesmo_canie = row["canie_txt_a"] == row["canie_txt_b"]
    mesmo_nome = row["nome_comparacao_a"] == row["nome_comparacao_b"]
    mesmo_nome_base = row["nome_base_a"] == row["nome_base_b"] and nome_base_significativo(row["nome_base_a"])
    mesmo_codigo = row["nome_codigo_a"] == row["nome_codigo_b"] and row["nome_codigo_a"] != ""
    return {"row": row, "mesmo_canie": mesmo_canie, "canie_diferente": not mesmo_canie,
            "mesmo_nome": mesmo_nome, "mesmo_nome_base": mesmo_nome_base,
            "mesmo_codigo": mesmo_codigo, "mesma_uf": row["uf_txt_a"] == row["uf_txt_b"],
            "mesmo_municipio": row["municipio_txt_a"] == row["municipio_txt_b"],
            "mesma_data": row["data_cadastro_dt_a"] == row["data_cadastro_dt_b"]}


## 3.18.1.2 — Distância e faixas temporais

A distância da Etapa 2 e a diferença absoluta entre datas são incorporadas ao contexto. Quatro indicadores distinguem cargas próximas de possíveis recadastros históricos.

**Processamento.** `calcular_diferenca_dias` produz a diferença absoluta entre as datas, preservando ausência quando uma delas não é válida. A distância métrica calculada na Etapa 2 é copiada para o contexto. Os limites de 30, 180 e 365 dias geram indicadores sobrepostos com significados distintos para as regras de negócio.

**Tratamento de ausências.** Cada faixa temporal exige `pd.notna(diferenca)`. Desse modo, datas ausentes não são interpretadas como próximas nem distantes e não fornecem evidência artificial à classificação.

**Contrato técnico com o código.** Esta célula define a função `adicionar_contexto_espacial_temporal(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_contexto_espacial_temporal(ctx):
    row = ctx["row"]
    diferenca = calcular_diferenca_dias(row["data_cadastro_dt_a"], row["data_cadastro_dt_b"])
    ctx.update({"distancia": row["distancia_metros"], "diferenca_dias_cadastro": diferenca,
                "datas_proximas": pd.notna(diferenca) and diferenca <= 30,
                "datas_moderadamente_proximas": pd.notna(diferenca) and 30 < diferenca <= 180,
                "datas_distantes": pd.notna(diferenca) and diferenca > 180,
                "datas_muito_distantes": pd.notna(diferenca) and diferenca > 365})
    return ctx


## 3.18.1.3 — Coordenação do contexto básico

A interface original é mantida por uma função curta que encadeia as duas preparações anteriores.

**Encadeamento.** A função recebe a linha do par, cria primeiro as evidências cadastrais e passa o dicionário resultante à preparação espacial e temporal. O retorno preserva uma única interface para o classificador principal.

**Razão da separação.** Manter as duas preparações em funções menores permite testar ou alterar um conjunto de evidências sem misturar igualdade cadastral com distância e datas. As regras posteriores continuam recebendo o mesmo objeto de contexto.

**Contrato técnico com o código.** Esta célula define a função `criar_contexto_cadastral_temporal(row)`. Ela recebe `row` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_contexto_cadastral_temporal(row):
    return adicionar_contexto_espacial_temporal(criar_contexto_cadastral(row))


## 3.18.2 — Visão geral: enriquecimento textual, morfológico e técnico do contexto

Este bloco acrescenta as evidências derivadas dos nomes: similaridade textual, equivalência morfológica, ordinais, códigos técnicos, letras finais e relação textual mínima.

Ele depende das funções auxiliares definidas nas seções 3.2–3.17 e do próprio par armazenado em `ctx["row"]`. Os resultados são incorporados ao mesmo dicionário criado no bloco anterior.

Separar essa preparação da parte cadastral torna explícito que as decisões combinam dimensões independentes: cadastro, tempo, espaço e texto.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


## 3.18.2.1 — Similaridade e equivalência morfológica

Este bloco calcula a razão de similaridade entre os nomes normalizados e produz versões morfologicamente equivalentes. Também identifica quando os tipos de cavidade são incompatíveis.

Os valores são anexados ao contexto compartilhado e serão usados por regras de igualdade de nome, morfologia e baixa similaridade.

**Contrato técnico com o código.** Esta célula define a função `adicionar_evidencias_morfologicas(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_evidencias_morfologicas(ctx):
    row = ctx["row"]
    similaridade_nome = calcular_similaridade_textual(
        row["nome_comparacao_a"], row["nome_comparacao_b"]
    )
    nome_morfologico_a = gerar_nome_morfologico_equivalente(row[f"{COL_NOME}_a"])
    nome_morfologico_b = gerar_nome_morfologico_equivalente(row[f"{COL_NOME}_b"])
    mesmo_nome_morfologico = (
        nome_morfologico_a == nome_morfologico_b and nome_morfologico_a != ""
    )
    tipo_morfologico_diferente = tipos_morfologicos_diferentes(
        row[f"{COL_NOME}_a"], row[f"{COL_NOME}_b"]
    )
    ctx.update({
        "similaridade_nome": similaridade_nome,
        "nome_morfologico_a": nome_morfologico_a,
        "nome_morfologico_b": nome_morfologico_b,
        "mesmo_nome_morfologico": mesmo_nome_morfologico,
        "tipo_morfologico_diferente": tipo_morfologico_diferente
    })
    return ctx


## 3.18.2.2 — Evidências ordinais

A comparação ordinal distingue nomes sem ordinal, com ordinal igual, com ordinal diferente ou com numeração presente em apenas um lado.

Os dois indicadores derivados protegem séries como “Gruta I” e “Gruta II” contra remoções indevidas.

**Saída adicionada ao contexto.** `situacao_ordinal` preserva a categoria detalhada devolvida por `comparar_ordinais`. Os dois booleanos derivados destacam os casos que impedem uma equivalência simples: ordinais diferentes e ordinal presente em apenas um dos nomes.

**Relação com regras posteriores.** Essas evidências são calculadas uma vez e reutilizadas na precedência de classificação. Elas têm papel protetivo, pois nomes quase idênticos podem designar cavidades distintas quando a numeração ordinal muda.

**Contrato técnico com o código.** Esta célula define a função `adicionar_evidencias_ordinais(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_evidencias_ordinais(ctx):
    row = ctx["row"]
    situacao_ordinal = comparar_ordinais(row[f"{COL_NOME}_a"], row[f"{COL_NOME}_b"])
    ctx.update({
        "situacao_ordinal": situacao_ordinal,
        "sufixo_ordinal_diferente": situacao_ordinal == "ordinal_diferente",
        "ordinal_presente_em_apenas_um_nome": situacao_ordinal == "ordinal_em_apenas_um"
    })
    return ctx


## 3.18.2.3 — Evidências de códigos técnicos

Este bloco compara numeração técnica, letras finais, forma canônica sem zeros e combinações de base, número e letra opcional.

Esses indicadores permitem separar códigos equivalentes de séries técnicas distintas antes que regras de similaridade mais genéricas sejam consideradas.

**Indicadores produzidos.** A função avalia diferença de número técnico, divergência de letra final, equivalência canônica após retirada de zeros e a situação de mesma base/número com letra opcional distinta. Cada teste representa uma hipótese específica e permanece separado no contexto para auditoria.

**Integração.** Os valores usam os nomes originais do par, enquanto as comparações textuais gerais usam colunas normalizadas. Essa combinação evita perder informação estrutural durante a limpeza do texto e fornece às regras posteriores evidências mais precisas que uma única pontuação de similaridade.

**Contrato técnico com o código.** Esta célula define a função `adicionar_evidencias_codigos(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_evidencias_codigos(ctx):
    row = ctx["row"]
    nome_a = row[f"{COL_NOME}_a"]
    nome_b = row[f"{COL_NOME}_b"]
    ctx.update({
        "codigo_tecnico_diferente": tem_codigo_tecnico_diferente(nome_a, nome_b),
        "codigo_com_letra_final_diferente": tem_mesmo_codigo_com_letra_final_diferente(nome_a, nome_b),
        "codigo_canonico_igual_sem_zero": mesmo_codigo_canonico_sem_zero(nome_a, nome_b),
        "mesma_base_numero_letra_opcional": tem_mesma_base_numero_e_letra_opcional(nome_a, nome_b)
    })
    return ctx


## 3.18.2.4 — Relação textual mínima e coordenação do enriquecimento

Depois de calcular similaridade, morfologia, ordinais e códigos, a função avalia se existe alguma relação textual mínima entre os registros.

A função coordenadora chama os três blocos anteriores em ordem e devolve um único contexto enriquecido para as 18 regras.

**Contrato técnico com o código.** Esta célula define a função `enriquecer_contexto_textual(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def enriquecer_contexto_textual(ctx):
    ctx = adicionar_evidencias_morfologicas(ctx)
    ctx = adicionar_evidencias_ordinais(ctx)
    ctx = adicionar_evidencias_codigos(ctx)
    ctx["relacao_textual_minima"] = existe_relacao_textual_minima(
        ctx["row"], ctx["similaridade_nome"]
    )
    return ctx


## 3.18.3 — Campos comuns devolvidos por todas as regras

Todas as classificações precisam retornar, além da classe e da decisão, informações auxiliares para auditoria. Este bloco reúne esses campos em `resultado_base`.

A similaridade, a diferença entre datas, a situação ordinal e os nomes morfológicos serão anexados a qualquer `pd.Series` retornada pelas regras. Isso garante que `df_pares` tenha estrutura consistente independentemente da regra acionada.

**Contrato técnico com o código.** Esta célula define a função `adicionar_resultado_base_ao_contexto(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_resultado_base_ao_contexto(ctx):
    ctx["resultado_base"] = {
        "similaridade_nome": ctx["similaridade_nome"],
        "diferenca_dias_cadastro": ctx["diferenca_dias_cadastro"],
        "situacao_ordinal": ctx["situacao_ordinal"],
        "nome_morfologico_a": ctx["nome_morfologico_a"],
        "nome_morfologico_b": ctx["nome_morfologico_b"]
    }
    return ctx


### 3.18.4 — Visão geral: regras 1 a 5: proteção de séries, ordinais e códigos distintos

As cinco primeiras regras são principalmente regras de proteção contra falsos positivos.

Elas evitam remover registros que parecem semelhantes, mas que provavelmente representam cavidades diferentes dentro de séries morfológicas, ordinais ou técnicas.

Essas regras retornam `nao_remover` quando identificam:

1. mesmo código com letra final diferente;
2. mesma base e número, mas letra opcional distinta;
3. ordinal diferente;
4. ordinal presente em apenas um dos nomes;
5. código técnico com mesma raiz, mas numeração diferente.

A lógica é que séries como `CAV-01`, `CAV-02`, `Gruta I`, `Gruta II`, `SM-054A` e `SM-054B` não devem ser tratadas como duplicidades apenas por terem nomes parecidos.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


#### 3.18.4.1 — Regra 1: mesmo código com letra final diferente.

**Finalidade.** Esta regra avalia especificamente `mesmo código com letra final diferente.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 1 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `codigo_com_letra_final_distinta`.

**Decisões possíveis:** `nao_remover`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_01(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_01(ctx):
    if ctx['codigo_com_letra_final_diferente'] and ctx['canie_diferente'] and ctx['mesma_uf'] and ctx['mesmo_municipio']:
        return pd.Series({'classe_suspeita': 'codigo_com_letra_final_distinta', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.4.2 — Regra 2: mesma base e número, mas letra opcional distinta.

**Finalidade.** Esta regra avalia especificamente `mesma base e número, mas letra opcional distinta.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 2 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `mesma_base_numero_letra_opcional_distinta`.

**Decisões possíveis:** `nao_remover`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_02(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_02(ctx):
    if ctx['canie_diferente'] and ctx['mesma_base_numero_letra_opcional'] and ctx['mesma_uf'] and ctx['mesmo_municipio']:
        return pd.Series({'classe_suspeita': 'mesma_base_numero_letra_opcional_distinta', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.4.3 — Regra 3: ordinal diferente.

**Finalidade.** Esta regra avalia especificamente `ordinal diferente.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 3 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `serie_numerada_ordinal_diferente`.

**Decisões possíveis:** `nao_remover`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_03(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_03(ctx):
    if ctx['canie_diferente'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['situacao_ordinal'] == 'ordinal_diferente'):
        return pd.Series({'classe_suspeita': 'serie_numerada_ordinal_diferente', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.4.4 — Regra 4: ordinal presente em apenas um nome.

**Finalidade.** Esta regra avalia especificamente `ordinal presente em apenas um nome.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 4 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `serie_nome_base_e_nome_numerado`.

**Decisões possíveis:** `nao_remover`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_04(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_04(ctx):
    if ctx['canie_diferente'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['situacao_ordinal'] == 'ordinal_em_apenas_um') and (ctx['mesmo_nome_base'] or ctx['relacao_textual_minima']):
        return pd.Series({'classe_suspeita': 'serie_nome_base_e_nome_numerado', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.4.5 — Regra 5: códigos técnicos com mesma raiz, mas numeração diferente.

**Finalidade.** Esta regra avalia especificamente `códigos técnicos com mesma raiz, mas numeração diferente.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 5 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `codigo_tecnico_numeracao_distinta`.

**Decisões possíveis:** `nao_remover`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_05(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_05(ctx):
    if ctx['codigo_tecnico_diferente'] and ctx['canie_diferente'] and ctx['mesma_uf'] and ctx['mesmo_municipio']:
        return pd.Series({'classe_suspeita': 'codigo_tecnico_numeracao_distinta', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


### 3.18.5 — Visão geral: regras 6 a 10: duplicidades automáticas fortes

As regras 6 a 10 identificam casos com evidência forte de duplicidade.

São classificados como `duplicidade_automatica` os pares que apresentam situações como:

- código canônico igual após remoção de separadores e zeros à esquerda;
- mesmo CANIE, mesmo nome, mesma localização e mesma data;
- mesmo nome com CANIE diferente e distância até 50 m;
- nome morfologicamente equivalente até 50 m;
- mesmo código técnico até 100 m.

Essas regras combinam forte evidência textual ou técnica com proximidade espacial.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


#### 3.18.5.1 — Regra 6: código canônico igual após remover separadores e zeros à esquerda.

**Finalidade.** Esta regra avalia especificamente `código canônico igual após remover separadores e zeros à esquerda.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 6 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `codigo_canonico_igual_sem_zero_ate_50m`.

**Decisões possíveis:** `duplicidade_automatica`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_06(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_06(ctx):
    if ctx['canie_diferente'] and ctx['codigo_canonico_igual_sem_zero'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 50):
        return pd.Series({'classe_suspeita': 'codigo_canonico_igual_sem_zero_ate_50m', 'decisao_preliminar': 'duplicidade_automatica', **ctx['resultado_base']})

    return None


#### 3.18.5.2 — Regra 7: duplicidade exata ou quase exata.

**Finalidade.** Esta regra avalia especificamente `duplicidade exata ou quase exata.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 7 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `duplicidade_exata_ou_quase_exata`.

**Decisões possíveis:** `duplicidade_automatica`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_07(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_07(ctx):
    if ctx['mesmo_canie'] and ctx['mesmo_nome'] and (ctx['distancia'] <= 1) and ctx['mesma_data']:
        return pd.Series({'classe_suspeita': 'duplicidade_exata_ou_quase_exata', 'decisao_preliminar': 'duplicidade_automatica', **ctx['resultado_base']})

    return None


#### 3.18.5.3 — Regra 8: mesmo nome, CANIE diferente e distância até 50 m.

**Finalidade.** Esta regra avalia especificamente `mesmo nome, canie diferente e distância até 50 m.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 8 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** definidas dinamicamente conforme datas e similaridade.

**Decisões possíveis:** `duplicidade_automatica`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_08(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_08(ctx):
    if ctx['canie_diferente'] and ctx['mesmo_nome'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 50):
        if ctx['datas_muito_distantes']:
            classe = 'mesmo_nome_ate_50m_datas_muito_distantes_possivel_recadastro'
        elif ctx['datas_distantes']:
            classe = 'mesmo_nome_ate_50m_datas_distantes_possivel_recadastro'
        elif ctx['datas_proximas']:
            classe = 'mesmo_nome_ate_50m_datas_proximas_possivel_carga_duplicada'
        else:
            classe = 'mesmo_nome_ate_50m'
        return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'duplicidade_automatica', **ctx['resultado_base']})

    return None


#### 3.18.5.4 — Regra 9: nome morfologicamente equivalente até 50 m.

**Finalidade.** Esta regra avalia especificamente `nome morfologicamente equivalente até 50 m.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 9 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** definidas dinamicamente conforme datas e similaridade.

**Decisões possíveis:** `duplicidade_automatica`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_09(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_09(ctx):
    if ctx['canie_diferente'] and ctx['mesmo_nome_morfologico'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 50) and (ctx['situacao_ordinal'] in ['sem_ordinal', 'ordinal_igual']):
        if ctx['datas_muito_distantes']:
            classe = 'nome_morfologico_equivalente_ate_50m_datas_muito_distantes_possivel_recadastro'
        elif ctx['datas_distantes']:
            classe = 'nome_morfologico_equivalente_ate_50m_datas_distantes_possivel_recadastro'
        elif ctx['datas_proximas']:
            classe = 'nome_morfologico_equivalente_ate_50m_datas_proximas_possivel_carga_duplicada'
        else:
            classe = 'nome_morfologico_equivalente_ate_50m'
        return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'duplicidade_automatica', **ctx['resultado_base']})

    return None


#### 3.18.5.5 — Regra 10: mesmo código técnico, CANIE diferente e distância até 100 m.

**Finalidade.** Esta regra avalia especificamente `mesmo código técnico, canie diferente e distância até 100 m.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 10 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** definidas dinamicamente conforme datas e similaridade.

**Decisões possíveis:** `duplicidade_automatica`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_10(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_10(ctx):
    if ctx['canie_diferente'] and ctx['mesmo_codigo'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 100):
        if ctx['datas_muito_distantes']:
            classe = 'mesmo_codigo_tecnico_ate_100m_datas_muito_distantes'
        elif ctx['datas_distantes']:
            classe = 'mesmo_codigo_tecnico_ate_100m_datas_distantes'
        elif ctx['datas_proximas']:
            classe = 'mesmo_codigo_tecnico_ate_100m_datas_proximas'
        else:
            classe = 'mesmo_codigo_tecnico_ate_100m'
        return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'duplicidade_automatica', **ctx['resultado_base']})

    return None


### 3.18.6 — Visão geral: regras 11 a 13: casos ambíguos que exigem cautela

As regras 11 a 13 tratam situações em que há indício de duplicidade, mas também há risco de eliminar cavidades distintas.

A Regra 11 envia para revisão manual casos com tipo morfológico não equivalente, mesmo nome-base e distância até 20 m. Isso cobre casos como `Abrigo x Gruta`, `Abismo x Abrigo` ou `Paleotoca x Gruta`.

A Regra 12 envia para revisão manual casos com mesmo nome e distância entre 50 m e 100 m, pois a distância ultrapassa o limite de remoção automática.

A Regra 13 avalia pares com mesmo nome-base significativo e localização muito próxima. Dependendo da similaridade textual e da diferença temporal, o par pode ser `nao_remover` ou `revisar_manual`.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


#### 3.18.6.1 — Regra 11: tipo morfológico não equivalente, nome-base igual e distância baixa.

**Finalidade.** Esta regra avalia especificamente `tipo morfológico não equivalente, nome-base igual e distância baixa.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 11 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `tipo_morfologico_nao_equivalente_nome_base_igual`.

**Decisões possíveis:** `revisar_manual`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_11(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_11(ctx):
    if ctx['canie_diferente'] and ctx['tipo_morfologico_diferente'] and ctx['mesmo_nome_base'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 20):
        return pd.Series({'classe_suspeita': 'tipo_morfologico_nao_equivalente_nome_base_igual', 'decisao_preliminar': 'revisar_manual', **ctx['resultado_base']})

    return None


#### 3.18.6.2 — Regra 12: mesmo nome, mas distância entre 50 m e 100 m.

**Finalidade.** Esta regra avalia especificamente `mesmo nome, mas distância entre 50 m e 100 m.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 12 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** definidas dinamicamente conforme datas e similaridade.

**Decisões possíveis:** `revisar_manual`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_12(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_12(ctx):
    if ctx['canie_diferente'] and ctx['mesmo_nome'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 100):
        if ctx['datas_muito_distantes']:
            classe = 'mesmo_nome_50a100m_datas_muito_distantes_possivel_recadastro'
        elif ctx['datas_distantes']:
            classe = 'mesmo_nome_50a100m_datas_distantes_possivel_recadastro'
        elif ctx['datas_proximas']:
            classe = 'mesmo_nome_50a100m_datas_proximas'
        else:
            classe = 'mesmo_nome_50a100m'
        return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'revisar_manual', **ctx['resultado_base']})

    return None


#### 3.18.6.3 — Regra 13: mesmo nome-base significativo e localização muito próxima.

**Finalidade.** Esta regra avalia especificamente `mesmo nome-base significativo e localização muito próxima.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 13 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `mesmo_nome_base_datas_proximas_baixa_similaridade`, `mesmo_nome_base_sem_evidencia_suficiente`.

**Decisões possíveis:** `nao_remover`, `revisar_manual`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_13(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_13(ctx):
    if ctx['canie_diferente'] and ctx['mesmo_nome_base'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 20):
        if (ctx['datas_proximas'] or ctx['datas_moderadamente_proximas']) and ctx['similaridade_nome'] < 0.88:
            return pd.Series({'classe_suspeita': 'mesmo_nome_base_datas_proximas_baixa_similaridade', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})
        if ctx['datas_muito_distantes'] or ctx['datas_distantes'] or ctx['similaridade_nome'] >= 0.88:
            if ctx['datas_muito_distantes']:
                classe = 'mesmo_nome_base_revisar_datas_muito_distantes'
            elif ctx['datas_distantes']:
                classe = 'mesmo_nome_base_revisar_datas_distantes'
            elif ctx['similaridade_nome'] >= 0.88:
                classe = 'mesmo_nome_base_revisar_alta_similaridade'
            else:
                classe = 'mesmo_nome_base_revisar'
            return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'revisar_manual', **ctx['resultado_base']})
        return pd.Series({'classe_suspeita': 'mesmo_nome_base_sem_evidencia_suficiente', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


### 3.18.7 — Visão geral: regras 14 a 18: relações textuais fortes, mesma localização e casos residuais

As regras 14 a 18 tratam situações mais específicas.

A Regra 14 transforma em duplicidade automática casos com relação textual forte, sem conflito de série, distância até 50 m, datas muito distantes e similaridade alta.

A Regra 15 trata nomes muito semelhantes e localização muito próxima. Com datas próximas, o script tende a não remover. Com datas distantes ou similaridade extrema, envia para revisão manual.

A Regra 16 trata pares com mesma localização, mas nomes diferentes. A mesma coordenada isoladamente não é suficiente para remoção.

A Regra 17 evita revisão manual quando há localização próxima, mas nenhuma relação textual suficiente.

A Regra 18 funciona como caso residual para relação textual forte, mantendo revisão manual quando a evidência temporal sugere possível recadastro.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


#### 3.18.7.1 — Regra 14: relação textual forte sem conflito de série.

**Finalidade.** Esta regra avalia especificamente `relação textual forte sem conflito de série.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 14 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `relacao_textual_forte_sem_conflito_ate_50m_datas_muito_distantes`.

**Decisões possíveis:** `duplicidade_automatica`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_14(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_14(ctx):
    if ctx['canie_diferente'] and ctx['relacao_textual_minima'] and (not ctx['tipo_morfologico_diferente']) and (not ctx['sufixo_ordinal_diferente']) and (not ctx['ordinal_presente_em_apenas_um_nome']) and (not ctx['codigo_tecnico_diferente']) and (not ctx['codigo_com_letra_final_diferente']) and (not ctx['mesma_base_numero_letra_opcional']) and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 50) and ctx['datas_muito_distantes'] and (ctx['similaridade_nome'] >= 0.9):
        return pd.Series({'classe_suspeita': 'relacao_textual_forte_sem_conflito_ate_50m_datas_muito_distantes', 'decisao_preliminar': 'duplicidade_automatica', **ctx['resultado_base']})

    return None


#### 3.18.7.2 — Regra 15: nomes muito semelhantes e localização muito próxima.

**Finalidade.** Esta regra avalia especificamente `nomes muito semelhantes e localização muito próxima.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 15 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `nome_muito_semelhante_localizacao_muito_proxima_datas_proximas`, `nome_muito_semelhante_sem_evidencia_suficiente`.

**Decisões possíveis:** `nao_remover`, `revisar_manual`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_15(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_15(ctx):
    if ctx['canie_diferente'] and ctx['similaridade_nome'] >= 0.92 and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 20) and (not ctx['sufixo_ordinal_diferente']) and (not ctx['ordinal_presente_em_apenas_um_nome']) and (not ctx['codigo_tecnico_diferente']) and (not ctx['codigo_com_letra_final_diferente']) and (not ctx['mesma_base_numero_letra_opcional']):
        if ctx['datas_proximas'] or ctx['datas_moderadamente_proximas']:
            return pd.Series({'classe_suspeita': 'nome_muito_semelhante_localizacao_muito_proxima_datas_proximas', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})
        if ctx['datas_distantes'] or ctx['datas_muito_distantes'] or ctx['similaridade_nome'] >= 0.96:
            if ctx['datas_muito_distantes']:
                classe = 'nome_muito_semelhante_revisar_datas_muito_distantes'
            elif ctx['datas_distantes']:
                classe = 'nome_muito_semelhante_revisar_datas_distantes'
            else:
                classe = 'nome_muito_semelhante_revisar_similaridade_extrema'
            return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'revisar_manual', **ctx['resultado_base']})
        return pd.Series({'classe_suspeita': 'nome_muito_semelhante_sem_evidencia_suficiente', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.7.3 — Regra 16: mesma localização, mas nomes diferentes.

**Finalidade.** Esta regra avalia especificamente `mesma localização, mas nomes diferentes.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 16 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `mesma_localizacao_nomes_diferentes`, `mesma_localizacao_nomes_diferentes_datas_proximas`.

**Decisões possíveis:** `nao_remover`, `revisar_manual`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_16(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_16(ctx):
    if ctx['canie_diferente'] and ctx['distancia'] <= 1 and (not ctx['mesmo_nome']):
        if ctx['datas_proximas'] or ctx['datas_moderadamente_proximas']:
            return pd.Series({'classe_suspeita': 'mesma_localizacao_nomes_diferentes_datas_proximas', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})
        if ctx['relacao_textual_minima'] and ctx['similaridade_nome'] >= 0.88 and ctx['datas_distantes'] and (not ctx['sufixo_ordinal_diferente']) and (not ctx['ordinal_presente_em_apenas_um_nome']) and (not ctx['codigo_tecnico_diferente']) and (not ctx['codigo_com_letra_final_diferente']) and (not ctx['mesma_base_numero_letra_opcional']):
            if ctx['datas_muito_distantes']:
                classe = 'mesma_localizacao_nomes_relacionados_datas_muito_distantes'
            else:
                classe = 'mesma_localizacao_nomes_relacionados_datas_distantes'
            return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'revisar_manual', **ctx['resultado_base']})
        return pd.Series({'classe_suspeita': 'mesma_localizacao_nomes_diferentes', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.7.4 — Regra 17: localização muito próxima, mas sem relação textual suficiente.

**Finalidade.** Esta regra avalia especificamente `localização muito próxima, mas sem relação textual suficiente.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 17 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** definidas dinamicamente conforme datas e similaridade.

**Decisões possíveis:** `nao_remover`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_17(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_17(ctx):
    if ctx['canie_diferente'] and ctx['distancia'] <= 10 and (not ctx['relacao_textual_minima']):
        if ctx['datas_proximas'] or ctx['datas_moderadamente_proximas']:
            classe = 'localizacao_muito_proxima_sem_relacao_textual_datas_proximas'
        elif ctx['datas_muito_distantes']:
            classe = 'localizacao_muito_proxima_sem_relacao_textual_datas_muito_distantes'
        else:
            classe = 'localizacao_muito_proxima_sem_relacao_textual'
        return pd.Series({'classe_suspeita': classe, 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})

    return None


#### 3.18.7.5 — Regra 18: caso residual com relação textual forte.

**Finalidade.** Esta regra avalia especificamente `caso residual com relação textual forte.` usando as evidências já armazenadas no contexto do par.

**Prioridade.** Ela ocupa a posição 18 na sequência. Se suas condições forem satisfeitas, a função retorna imediatamente a classificação; caso contrário, retorna `None` e permite que a regra seguinte seja testada. Essa ordem evita que uma regra genérica substitua uma proteção ou evidência mais específica.

**Classes possíveis neste bloco:** `relacao_textual_forte_residual_ate_50m_datas_muito_distantes`.

**Decisões possíveis:** `revisar_manual`.

O retorno inclui também `resultado_base`, permitindo auditar quais evidências textuais, temporais e morfológicas acompanharam a decisão.

**Contrato técnico com o código.** Esta célula define a função `aplicar_regra_18(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_regra_18(ctx):
    if ctx['canie_diferente'] and ctx['relacao_textual_minima'] and ctx['mesma_uf'] and ctx['mesmo_municipio'] and (ctx['distancia'] <= 50) and ctx['datas_muito_distantes'] and (ctx['similaridade_nome'] >= 0.94) and (not ctx['sufixo_ordinal_diferente']) and (not ctx['ordinal_presente_em_apenas_um_nome']) and (not ctx['codigo_tecnico_diferente']) and (not ctx['codigo_com_letra_final_diferente']) and (not ctx['mesma_base_numero_letra_opcional']):
        return pd.Series({'classe_suspeita': 'relacao_textual_forte_residual_ate_50m_datas_muito_distantes', 'decisao_preliminar': 'revisar_manual', **ctx['resultado_base']})

    return None


### 3.18.8 — Visão geral: caso residual sem evidência suficiente

Se nenhuma das regras anteriores for satisfeita, o par é classificado como baixa suspeita ou possível homonímia.

A decisão preliminar nesse caso é:

`nao_remover`

Isso significa que o par não apresentou evidência suficiente para remoção automática nem para revisão prioritária.

**Posição na precedência.** Esta é a saída conservadora alcançada somente depois de testar todas as regras específicas de duplicidade, revisão e distinção. Portanto, `nao_remover` não prova que os registros representam cavidades diferentes; informa apenas que as evidências disponíveis não autorizam outra ação.

**Consequência.** O par permanece documentado na tabela de auditoria, mas não forma grupo automático de remoção. Essa escolha protege a base consolidada contra exclusões sustentadas apenas por similaridade fraca ou contexto cadastral insuficiente.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


#### 3.18.8.1 — Caso residual: baixa suspeita ou homonímia

Este bloco é alcançado somente quando nenhuma das 18 regras anteriores encontrou evidência suficiente para remoção automática ou revisão prioritária.

O par recebe `baixa_suspeita_ou_homonimia` e `nao_remover`. A decisão é conservadora: estar na mesma lista de candidatos não é, por si só, evidência de duplicidade.

A existência de uma função residual separada torna explícito o comportamento padrão e impede retornos nulos na classificação.

**Contrato técnico com o código.** Esta célula define a função `aplicar_caso_residual(ctx)`. Ela recebe `ctx` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def aplicar_caso_residual(ctx):
    return pd.Series({'classe_suspeita': 'baixa_suspeita_ou_homonimia', 'decisao_preliminar': 'nao_remover', **ctx['resultado_base']})


## 3.18.9 — Função coordenadora da classificação

A função pública `classificar_suspeita` agora apenas coordena o processo.

Ela cria o contexto cadastral e temporal, acrescenta as evidências textuais, monta os campos comuns e percorre as 18 funções de regra na ordem original. O primeiro resultado diferente de `None` encerra a avaliação. Se nenhuma regra for acionada, aplica o caso residual.

Essa estrutura preserva a prioridade decisória do script anterior, mas permite testar, documentar e manter cada regra isoladamente.

**Contrato técnico com o código.** Esta célula define a função `classificar_suspeita(row)`. Ela recebe `row` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def classificar_suspeita(row):
    ctx = criar_contexto_cadastral_temporal(row)
    ctx = enriquecer_contexto_textual(ctx)
    ctx = adicionar_resultado_base_ao_contexto(ctx)

    regras = (
        aplicar_regra_01,
        aplicar_regra_02,
        aplicar_regra_03,
        aplicar_regra_04,
        aplicar_regra_05,
        aplicar_regra_06,
        aplicar_regra_07,
        aplicar_regra_08,
        aplicar_regra_09,
        aplicar_regra_10,
        aplicar_regra_11,
        aplicar_regra_12,
        aplicar_regra_13,
        aplicar_regra_14,
        aplicar_regra_15,
        aplicar_regra_16,
        aplicar_regra_17,
        aplicar_regra_18
    )

    for regra in regras:
        resultado = regra(ctx)
        if resultado is not None:
            return resultado

    return aplicar_caso_residual(ctx)


## 3.19 — Execução da classificação dos pares

Depois de definidas todas as funções, o script executa a classificação dos pares.

Se `df_pares` estiver vazio, é criado um DataFrame vazio chamado `df_pares_suspeitos` e a etapa é encerrada.

Se houver pares candidatos, o script:

1. classifica a distância de cada par;
2. aplica a função `classificar_suspeita` linha a linha;
3. concatena os resultados ao `df_pares`;
4. cria `df_pares_suspeitos`, contendo apenas os pares classificados como:
   - `duplicidade_automatica`;
   - `revisar_manual`.

Os pares classificados como `nao_remover` permanecem em `df_pares`, mas não entram em `df_pares_suspeitos`.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_pares_suspeitos`, `df_pares`, `classificacoes`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if df_pares.empty:
    df_pares_suspeitos = pd.DataFrame()

    print("Etapa 3 concluída.")
    print("Não há pares candidatos para classificar.")
else:
    df_pares["classe_distancia"] = df_pares["distancia_metros"].apply(
        classificar_distancia
    )

    classificacoes = df_pares.apply(
        classificar_suspeita,
        axis=1
    )

    df_pares = pd.concat(
        [df_pares, classificacoes],
        axis=1
    )

    df_pares_suspeitos = df_pares[
        df_pares["decisao_preliminar"].isin([
            "duplicidade_automatica",
            "revisar_manual"
        ])
    ].copy()

    print("Etapa 3 concluída.")
    print("Resumo das decisões preliminares:")
    print(df_pares["decisao_preliminar"].value_counts(dropna=False))

    print("\nResumo das classes de suspeita:")
    print(df_pares["classe_suspeita"].value_counts(dropna=False))

    print("\nResumo da diferença de dias entre cadastros nos pares suspeitos:")
    if not df_pares_suspeitos.empty and "diferenca_dias_cadastro" in df_pares_suspeitos.columns:
        print(df_pares_suspeitos["diferenca_dias_cadastro"].describe())
    else:
        print("Sem pares suspeitos com diferença de datas calculada.")


## Síntese da Etapa 3

### Resultado produzido

Cada par recebe `classe_suspeita`, `decisao_preliminar` e campos auxiliares que explicam a decisão. `df_pares_suspeitos` restringe o universo aos casos que exigem remoção automática ou avaliação humana.

### Verificações esperadas

- toda linha de `df_pares` deve receber uma decisão válida;
- regras protetivas devem ser avaliadas antes das regras de remoção;
- `duplicidade_automatica` deve depender de evidência forte e explícita;
- `revisar_manual` não deve provocar exclusão sem decisão posterior.

### Ligação com a etapa seguinte

A Etapa 3.20 permite decidir manualmente parte das pendências. Depois, a Etapa 4 transforma pares confirmados em grupos transitivos.

**Natureza desta célula.** Esta síntese encerra o classificador automático e não executa código.


# Etapa 3.20 — Registro controlado das decisões manuais

## Finalidade

Nem todo indício pode ser resolvido com segurança por regra automática. Esta etapa oferece uma interface explícita para decidir os pares classificados como `revisar_manual`, preservando a autoria e a rastreabilidade da decisão.

## Entradas e escolhas permitidas

O processo utiliza `df_pares`, `df_pares_suspeitos` e `COL_NOME`. O usuário pode declarar que um código `REV_MAN_...` representa cavidades diferentes ou que representa a mesma cavidade, indicando neste segundo caso qual CANIE deve ser removido. Pares não informados permanecem pendentes.

## Processamento

Os pares pendentes recebem identificadores legíveis e uma visualização reduzida com nomes, CANIEs, coordenadas, datas e distância. As decisões informadas são validadas contra os códigos existentes, aplicadas sobre uma cópia da tabela de pares e registradas em tabela própria de auditoria.

## Produtos

Os principais resultados são `df_pares_31`, `canies_remover_manual_31`, `df_decisoes_manuais_31` e as tabelas de visualização. Esta etapa registra decisões; a remoção no nível dos registros ocorre somente nas Etapas 4 e 6.

## Como ler a etapa

Os tópicos 3.20.1 e 3.20.2 configuram e validam a entrada humana. Os tópicos 3.20.3 a 3.20.6 preparam a visualização. Os tópicos 3.20.7 a 3.20.13 aplicam, auditam e resumem as decisões.

**Natureza desta célula.** Esta abertura documenta o procedimento manual e não executa código. A execução começa em 3.20.1.


## 3.20.1 — Decisões manuais definidas pelo autor

Este bloco registra a decisão final dos pares que a classificação automática encaminhou para `revisar_manual`. Após a análise dos nomes, das coordenadas, das distâncias e do contexto cadastral, o autor optou por considerar os 13 pares como representações duplicadas da mesma cavidade.

`PARES_CAVIDADES_DIFERENTES_31` permanece vazio porque nenhum dos pares foi confirmado como duas cavidades independentes. Em `PARES_MESMA_CAVIDADE_REMOVER_31`, cada chave `REV_MAN_...` identifica o par avaliado e a lista associada informa explicitamente o número CANIE que deve ser retirado da base consolidada.

### Critério de preservação adotado

Quando os registros empregam tipos morfológicos diferentes, foi adotada a seguinte ordem de prioridade:

`Abismo > Caverna > Gruta > Toca > Abrigo`

O registro associado ao termo de maior prioridade é preservado, enquanto o de menor prioridade é indicado para remoção. O par `REV_MAN_00006`, que não depende dessa hierarquia, mantém a decisão anteriormente documentada de preservar `012192.04920.31.46503` e remover `012206.04934.31.46503`.

### Responsabilidade e limite da decisão

Esta classificação constitui uma escolha metodológica do autor, realizada a partir das informações disponíveis na base e da análise remota dos pares. A coincidência espacial e a semelhança cadastral fornecem evidência para o tratamento como duplicidade, mas não demonstram de forma absoluta que os registros representam a mesma feição natural.

Quando houver dúvida geográfica, morfológica ou cadastral, recomenda-se revisão documental complementar e, se tecnicamente viável, verificação presencial em campo. Uma inspeção presencial pode confirmar se os pontos representam a mesma cavidade, entradas distintas de um mesmo sistema ou feições naturais independentes. Caso surja nova evidência, as decisões desta lista devem ser revistas antes de uma nova consolidação definitiva.

**Contrato técnico com o código.** A célula seguinte define `PARES_CAVIDADES_DIFERENTES_31` e `PARES_MESMA_CAVIDADE_REMOVER_31`. Esses objetos não removem linhas imediatamente; eles registram a decisão autoral que será validada, incorporada a `df_pares_31` e aplicada aos registros nas Etapas 4 e 6.


In [ ]:
PARES_CAVIDADES_DIFERENTES_31 = []

PARES_MESMA_CAVIDADE_REMOVER_31 = {
    "REV_MAN_00001": ["019780.00529.29.19157"],
    "REV_MAN_00002": ["005528.00015.51.03379"],
    "REV_MAN_00003": ["023894.07639.31.06200"],
    "REV_MAN_00004": ["017799.07591.31.17504"],
    "REV_MAN_00005": ["035773.11051.31.41108"],
    "REV_MAN_00006": ["012206.04934.31.46503"],
    "REV_MAN_00007": ["013410.05773.31.53608"],
    "REV_MAN_00008": ["014819.07112.31.59001"],
    "REV_MAN_00009": ["014852.07142.31.59001"],
    "REV_MAN_00010": ["026314.00065.26.15805"],
    "REV_MAN_00011": ["016623.00636.24.04309"],
    "REV_MAN_00012": ["019525.00001.28.07204"],
    "REV_MAN_00013": ["020033.00664.29.14406"]
}


## 3.20.2 — Validação mínima dos objetos necessários

Antes de executar a revisão manual, o script verifica se os objetos necessários já existem no ambiente do notebook.

Os objetos exigidos são:

- `df_pares`;
- `df_pares_suspeitos`;
- `COL_NOME`.

Essa verificação impede que a Etapa 3.20 seja executada antes da Etapa 3.

Se algum objeto estiver ausente, o script interrompe a execução e informa quais objetos estão faltando. Isso evita erros menos claros nas etapas seguintes.

**Contrato técnico com o código.** O bloco cria ou atualiza `objetos_necessarios_31`, `objetos_ausentes_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
objetos_necessarios_31 = [
    "df_pares",
    "df_pares_suspeitos",
    "COL_NOME"
]

objetos_ausentes_31 = [
    objeto for objeto in objetos_necessarios_31
    if objeto not in globals()
]

if len(objetos_ausentes_31) > 0:
    raise NameError(
        "A Etapa 3.20 deve ser executada depois da Etapa 3. "
        f"Objetos ausentes: {objetos_ausentes_31}"
    )


## 3.20.3 — Seleção dos pares efetivamente pendentes

A função recebe a tabela de pares suspeitos e mantém somente `revisar_manual`. Se a tabela estiver vazia, devolve um DataFrame vazio.

Essa separação evita misturar duplicidades automáticas com casos que realmente exigem intervenção humana.

**Entrada e saída.** `df_pares_suspeitos` vem da classificação da Etapa 3. O filtro é aplicado sobre `decisao_preliminar` e o `.copy()` isola o subconjunto para que códigos e decisões manuais possam ser acrescentados sem modificar acidentalmente a tabela de origem.

**Caso vazio.** O retorno antecipado mantém o fluxo executável quando nenhum par exige revisão. As células seguintes conseguem então criar visualizações e relatórios vazios com esquema controlado.

Neste bloco, o script cria a tabela `df_revisao_manual_31`.

Essa tabela é formada a partir de `df_pares_suspeitos`, mantendo apenas os pares cuja decisão preliminar é:

`revisar_manual`

Ou seja, entram aqui apenas os pares que a Etapa 3 considerou suspeitos, mas sem segurança suficiente para remoção automática.

Se não houver pares classificados como `revisar_manual`, a etapa cria tabelas vazias e informa que não há casos para revisar.

**Contrato técnico com o código.** O bloco cria ou atualiza `selecionar_revisao_manual`, `df_revisao_manual_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
def selecionar_revisao_manual(df_pares_suspeitos):
    if df_pares_suspeitos.empty:
        return pd.DataFrame()
    return df_pares_suspeitos[
        df_pares_suspeitos["decisao_preliminar"] == "revisar_manual"
    ].copy()

df_revisao_manual_31 = selecionar_revisao_manual(df_pares_suspeitos)


## 3.20.4 — Identificação legível e coordenadas de comparação

A tabela é reindexada e cada par recebe um código `REV_MAN_...`. Em seguida, latitude e longitude são combinadas em campos legíveis e a distância é arredondada.

Esses campos são exclusivamente de apresentação; IDs, coordenadas originais e decisões permanecem preservados na tabela completa.

Quando existem pares para revisar, o script reinicia o índice da tabela e cria a coluna:

`codigo_revisao_manual`

Essa coluna recebe códigos sequenciais no formato:

- `REV_MAN_00001`;
- `REV_MAN_00002`;
- `REV_MAN_00003`.

Esses códigos tornam a revisão manual mais prática, porque o usuário pode se referir a cada par por um identificador curto, em vez de manipular diretamente `id_linha_a`, `id_linha_b` ou os números CANIE.

Esses códigos são os mesmos que serão usados nas listas de decisão manual.

Neste bloco é criada a tabela `df_revisao_manual_visual_31`.

Ela é uma versão simplificada de `df_revisao_manual_31`, feita para facilitar a decisão humana.

A visualização mostra apenas os campos mais importantes para análise rápida:

- código da revisão manual;
- CANIE do registro A;
- nome da cavidade A;
- coordenada A;
- data de cadastro A;
- CANIE do registro B;
- nome da cavidade B;
- coordenada B;
- data de cadastro B;
- distância em metros.

As coordenadas são montadas no formato:

`latitude, longitude`

A distância é arredondada para duas casas decimais.

**Contrato técnico com o código.** Esta célula define a função `adicionar_campos_visuais_revisao(df_revisao)`. Ela recebe `df_revisao` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def adicionar_campos_visuais_revisao(df_revisao):
    df_visual = df_revisao.reset_index(drop=True).copy()
    df_visual["codigo_revisao_manual"] = [
        f"REV_MAN_{i + 1:05d}" for i in range(len(df_visual))
    ]
    df_visual["coordenada_a"] = (
        df_visual["latitude_num_a"].round(6).astype(str) + ", "
        + df_visual["longitude_num_a"].round(6).astype(str)
    )
    df_visual["coordenada_b"] = (
        df_visual["latitude_num_b"].round(6).astype(str) + ", "
        + df_visual["longitude_num_b"].round(6).astype(str)
    )
    df_visual["distancia_metros"] = df_visual["distancia_metros"].round(2)
    return df_visual


## 3.20.5 — Seleção e renomeação da visualização

A função reduz a tabela aos campos essenciais para decisão: códigos, CANIEs, nomes, coordenadas, datas e distância.

Os nomes técnicos com sufixos `_a` e `_b` são substituídos por rótulos mais claros, sem alterar a tabela completa usada no processamento.

**Processamento.** A lista inicial define a ordem desejada das informações. A segunda seleção conserva apenas colunas realmente existentes, tornando a visualização tolerante a pequenas variações do esquema. Por fim, `rename` troca nomes internos por rótulos próprios para leitura humana.

**Limite desta transformação.** Nenhuma decisão é tomada e nenhum valor cadastral é alterado. O DataFrame retornado é uma projeção de apoio; `df_pares_31` e a base de trabalho continuam usando os campos técnicos completos.

Após criar os campos de coordenada, o script seleciona apenas as colunas necessárias para a visualização.

Em seguida, algumas colunas são renomeadas para nomes mais claros:

- `canie_txt_a` vira `canie_a`;
- `Nome_a` vira `nome_caverna_a`;
- `data_cadastro_dt_a` vira `data_cadastro_a`;
- `canie_txt_b` vira `canie_b`;
- `Nome_b` vira `nome_caverna_b`;
- `data_cadastro_dt_b` vira `data_cadastro_b`.

O uso de `f"{COL_NOME}_a"` e `f"{COL_NOME}_b"` permite que o script funcione mesmo se o nome da coluna original de nome da cavidade estiver parametrizado em `COL_NOME`.

**Contrato técnico com o código.** Esta célula define a função `selecionar_colunas_visuais_revisao(df_visual)`. Ela recebe `df_visual` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def selecionar_colunas_visuais_revisao(df_visual):
    colunas = [
        "codigo_revisao_manual", "canie_txt_a", f"{COL_NOME}_a",
        "coordenada_a", "data_cadastro_dt_a", "canie_txt_b",
        f"{COL_NOME}_b", "coordenada_b", "data_cadastro_dt_b",
        "distancia_metros"
    ]
    colunas = [coluna for coluna in colunas if coluna in df_visual.columns]
    return df_visual[colunas].copy().rename(columns={
        "canie_txt_a": "canie_a", f"{COL_NOME}_a": "nome_caverna_a",
        "data_cadastro_dt_a": "data_cadastro_a", "canie_txt_b": "canie_b",
        f"{COL_NOME}_b": "nome_caverna_b", "data_cadastro_dt_b": "data_cadastro_b"
    })


## 3.20.6 — Montagem e exibição da revisão manual

Se não houver pares pendentes, são criadas tabelas vazias e a etapa informa essa condição. Caso contrário, as funções anteriores montam a visualização e a exibem.

A tabela completa recebe os mesmos códigos de revisão antes de seguir para as decisões manuais posteriores.

Neste bloco, a tabela simplificada é exibida no notebook com `display`.

Essa tabela é a principal referência visual para o preenchimento das decisões manuais.

O usuário deve olhar cada linha, verificar os nomes, coordenadas, datas e distância, e então decidir se o par representa:

1. cavidades diferentes;
2. a mesma cavidade, com remoção de um CANIE;
3. caso ainda pendente para revisão/in loco.

Se um par não for informado nas listas manuais, ele permanecerá com a decisão original `revisar_manual`.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_revisao_manual_31`, `df_revisao_manual_visual_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if df_revisao_manual_31.empty:
    df_revisao_manual_31 = pd.DataFrame()
    df_revisao_manual_visual_31 = pd.DataFrame()
    print("Etapa 3.20 concluída.")
    print("Não há pares classificados como revisar_manual.")
else:
    df_revisao_manual_31 = adicionar_campos_visuais_revisao(df_revisao_manual_31)
    df_revisao_manual_visual_31 = selecionar_colunas_visuais_revisao(
        df_revisao_manual_31.copy()
    )
    print("Etapa 3.20 — Visualização simplificada dos pares para decisão manual:")
    display(df_revisao_manual_visual_31)
    print(f"Total de pares para revisar manualmente: {len(df_revisao_manual_visual_31):,}")


## 3.20.7 — Criação da tabela `df_pares_31`

Neste bloco, o script cria `df_pares_31`, uma cópia de `df_pares`.

Essa nova tabela recebe três colunas adicionais:

- `decisao_final_31`;
- `codigo_revisao_manual`;
- `observacao_manual_31`.

Inicialmente, `decisao_final_31` recebe o mesmo valor de `decisao_preliminar`.

Isso significa que, se o usuário não informar nenhuma decisão manual, o comportamento original da Etapa 3 é preservado.

A coluna `codigo_revisao_manual` será preenchida apenas nos pares que vieram de `df_revisao_manual_31`.

A coluna `observacao_manual_31` registra a justificativa textual da decisão manual.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_pares_31`, `canies_remover_manual_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df_pares_31 = df_pares.copy()

df_pares_31["decisao_final_31"] = df_pares_31["decisao_preliminar"]
df_pares_31["codigo_revisao_manual"] = ""
df_pares_31["observacao_manual_31"] = ""

canies_remover_manual_31 = []


## 3.20.8 — Associação dos códigos de revisão à tabela de pares

Quando existem pares para revisão manual, o script cria um mapa contendo:

- `id_linha_a`;
- `id_linha_b`;
- `codigo_revisao_manual`.

Esse mapa é unido a `df_pares_31`.

Com isso, cada par classificado como `revisar_manual` passa a carregar seu código de revisão, como `REV_MAN_00001`.

O uso de `combine_first` preserva a estrutura da coluna `codigo_revisao_manual` e preenche apenas os valores vindos da tabela de revisão.

**Contrato técnico com o código.** O bloco cria ou atualiza `mapa_codigo_revisao`, `df_pares_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if not df_revisao_manual_31.empty:
    mapa_codigo_revisao = df_revisao_manual_31[
        [
            "id_linha_a",
            "id_linha_b",
            "codigo_revisao_manual"
        ]
    ].copy()

    df_pares_31 = df_pares_31.merge(
        mapa_codigo_revisao,
        on=[
            "id_linha_a",
            "id_linha_b"
        ],
        how="left",
        suffixes=("", "_novo")
    )

    df_pares_31["codigo_revisao_manual"] = df_pares_31[
        "codigo_revisao_manual_novo"
    ].combine_first(
        df_pares_31["codigo_revisao_manual"]
    )

    df_pares_31 = df_pares_31.drop(
        columns=[
            "codigo_revisao_manual_novo"
        ],
        errors="ignore"
    )


## 3.20.9 — Aplicação da decisão: cavidades diferentes

Neste bloco, o script aplica a decisão manual para os pares informados em:

`PARES_CAVIDADES_DIFERENTES_31`

Esses pares recebem a decisão final:

`nao_remover_manual`

Isso significa que o usuário decidiu que os dois registros não devem ser tratados como duplicidade.

Nenhum CANIE é removido por essa decisão.

A coluna `observacao_manual_31` registra que o par foi revisado manualmente e definido como cavidades diferentes.

**Contrato técnico com o código.** O bloco cria ou atualiza `pares_diferentes_31`, `mask_pares_diferentes_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
pares_diferentes_31 = set(PARES_CAVIDADES_DIFERENTES_31)

mask_pares_diferentes_31 = df_pares_31["codigo_revisao_manual"].isin(
    pares_diferentes_31
)

df_pares_31.loc[
    mask_pares_diferentes_31,
    "decisao_final_31"
] = "nao_remover_manual"

df_pares_31.loc[
    mask_pares_diferentes_31,
    "observacao_manual_31"
] = (
    "Par revisado manualmente e definido como cavidades diferentes. "
    "Nenhum registro deve ser removido por este par."
)


## 3.20.10 — Aplicação da decisão: mesma cavidade

Neste bloco, o script aplica a decisão manual para os pares informados em:

`PARES_MESMA_CAVIDADE_REMOVER_31`

Para cada par definido como mesma cavidade, o usuário informa explicitamente qual CANIE deve ser removido.

Esses pares recebem a decisão final:

`duplicidade_manual_mesma_cavidade`

Além disso, os CANIEs indicados são acumulados na lista:

`canies_remover_manual_31`

Essa lista será usada nas etapas seguintes para marcar os registros que devem ser removidos manualmente da base consolidada.

**Contrato técnico com o código.** O bloco cria ou atualiza `canies_remover_manual_31`, `lista_canies`, `mask_codigo`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
for codigo_revisao, lista_canies in PARES_MESMA_CAVIDADE_REMOVER_31.items():
    lista_canies = [
        str(canie).strip()
        for canie in lista_canies
    ]

    canies_remover_manual_31.extend(lista_canies)

    mask_codigo = df_pares_31["codigo_revisao_manual"] == codigo_revisao

    df_pares_31.loc[
        mask_codigo,
        "decisao_final_31"
    ] = "duplicidade_manual_mesma_cavidade"

    df_pares_31.loc[
        mask_codigo,
        "observacao_manual_31"
    ] = (
        "Par revisado manualmente e definido como mesma cavidade. "
        "A remoção seguirá a lista canies_remover_manual_31."
    )

canies_remover_manual_31 = sorted(set(canies_remover_manual_31))


## 3.20.11 — Criação da tabela de auditoria das decisões manuais

Neste bloco é criada a tabela `df_decisoes_manuais_31`.

Ela registra, de forma tabular, as decisões preenchidas pelo usuário no script.

Para pares definidos como cavidades diferentes, a tabela registra:

- código da revisão;
- tipo de decisão manual;
- campo de CANIEs para remoção vazio;
- status `informado_no_script`.

Para pares definidos como mesma cavidade, a tabela registra:

- código da revisão;
- tipo de decisão manual;
- CANIEs indicados para remoção;
- status `informado_no_script`.

Essa tabela é útil para o relatório de auditoria e para rastrear quais decisões foram tomadas manualmente.

**Contrato técnico com o código.** O bloco cria ou atualiza `registros_decisoes_31`, `df_decisoes_manuais_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
registros_decisoes_31 = []

for codigo in PARES_CAVIDADES_DIFERENTES_31:
    registros_decisoes_31.append({
        "codigo_revisao_manual": codigo,
        "tipo_decisao_manual": "cavidades_diferentes",
        "canies_remover": "",
        "status": "informado_no_script"
    })

for codigo, lista_canies in PARES_MESMA_CAVIDADE_REMOVER_31.items():
    registros_decisoes_31.append({
        "codigo_revisao_manual": codigo,
        "tipo_decisao_manual": "mesma_cavidade",
        "canies_remover": " | ".join(lista_canies),
        "status": "informado_no_script"
    })

df_decisoes_manuais_31 = pd.DataFrame(registros_decisoes_31)


## 3.20.12 — Validação dos códigos informados

Neste bloco, o script compara os códigos existentes na tabela de revisão com os códigos informados manualmente.

São criados três conjuntos:

- `codigos_existentes_31`: códigos realmente presentes em `df_revisao_manual_31`;
- `codigos_informados_31`: códigos informados nas listas ou dicionários de decisão manual;
- diferenças entre eles.

A validação identifica dois problemas possíveis:

1. códigos informados que não existem na revisão manual;
2. códigos existentes que ainda não receberam decisão manual.

Essa checagem ajuda a evitar erros de digitação, como informar `REV_MAN_00060` quando o correto seria `REV_MAN_00006`.

**Contrato técnico com o código.** O bloco cria ou atualiza `codigos_existentes_31`, `codigos_informados_31`, `codigos_nao_encontrados_31`, `codigos_sem_decisao_31`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
codigos_existentes_31 = set(
    df_revisao_manual_31["codigo_revisao_manual"]
) if not df_revisao_manual_31.empty else set()

codigos_informados_31 = set(PARES_CAVIDADES_DIFERENTES_31).union(
    set(PARES_MESMA_CAVIDADE_REMOVER_31.keys())
)

codigos_nao_encontrados_31 = sorted(
    codigos_informados_31 - codigos_existentes_31
)

codigos_sem_decisao_31 = sorted(
    codigos_existentes_31 - codigos_informados_31
)

if len(codigos_nao_encontrados_31) > 0:
    print("\nAtenção: os seguintes códigos foram informados, mas não existem na revisão manual:")
    for codigo in codigos_nao_encontrados_31:
        print(f"- {codigo}")

if len(codigos_sem_decisao_31) > 0:
    print("\nAtenção: os seguintes pares ainda não receberam decisão manual:")
    for codigo in codigos_sem_decisao_31:
        print(f"- {codigo}")


## 3.20.13 — Resumo final das decisões manuais

Este bloco encerra a Etapa 3.20 e apresenta uma conferência das decisões que serão consumidas pelas etapas posteriores.

Primeiro, ele mostra a distribuição de `decisao_final_31` em `df_pares_31`. Essa contagem permite verificar quantos pares permaneceram como duplicidade automática, revisão manual ou preservação.

Depois, lista os CANIEs incluídos em `canies_remover_manual_31`. Essa lista é especialmente importante porque a Etapa 6 usa esses identificadores para marcar registros como `duplicado_remover_manual`.

Por fim, exibe `df_decisoes_manuais_31`, que funciona como trilha de auditoria: registra o código de revisão, o tipo de decisão, o CANIE removido e a observação associada. Se nenhuma decisão tiver sido informada, o bloco comunica explicitamente essa condição.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("\nEtapa 3.20 concluída.")

print("\nResumo da decisão final dos pares:")
print(df_pares_31["decisao_final_31"].value_counts(dropna=False))

print("\nCANIEs indicados manualmente para remoção:")
if len(canies_remover_manual_31) == 0:
    print("Nenhum CANIE indicado manualmente para remoção.")
else:
    for canie in canies_remover_manual_31:
        print(f"- {canie}")

print("\nTabela de decisões manuais informadas:")
if df_decisoes_manuais_31.empty:
    print("Nenhuma decisão manual informada no script.")
else:
    display(df_decisoes_manuais_31)


## Síntese da Etapa 3.20

### Resultado produzido

`df_pares_31` contém a decisão efetiva dos pares; `canies_remover_manual_31` registra identificadores autorizados para remoção; `df_decisoes_manuais_31` preserva a trilha das escolhas; e pares não decididos continuam como `revisar_manual`.

### Verificações esperadas

- todo código informado deve existir na visualização de revisão;
- um par não pode ser simultaneamente “cavidades diferentes” e “mesma cavidade”;
- uma decisão de mesma cavidade deve indicar explicitamente o CANIE a remover;
- decisões manuais não devem alterar pares que não foram citados.

### Limite e ligação com a próxima etapa

A decisão registrada não remove linhas por si só. A Etapa 4 usa essas decisões para formar grupos, e a Etapa 6 atribui os status finais aos registros.

**Natureza desta célula.** Esta síntese encerra o procedimento manual e não executa código.


# Etapa 4 — Construção dos grupos transitivos de duplicidade

## Finalidade

Decisões são produzidas no nível dos pares, mas a seleção do registro principal precisa considerar conjuntos completos. Esta etapa converte relações A–B em componentes transitivos: se A se relaciona com B e B com C, os três pertencem ao mesmo grupo.

## Entradas

O processo prioriza `df_pares_31`, quando a decisão manual foi executada, e utiliza a coluna de decisão final disponível. Também recebe a lista de CANIEs explicitamente indicados para remoção manual.

## Processamento e produtos

`UnionFind` constrói dois universos independentes. `grupo_duplicidade_auto` contém apenas duplicidades confirmadas e orientará a escolha do principal. `grupo_duplicidade_revisao` inclui também pendências e serve para auditoria espacial. Os grupos recebem identificadores determinísticos, e a marca de remoção manual é propagada para `df`.

## Como ler a etapa

O tópico 4.1 define a estrutura de agrupamento; 4.2 seleciona a decisão; 4.3 forma os grupos automáticos; 4.4 forma os grupos ampliados; 4.5 e 4.6 propagam e conferem os resultados.

**Natureza desta célula.** Esta abertura explica por que pares precisam ser transformados em grupos. A execução começa em 4.1.


## 4.1 — Estrutura `UnionFind` para formar grupos transitivos

A Etapa 3 classifica pares, mas uma duplicidade pode envolver mais de dois registros. A classe `UnionFind` transforma vínculos par-a-par em componentes conectados.

Se A está ligado a B e B está ligado a C, `union` coloca A, B e C no mesmo conjunto, mesmo que A e C nunca tenham sido comparados diretamente. O método `find` identifica a raiz de cada conjunto e aplica compressão de caminho para acelerar consultas posteriores.

Essa estrutura será usada separadamente para grupos de duplicidade automática e grupos ampliados de revisão.

**Contrato técnico com o código.** Esta célula define a estrutura `UnionFind`, que encapsula estado e operações reutilizadas posteriormente. A classe é registrada em memória neste ponto; suas instâncias serão criadas apenas quando o agrupamento for iniciado.


In [ ]:
class UnionFind:
    """
    Estrutura para agrupar registros conectados por pares de duplicidade.

    Exemplo:
    se A é duplicado de B, e B é duplicado de C,
    então A, B e C pertencem ao mesmo grupo.
    """

    def __init__(self):
        self.parent = {}

    def find(self, item):
        if item not in self.parent:
            self.parent[item] = item

        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])

        return self.parent[item]

    def union(self, item_a, item_b):
        raiz_a = self.find(item_a)
        raiz_b = self.find(item_b)

        if raiz_a != raiz_b:
            self.parent[raiz_b] = raiz_a


## 4.2 — Seleção da tabela e da coluna de decisão

O fluxo pode chegar à Etapa 4 por dois caminhos. Se a Etapa 3.20 foi executada, `df_pares_31` contém decisões manuais incorporadas e deve ter prioridade. Caso contrário, o script usa `df_pares` e a decisão preliminar da Etapa 3.

`df_pares_tratamento` padroniza essa escolha, enquanto `COL_DECISAO_TRATAMENTO` informa qual coluna representa a decisão válida. As Etapas 4, 6, 9 e 10 passam a trabalhar com essa referência única, evitando duplicar lógica condicional.

**Contrato técnico com o código.** O bloco cria ou atualiza `canies_remover_manual_31`, `df_pares_tratamento`, `COL_DECISAO_TRATAMENTO`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if "df_pares_31" in globals() and "decisao_final_31" in df_pares_31.columns:
    df_pares_tratamento = df_pares_31.copy()
    COL_DECISAO_TRATAMENTO = "decisao_final_31"
else:
    df_pares_tratamento = df_pares.copy()
    COL_DECISAO_TRATAMENTO = "decisao_preliminar"

if "canies_remover_manual_31" not in globals():
    canies_remover_manual_31 = []

canies_remover_manual_31 = [
    str(canie).strip()
    for canie in canies_remover_manual_31
]


## 4.3 — Formação dos grupos de duplicidade automática

Este bloco seleciona pares cuja decisão final indica `duplicidade_automatica` ou `duplicidade_manual_mesma_cavidade`. Cada par une seus dois `id_linha` na estrutura `UnionFind`.

Após processar todos os vínculos, o script reúne os membros por raiz, ordena os grupos e cria códigos legíveis como `DUP_AUTO_00001`. O resultado é gravado em `grupo_duplicidade_auto` para todos os registros envolvidos.

Esses grupos alimentam diretamente a Etapa 5, que escolherá um único registro principal em cada conjunto.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 4.3.1 — União dos pares automáticos

Pares confirmados unem seus IDs em `UnionFind`, formando componentes transitivos.

**Entrada.** O filtro usa `df_pares_tratamento`, definido na célula anterior, e aceita decisões automáticas e confirmações manuais de mesma cavidade. Para cada par selecionado, os identificadores `id_linha_a` e `id_linha_b` são unidos.

**Transitividade.** Se A está unido a B e B a C, `UnionFind` coloca os três no mesmo componente mesmo que o par A–C não exista. Esse comportamento é necessário para representar grupos completos de registros relacionados e alimenta a escolha de um único principal na Etapa 5.

**Contrato técnico com o código.** O bloco cria ou atualiza `uf_auto`, `pares_auto`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
uf_auto = UnionFind()

if not df_pares_tratamento.empty:
    pares_auto = df_pares_tratamento[
        df_pares_tratamento[COL_DECISAO_TRATAMENTO].isin([
            "duplicidade_automatica",
            "duplicidade_manual_mesma_cavidade"
        ])
    ].copy()

    for _, row in pares_auto.iterrows():
        uf_auto.union(
            row["id_linha_a"],
            row["id_linha_b"]
        )


### 4.3.2 — Associação dos registros às raízes

Cada registro participante é relacionado à raiz de seu componente; linhas sem união ficam fora.

**Processamento.** Para cada `id_linha` da base, `find` consulta a raiz canônica no `UnionFind`. O registro entra em `mapa_grupo_auto` somente quando pertence a uma união; registros únicos não recebem grupo artificial.

**Saída intermediária.** O dicionário associa cada membro à raiz interna de seu componente. Essas raízes ainda não são rótulos de apresentação e serão convertidas em códigos estáveis na próxima célula.

**Contrato técnico com o código.** O bloco cria ou atualiza `mapa_grupo_auto`, `raiz`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


**Proteção contra grupos unitários artificiais.** O mapeamento percorre somente as chaves já presentes em `uf_auto.parent`, isto é, IDs introduzidos por uma operação `union`. Consultar todos os registros com `find` criaria pais para elementos isolados e os classificaria incorretamente como grupos de um único membro.


In [ ]:
mapa_grupo_auto = {
    id_linha: uf_auto.find(id_linha)
    for id_linha in sorted(uf_auto.parent)
}


### 4.3.3 — Numeração reproduzível dos grupos

Raízes ordenadas recebem códigos `DUP_AUTO_...`, estabilizando a identificação.

**Reprodutibilidade.** O conjunto de raízes é ordenado antes da enumeração, evitando que a ordem interna do dicionário determine os rótulos. A formatação com cinco dígitos mantém ordenação lexical e leitura uniforme.

**Saída.** `mapa_nome_grupo_auto` traduz cada raiz interna para um identificador `DUP_AUTO_00001`, `DUP_AUTO_00002` e assim por diante. O código identifica o grupo, não o registro principal.

**Contrato técnico com o código.** O bloco cria ou atualiza `raizes_auto`, `mapa_nome_grupo_auto`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
raizes_auto = sorted(set(mapa_grupo_auto.values()))

mapa_nome_grupo_auto = {
    raiz: f"DUP_AUTO_{posicao + 1:05d}"
    for posicao, raiz in enumerate(raizes_auto)
}


### 4.3.4 — Propagação dos grupos automáticos

O código do grupo é mapeado para `df`; registros não envolvidos recebem texto vazio.

**Mapeamento.** Para cada linha, a expressão procura primeiro sua raiz em `mapa_grupo_auto` e depois o rótulo correspondente em `mapa_nome_grupo_auto`. O valor padrão `""` representa participação inexistente em grupo automático.

**Uso posterior.** `grupo_duplicidade_auto` passa a integrar `df` e será usado para escolher o registro principal, atribuir status e produzir as tabelas e camadas de auditoria.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["grupo_duplicidade_auto"] = df["id_linha"].map(
    lambda id_linha: mapa_nome_grupo_auto.get(
        mapa_grupo_auto.get(id_linha),
        ""
    )
)


## 4.4 — Formação dos grupos ampliados de revisão

Os grupos de revisão incluem tanto duplicidades confirmadas quanto pares que ainda permanecem como `revisar_manual`. Essa visão ampliada ajuda a enxergar cadeias em que registros confirmados e pendentes pertencem ao mesmo contexto espacial ou cadastral.

O procedimento repete a lógica transitiva do `UnionFind`, mas grava códigos `DUP_REV_...` em `grupo_duplicidade_revisao`. Esses códigos são usados no relatório de auditoria e na camada de centroides do GeoPackage.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 4.4.1 — União de duplicidades e revisões

A visão ampliada incorpora pares confirmados e pendentes para representar o contexto completo.

**Diferença para o agrupamento automático.** Esta visão inclui também pares ainda marcados como `revisar_manual`, além das duplicidades confirmadas. Pares declarados como cavidades diferentes não são unidos.

**Finalidade.** O grupo ampliado não autoriza remoção. Ele preserva o contexto relacional para inspeção humana e para a camada de centroides, enquanto `grupo_duplicidade_auto` continua sendo a referência exclusiva das decisões automáticas.

**Contrato técnico com o código.** O bloco cria ou atualiza `uf_revisao`, `pares_revisao`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
uf_revisao = UnionFind()

if not df_pares_tratamento.empty:
    pares_revisao = df_pares_tratamento[
        df_pares_tratamento[COL_DECISAO_TRATAMENTO].isin([
            "duplicidade_automatica",
            "revisar_manual",
            "duplicidade_manual_mesma_cavidade"
        ])
    ].copy()

    for _, row in pares_revisao.iterrows():
        uf_revisao.union(
            row["id_linha_a"],
            row["id_linha_b"]
        )


### 4.4.2 — Associação às raízes ampliadas

Registros participantes são mapeados para componentes transitivos de revisão.

**Processamento.** A raiz de cada `id_linha` é consultada no segundo objeto `UnionFind`. Apenas participantes efetivos são incluídos em `mapa_grupo_revisao`, deixando registros sem relação fora dos grupos.

**Saída intermediária.** O mapa associa membros às raízes ampliadas e será convertido em identificadores legíveis na célula seguinte. Ele é independente do mapa automático, pois representa um universo de pares maior.

**Contrato técnico com o código.** O bloco cria ou atualiza `mapa_grupo_revisao`, `raiz`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


**Proteção contra grupos unitários artificiais.** Assim como no agrupamento automático, somente IDs efetivamente envolvidos em uniões são associados às raízes ampliadas. Registros sem par confirmado ou pendente permanecem com código de grupo vazio.


In [ ]:
mapa_grupo_revisao = {
    id_linha: uf_revisao.find(id_linha)
    for id_linha in sorted(uf_revisao.parent)
}


### 4.4.3 — Numeração dos grupos ampliados

As raízes recebem códigos `DUP_REV_...` independentes dos grupos automáticos.

**Formação dos códigos.** As raízes únicas são ordenadas e enumeradas com cinco dígitos, produzindo rótulos `DUP_REV_...`. A ordenação torna a geração determinística para uma mesma base e uma mesma ordem de linhas.

**Interpretação.** Um código de revisão indica proximidade relacional, não confirmação de duplicidade. Por isso, ele não deve ser usado isoladamente para excluir registros.

**Contrato técnico com o código.** O bloco cria ou atualiza `raizes_revisao`, `mapa_nome_grupo_revisao`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
raizes_revisao = sorted(set(mapa_grupo_revisao.values()))

mapa_nome_grupo_revisao = {
    raiz: f"DUP_REV_{posicao + 1:05d}"
    for posicao, raiz in enumerate(raizes_revisao)
}


### 4.4.4 — Propagação dos grupos de revisão

Os códigos são gravados em `df` para auditoria e criação de centroides.

**Mapeamento.** A busca em dois níveis converte `id_linha` em raiz e raiz em rótulo. Registros que não participam de nenhum componente ampliado recebem `""`, distinguindo ausência de grupo de um identificador válido.

**Relação com as próximas etapas.** A coluna resultante acompanha a base tratada, aparece no relatório de auditoria e define os conjuntos sintetizados pelos centroides do GeoPackage.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["grupo_duplicidade_revisao"] = df["id_linha"].map(
    lambda id_linha: mapa_nome_grupo_revisao.get(
        mapa_grupo_revisao.get(id_linha),
        ""
    )
)


## 4.5 — Marcação das remoções manuais no nível dos registros

As decisões manuais são registradas inicialmente no nível dos pares. Aqui, a lista `canies_remover_manual_31` é convertida em uma máscara booleana sobre `df`.

O campo `remover_por_decisao_manual_31` indica, registro a registro, quais CANIEs foram explicitamente escolhidos para remoção. Essa marca não exclui nada ainda; ela será interpretada na Etapa 6 ao definir `status_tratamento`.

**Contrato técnico com o código.** O bloco cria ou atualiza `df`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df["remover_por_decisao_manual_31"] = (
    df["canie_txt"]
    .fillna("")
    .astype(str)
    .str.strip()
    .isin(canies_remover_manual_31)
)


## 4.6 — Conferência dos grupos criados

O encerramento da Etapa 4 imprime quantos grupos automáticos e quantos grupos ampliados de revisão foram formados.

Essa conferência permite detectar resultados inesperados antes da escolha dos registros principais. Uma quantidade anormalmente alta ou baixa pode indicar mudanças nas regras de classificação, nas decisões manuais ou na base de entrada.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 4 concluída.")

print(
    "Grupos automáticos:",
    df["grupo_duplicidade_auto"].replace("", np.nan).nunique()
)

print(
    "Grupos de revisão:",
    df["grupo_duplicidade_revisao"].replace("", np.nan).nunique()
)

print(
    "CANIEs com remoção manual indicada:",
    df["remover_por_decisao_manual_31"].sum()
)


## Síntese da Etapa 4

### Resultado produzido

`df` contém `grupo_duplicidade_auto`, `grupo_duplicidade_revisao` e a marca de remoção manual. Os dois tipos de grupo permanecem separados porque confirmação de duplicidade e contexto para revisão possuem finalidades diferentes.

### Verificações esperadas

- cada membro de um componente deve apontar para o mesmo código de grupo;
- registros únicos devem conservar grupo vazio;
- grupos automáticos não devem incorporar pares decididos como cavidades diferentes;
- a quantidade de remoções manuais marcadas deve ser compatível com a lista informada na Etapa 3.20.

### Ligação com a etapa seguinte

A Etapa 5 usa somente `grupo_duplicidade_auto` para escolher um representante por conjunto confirmado.

**Natureza desta célula.** Esta síntese não executa código.


# Etapa 5 — Escolha do registro principal em cada grupo automático

## Finalidade

Cada grupo confirmado deve conservar exatamente um representante. Esta etapa aplica uma ordenação determinística para escolher o registro mais adequado, evitando decisões dependentes da ordem ocasional da planilha.

## Entradas e critérios

O processo utiliza `df` e `grupo_duplicidade_auto`. Dentro de cada grupo, o primeiro critério exclui da disputa por principal qualquer registro explicitamente marcado para remoção manual. Entre os candidatos restantes, prioriza maior nível de validação, presença de coordenada válida, data de cadastro mais recente, nome mais completo e, por fim, CANIE em ordem textual como desempate estável.

## Produto

O dicionário `registro_principal_por_grupo` relaciona cada grupo ao `id_linha` preservado. A etapa não atribui ainda os status finais; fornece a decisão estrutural consumida pela Etapa 6.

## Como ler a etapa

O primeiro bloco prepara os critérios auxiliares. O tópico 5.1 ordena os membros e seleciona o primeiro colocado; 5.2 confere quantos grupos receberam principal.

**Natureza desta célula.** Esta abertura descreve a política de precedência; a célula de código imediatamente seguinte prepara os campos usados na seleção.


**Tratamento de nome ausente.** Para o critério de completude, nome nulo tem comprimento zero. Ele não é convertido na palavra `"nan"`, que teria três caracteres e introduziria uma prioridade inexistente.


In [ ]:
df["tem_coordenada_valida"] = (
    df["longitude_num"].notna()
    & df["latitude_num"].notna()
    & df["geometria_valida"].fillna(False)
)

df["tamanho_nome"] = df[COL_NOME].fillna("").astype(str).str.len()

df["canie_principal"] = ""
df["id_linha_principal"] = np.nan

grupos_auto = (
    df.loc[df["grupo_duplicidade_auto"] != "", "grupo_duplicidade_auto"]
    .dropna()
    .unique()
)


## 5.1 — Ordenação dos membros e escolha do principal

Para cada `grupo_duplicidade_auto`, o script separa seus membros e os ordena pelos critérios definidos anteriormente.

O primeiro registro após a ordenação é adotado como principal. Seu `id_linha` e seu CANIE são propagados para todos os membros do grupo por meio de `id_linha_principal` e `canie_principal`.

Essa propagação permite que a Etapa 6 diferencie claramente o registro preservado dos registros duplicados que serão retirados apenas da base consolidada.

**Contrato técnico com o código.** O bloco cria ou atualiza `membros`, `principal`, `id_principal`, `canie_principal`, `linhas_do_grupo`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


**Precedência da decisão autoral.** `remover_por_decisao_manual_31` é ordenado de forma ascendente antes dos critérios cadastrais. Portanto, `False` vem antes de `True`, impedindo que um CANIE explicitamente destinado à remoção seja escolhido como representante e evitando a exclusão dos dois lados de um mesmo par.


In [ ]:
for grupo in grupos_auto:
    membros = df[df["grupo_duplicidade_auto"] == grupo].copy()

    membros = membros.sort_values(
        by=[
            "remover_por_decisao_manual_31",
            "nivel_validacao_num",
            "tem_coordenada_valida",
            "data_cadastro_dt",
            "tamanho_nome",
            "canie_txt"
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            True
        ],
        na_position="last"
    )

    principal = membros.iloc[0]

    id_principal = principal["id_linha"]
    canie_principal = principal["canie_txt"]

    linhas_do_grupo = df["grupo_duplicidade_auto"] == grupo

    df.loc[linhas_do_grupo, "id_linha_principal"] = id_principal
    df.loc[linhas_do_grupo, "canie_principal"] = canie_principal


## 5.2 — Resumo da escolha dos registros principais

O bloco final informa quantos grupos automáticos receberam um registro principal. O valor deve ser igual à quantidade de grupos automáticos criada na Etapa 4.

A mensagem funciona como uma verificação de integridade antes da atribuição dos status de tratamento.

**Verificação esperada.** `grupos_auto` foi construído a partir dos valores não vazios de `grupo_duplicidade_auto`. Cada um deve ter exatamente um principal após a ordenação por critérios. Uma divergência entre grupos e principais indicaria falha na seleção e deve ser investigada antes da Etapa 6.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 5 concluída.")
print(f"Grupos automáticos com registro principal definido: {len(grupos_auto):,}")


## Síntese da Etapa 5

### Resultado produzido

`registro_principal_por_grupo` associa cada grupo automático a exatamente um `id_linha`, escolhido segundo critérios documentados e determinísticos.

### Verificações esperadas

- a quantidade de principais deve coincidir com a quantidade de grupos automáticos;
- nenhum grupo deve possuir dois principais;
- empates devem ser resolvidos pelo critério final estável, sem depender da ordem de execução.

### Ligação com a etapa seguinte

A Etapa 6 converte essa escolha, junto das decisões manuais e pendências, em status no nível de cada registro.

**Natureza desta célula.** Esta síntese não executa código.


# Etapa 6 — Tradução das decisões em status por registro

## Finalidade

Até aqui, as evidências estão distribuídas entre pares, grupos e escolhas manuais. Esta etapa consolida essas informações em um único status por registro, que será a referência para exportação e remoção controlada.

## Entradas

São utilizados os grupos automáticos e ampliados da Etapa 4, os principais escolhidos na Etapa 5, as decisões manuais e a lista explícita de CANIEs a remover.

## Processamento e precedência

Todos os registros começam como `unico`. Membros de grupos confirmados tornam-se `principal` ou `duplicado_remover`. Remoções manuais explícitas substituem o status automático quando aplicáveis. Decisões de cavidades diferentes preservam os envolvidos, e pendências restantes recebem `revisar_manual`. A ordem dos blocos é parte da regra de negócio.

## Produtos

`status_tratamento` informa a ação; `motivo_tratamento` registra a justificativa. Essas colunas orientam a base consolidada e aparecem nos produtos de auditoria.

## Como ler a etapa

Os tópicos 6.1 a 6.4 aplicam cada camada de decisão em ordem; 6.5 resume o resultado.

**Natureza desta célula.** Esta abertura explica a precedência dos status. O código seguinte inicializa as colunas antes das regras específicas.


In [ ]:
df["status_tratamento"] = "unico"
df["motivo_tratamento"] = ""


## 6.1 — Marcação dos grupos automáticos

Registros pertencentes a `grupo_duplicidade_auto` são inicialmente marcados como `duplicado_remover`. Em cada grupo, o membro cujo `id_linha` coincide com `id_linha_principal` é corrigido para `principal`.

O script também registra justificativas textuais diferentes para o principal e para os duplicados. Isso mantém rastreabilidade no relatório final e informa qual CANIE representa cada conjunto.

**Contrato técnico com o código.** O bloco cria ou atualiza `tem_grupo_auto`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
tem_grupo_auto = df["grupo_duplicidade_auto"] != ""

df.loc[
    tem_grupo_auto,
    "status_tratamento"
] = "duplicado_remover"

df.loc[
    tem_grupo_auto
    & (df["id_linha"] == df["id_linha_principal"]),
    "status_tratamento"
] = "principal"

df.loc[
    tem_grupo_auto
    & (df["id_linha"] == df["id_linha_principal"]),
    "motivo_tratamento"
] = (
    "Registro principal escolhido por prioridade: maior nível de validação, "
    "coordenada válida, data de cadastro mais recente e maior completude do nome."
)

df.loc[
    tem_grupo_auto
    & (df["id_linha"] != df["id_linha_principal"]),
    "motivo_tratamento"
] = (
    "Registro marcado como duplicado automático. "
    "O registro principal do grupo está indicado no campo canie_principal."
)


## 6.2 — Aplicação das remoções manuais explícitas

A máscara `remover_por_decisao_manual_31`, criada na Etapa 4, tem precedência sobre a classificação automática.

Os registros marcados recebem `duplicado_remover_manual` e uma justificativa que aponta para a decisão da Etapa 3.20. Essa distinção é importante para auditoria: permite separar exclusões automáticas de exclusões deliberadas pelo usuário.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
df.loc[
    df["remover_por_decisao_manual_31"],
    "status_tratamento"
] = "duplicado_remover_manual"

df.loc[
    df["remover_por_decisao_manual_31"],
    "motivo_tratamento"
] = (
    "Registro removido por decisão manual na Etapa 3.20. "
    "O CANIE foi informado em canies_remover_manual_31."
)


## 6.3 — Preservação de pares decididos como cavidades diferentes

Pares classificados manualmente como `nao_remover_manual_grupo_diferente` representam feições que não devem ser eliminadas, mesmo que tenham sido candidatas por proximidade ou semelhança.

O bloco coleta os IDs dos dois lados desses pares e marca os respectivos registros como `nao_remover_manual`, exceto quando um CANIE foi explicitamente indicado para remoção. Assim, a decisão manual de preservação não anula uma remoção manual mais específica.

**Contrato técnico com o código.** O bloco cria ou atualiza `ids_nao_remover_manual`, `mask_nao_remover_manual`, `pares_nao_remover_manual`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
ids_nao_remover_manual = set()

if not df_pares_tratamento.empty and COL_DECISAO_TRATAMENTO in df_pares_tratamento.columns:
    pares_nao_remover_manual = df_pares_tratamento[
        df_pares_tratamento[COL_DECISAO_TRATAMENTO]
        == "nao_remover_manual_grupo_diferente"
    ].copy()

    ids_nao_remover_manual = set(pares_nao_remover_manual["id_linha_a"]).union(
        set(pares_nao_remover_manual["id_linha_b"])
    )

mask_nao_remover_manual = (
    df["id_linha"].isin(ids_nao_remover_manual)
    & ~df["remover_por_decisao_manual_31"]
)

df.loc[
    mask_nao_remover_manual,
    "status_tratamento"
] = "nao_remover_manual"

df.loc[
    mask_nao_remover_manual,
    "motivo_tratamento"
] = (
    "Registro preservado por decisão manual na Etapa 3.20. "
    "O grupo foi considerado como cavidades diferentes."
)


## 6.4 — Identificação dos casos ainda pendentes

Pares que continuam com `revisar_manual` são convertidos em um conjunto de IDs envolvidos. Registros ainda classificados como `unico` passam para `revisar_manual`.

O filtro sobre o status atual evita sobrescrever registros já definidos como principais, duplicados automáticos ou remoções manuais. Esses casos permanecem na base consolidada até que haja decisão humana suficiente.

**Contrato técnico com o código.** O bloco cria ou atualiza `ids_revisao`, `mask_revisao`, `pares_revisao`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
ids_revisao = set()

if not df_pares_tratamento.empty and COL_DECISAO_TRATAMENTO in df_pares_tratamento.columns:
    pares_revisao = df_pares_tratamento[
        df_pares_tratamento[COL_DECISAO_TRATAMENTO] == "revisar_manual"
    ].copy()

    ids_revisao = set(pares_revisao["id_linha_a"]).union(
        set(pares_revisao["id_linha_b"])
    )

mask_revisao = (
    df["id_linha"].isin(ids_revisao)
    & (df["status_tratamento"] == "unico")
)

df.loc[
    mask_revisao,
    "status_tratamento"
] = "revisar_manual"

df.loc[
    mask_revisao,
    "motivo_tratamento"
] = (
    "Registro envolvido em suspeita de duplicidade, "
    "mas sem segurança para remoção automática."
)


## 6.5 — Resumo dos status de tratamento

O encerramento apresenta a distribuição de `status_tratamento`. Essa tabela é uma das principais verificações do fluxo porque mostra quantos registros serão preservados, removidos automaticamente, removidos manualmente ou mantidos para revisão.

As mesmas categorias serão usadas nas Etapas 7, 8, 9 e 10 para gerar produtos específicos.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 6 concluída.")
print("Resumo do status de tratamento:")
print(df["status_tratamento"].value_counts(dropna=False))


## Síntese da Etapa 6

### Resultado produzido

Cada linha de `df` possui `status_tratamento` e `motivo_tratamento`, refletindo a precedência entre escolha automática, remoção manual, preservação manual e revisão pendente.

### Verificações esperadas

- todo registro deve possuir um status reconhecido;
- cada grupo automático deve conter um único `principal`;
- somente registros autorizados devem receber status de remoção;
- pendências devem permanecer preservadas até nova decisão.

### Ligação com a etapa seguinte

A Etapa 7 extrai os registros e pares com status de revisão para análise humana.

**Natureza desta célula.** Esta síntese não executa código.


# Etapa 7 — Preparação dos produtos de revisão manual

## Finalidade

Casos não resolvidos automaticamente precisam ser entregues em formatos adequados à análise humana. Esta etapa separa as linhas pendentes e os pares que originaram a suspeita, sem alterar seus status.

## Entradas e produtos

O processo lê `df` com `status_tratamento` e `df_pares_tratamento` com a decisão efetiva. Produz `df_revisar_linhas`, para examinar cada cadastro, e `df_revisar_pares`, para comparar A e B lado a lado com distância e evidências relevantes.

## Limite operacional

Essas tabelas são filas de trabalho, não listas de exclusão. Um caso permanece na base consolidada enquanto não houver decisão explícita que autorize sua remoção.

## Como ler a etapa

O primeiro bloco seleciona registros pendentes; 7.1 recupera os pares relacionados; 7.2 reduz o esquema para análise; 7.3 confere as quantidades.

**Natureza desta célula.** Esta abertura apresenta os produtos de revisão. O código seguinte cria a tabela no nível dos registros.


In [ ]:
df_revisar_linhas = df[
    df["status_tratamento"] == "revisar_manual"
].copy()


## 7.1 — Seleção dos pares que continuam pendentes

Além das linhas individuais, é necessário preservar a relação entre os dois registros que ainda exige decisão. O filtro deve usar a mesma tabela e a mesma coluna de decisão consumidas pelo tratamento, e não a decisão preliminar anterior à intervenção humana.

O bloco consulta `df_pares_tratamento` e `COL_DECISAO_TRATAMENTO`, mantendo somente o valor `revisar_manual`. Assim, um par originalmente apresentado ao usuário, mas depois resolvido como mesma cavidade ou cavidades diferentes, permanece na trilha histórica da Etapa 9.4 e deixa de aparecer na fila operacional pendente.

Se a tabela ou a coluna de decisão não estiver disponível, o retorno é um DataFrame vazio, preservando a execução segura da exportação.

**Contrato técnico com o código.** O bloco cria `df_revisar_pares` a partir da decisão efetiva final. Sua quantidade deve ser compatível com a camada `pares_revisao_manual_linhas`, descontados apenas pares sem coordenadas completas.


In [ ]:
if (
    not df_pares_tratamento.empty
    and COL_DECISAO_TRATAMENTO in df_pares_tratamento.columns
):
    df_revisar_pares = df_pares_tratamento[
        df_pares_tratamento[COL_DECISAO_TRATAMENTO] == "revisar_manual"
    ].copy()
else:
    df_revisar_pares = pd.DataFrame()


## 7.2 — Seleção das colunas úteis para análise par-a-par

A tabela de pares pode conter muitas colunas auxiliares. Este bloco escolhe apenas identificadores, nomes, localização, datas, níveis de validação, distância, similaridade e justificativas de candidatura.

A checagem de existência das colunas torna a seleção tolerante a tabelas vazias ou pequenas mudanças no fluxo. O resultado `df_revisar_pares` será gravado como uma aba específica no relatório de auditoria.

**Contrato técnico com o código.** O bloco cria ou atualiza `colunas_pares_revisao`, `df_revisar_pares`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
colunas_pares_revisao = [
    "id_linha_a",
    "canie_txt_a",
    f"{COL_NOME}_a",
    "uf_txt_a",
    "municipio_txt_a",
    "longitude_num_a",
    "latitude_num_a",
    "x_m_a",
    "y_m_a",
    "data_cadastro_dt_a",
    "nivel_validacao_num_a",

    "id_linha_b",
    "canie_txt_b",
    f"{COL_NOME}_b",
    "uf_txt_b",
    "municipio_txt_b",
    "longitude_num_b",
    "latitude_num_b",
    "x_m_b",
    "y_m_b",
    "data_cadastro_dt_b",
    "nivel_validacao_num_b",

    "distancia_metros",
    "classe_distancia",
    "similaridade_nome",
    "classe_suspeita",
    "motivo_candidatura",
    "decisao_preliminar"
]

colunas_pares_revisao = [
    coluna for coluna in colunas_pares_revisao
    if coluna in df_revisar_pares.columns
]

df_revisar_pares = df_revisar_pares[
    colunas_pares_revisao
].copy()


## 7.3 — Resumo dos casos pendentes

O encerramento informa quantas linhas e quantos pares ainda requerem revisão manual.

Essas duas contagens não precisam ser iguais: um mesmo registro pode participar de mais de um par, e um par sempre envolve duas linhas. A distinção ajuda a dimensionar o esforço de revisão.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 7 concluída.")
print(f"Linhas para revisão manual: {len(df_revisar_linhas):,}")
print(f"Pares para revisão manual: {len(df_revisar_pares):,}")


## Síntese da Etapa 7

### Resultado produzido

`df_revisar_linhas` apresenta os cadastros pendentes individualmente, e `df_revisar_pares` preserva as relações que justificam a revisão, com um esquema reduzido para leitura operacional.

### Verificações esperadas

- todos os registros das tabelas devem continuar com status `revisar_manual`;
- pares resolvidos automática ou manualmente não devem permanecer na fila;
- a ausência de pendências deve produzir tabelas vazias, não erro.

### Ligação com a etapa seguinte

A Etapa 8 preserva essas pendências na base consolidada e exclui apenas remoções autorizadas.

**Natureza desta célula.** Esta síntese não executa código.


# Etapa 8 — Construção da base consolidada

## Finalidade

Esta etapa materializa o principal produto tabular do tratamento: uma cópia da base de trabalho sem os registros cuja remoção foi autorizada automaticamente ou manualmente.

## Regra de inclusão

São excluídos somente `duplicado_remover` e `duplicado_remover_manual`. Registros `principal`, `unico`, `nao_remover_manual` e `revisar_manual` permanecem. A filtragem é deliberadamente conservadora: pendência não equivale a autorização de exclusão.

## Produto

`df_consolidado` é gravado em `02_outputs/canie_consolidado_sem_duplicatas_automaticas_e_manuais.xlsx`. `df` e `df_original` continuam intactos para auditoria e comparação.

## Como ler a etapa

O primeiro bloco executa a filtragem; 8.1 grava o arquivo; 8.2 compara contagens e apresenta o caminho produzido.

**Natureza desta célula.** Esta abertura define exatamente o que “consolidada” significa neste projeto. O código seguinte aplica a regra de exclusão.


In [ ]:
df_consolidado = df[
    ~df["status_tratamento"].isin([
        "duplicado_remover",
        "duplicado_remover_manual"
    ])
].copy()


## 8.1 — Definição do arquivo e exportação para Excel

O arquivo é gravado em `02_outputs`, usando `PASTA_SAIDA` definida na Etapa 1. A opção `index=False` evita incluir o índice interno do pandas como coluna adicional.

A exportação não modifica `df_original` nem `df`; ela materializa apenas a visão consolidada construída no bloco anterior.

**Contrato técnico com o código.** O bloco cria ou atualiza `caminho_consolidado`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
caminho_consolidado = (
    PASTA_SAIDA
    / "canie_consolidado_sem_duplicatas_automaticas_e_manuais.xlsx"
)

df_consolidado.to_excel(
    caminho_consolidado,
    index=False
)


## 8.2 — Conferência da consolidação

O resumo compara a quantidade da base original com a base consolidada e calcula quantos registros foram retirados.

Também imprime o caminho completo do arquivo, permitindo confirmar que a nova organização de outputs foi respeitada.

**Leitura dos números.** A diferença entre `df_original` e `df_consolidado` deve corresponder aos status de remoção definidos na Etapa 6. Casos `revisar_manual` permanecem na base e, portanto, não entram nessa diferença.

**Produto verificado.** `caminho_consolidado` aponta para a planilha gravada em `02_outputs`. A mensagem confirma a tentativa concluída de escrita, enquanto a comparação numérica oferece uma checagem imediata de coerência.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 8 concluída.")
print(f"Registros na base original: {len(df_original):,}")
print(f"Registros na base consolidada: {len(df_consolidado):,}")
print(f"Registros removidos: {len(df_original) - len(df_consolidado):,}")
print(f"Arquivo gerado: {caminho_consolidado}")


## Síntese da Etapa 8

### Resultado produzido

`df_consolidado` e sua planilha em `02_outputs` preservam todos os registros, exceto aqueles classificados como remoção automática ou manual.

### Verificações esperadas

- `len(df_original) - len(df_consolidado)` deve coincidir com a quantidade de registros removidos;
- casos pendentes devem permanecer no consolidado;
- a exportação não deve modificar `df_original` nem `df`;
- o caminho informado deve estar dentro de `02_outputs`.

### Ligação com a etapa seguinte

A Etapa 9 reúne o consolidado e toda a trilha decisória em um relatório de auditoria multirrelacional.

**Natureza desta célula.** Esta síntese não executa código.


# Etapa 9 — Construção do relatório de auditoria

## Finalidade

Uma base consolidada precisa ser acompanhada de evidências que permitam reconstruir as decisões. Esta etapa reúne bases, pares, grupos, decisões manuais, pendências e contagens em um único arquivo Excel com múltiplas abas.

## Entradas

São utilizados `df_original`, `df`, `df_consolidado`, as tabelas de pares e grupos, os produtos de revisão e os objetos opcionais criados na Etapa 3.20.

## Processamento

Primeiro são calculados resumos em nível de registro e de par. Depois são separados subconjuntos temáticos e garantidos esquemas vazios para objetos opcionais. Por fim, um único `ExcelWriter` grava todas as abas e é fechado explicitamente.

## Produto

O relatório preserva tanto o resultado quanto a trilha que o originou. Contagens de registros e contagens de pares têm granularidades distintas e devem ser interpretadas separadamente.

## Como ler a etapa

Os tópicos 9.1 e 9.2 criam resumos; 9.3 a 9.5 preparam tabelas e caminho; 9.6 grava as abas; 9.7 confirma a conclusão.

**Natureza desta célula.** Esta abertura descreve a arquitetura do relatório; a execução começa em 9.1.


## 9.1 — Resumo por status dos registros

A contagem de `status_tratamento` resume o resultado no nível dos registros.

**Entrada.** A coluna foi atribuída na Etapa 6 e contém uma categoria por registro, como único, principal, remoção automática, remoção manual ou revisão. `value_counts(dropna=False)` inclui eventuais ausências para que problemas de classificação não desapareçam do resumo.

**Saída.** O resultado é convertido em tabela com as colunas `status_tratamento` e `quantidade`. Essa tabela será exportada como uma aba do relatório de auditoria na Etapa 9.6.

**Contrato técnico com o código.** O bloco cria ou atualiza `resumo_status`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
resumo_status = (
    df["status_tratamento"]
    .value_counts(dropna=False)
    .rename_axis("status_tratamento")
    .reset_index(name="quantidade")
)


## 9.2 — Resumos dos pares

Classes de suspeita, distância e decisão final são agregadas; esquemas vazios mantêm a exportação estável.

**Agregações.** Quando existem pares, três contagens independentes resumem `classe_suspeita`, `classe_distancia` e a coluna de decisão efetivamente usada no tratamento. `dropna=False` mantém valores ausentes visíveis como possível sinal de inconsistência.

**Caso sem pares.** DataFrames vazios com nomes de colunas definidos preservam o contrato da exportação. Assim, o relatório mantém suas abas e seu esquema mesmo em uma execução sem candidatos.

**Contrato técnico com o código.** O bloco cria ou atualiza `resumo_classes`, `resumo_distancias`, `resumo_decisao_final`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if not df_pares_tratamento.empty:
    resumo_classes = (
        df_pares_tratamento["classe_suspeita"]
        .value_counts(dropna=False)
        .rename_axis("classe_suspeita")
        .reset_index(name="quantidade_pares")
    )

    resumo_distancias = (
        df_pares_tratamento["classe_distancia"]
        .value_counts(dropna=False)
        .rename_axis("classe_distancia")
        .reset_index(name="quantidade_pares")
    )

    resumo_decisao_final = (
        df_pares_tratamento[COL_DECISAO_TRATAMENTO]
        .value_counts(dropna=False)
        .rename_axis("decisao_final")
        .reset_index(name="quantidade_pares")
    )
else:
    resumo_classes = pd.DataFrame(
        columns=["classe_suspeita", "quantidade_pares"]
    )

    resumo_distancias = pd.DataFrame(
        columns=["classe_distancia", "quantidade_pares"]
    )

    resumo_decisao_final = pd.DataFrame(
        columns=["decisao_final", "quantidade_pares"]
    )


## 9.3 — Separação das tabelas temáticas

Este bloco prepara subconjuntos usados como evidências do tratamento: registros envolvidos em grupos automáticos, remoções manuais e pares que resultaram em duplicidade automática ou manual.

Cada subconjunto responde a uma pergunta de auditoria diferente, como “quais registros foram afetados?” ou “quais pares justificaram uma remoção?”.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_duplicados_auto`, `df_removidos_manuais`, `df_pares_auto`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
df_duplicados_auto = df[
    df["grupo_duplicidade_auto"] != ""
].copy()

df_removidos_manuais = df[
    df["status_tratamento"] == "duplicado_remover_manual"
].copy()

if not df_pares_tratamento.empty:
    df_pares_auto = df_pares_tratamento[
        df_pares_tratamento[COL_DECISAO_TRATAMENTO].isin([
            "duplicidade_automatica",
            "duplicidade_manual_mesma_cavidade"
        ])
    ].copy()
else:
    df_pares_auto = pd.DataFrame()


## 9.4 — Construção segura das tabelas opcionais

Alguns produtos dependem da execução da Etapa 3.20. Para manter o relatório útil e executável em todos os cenários, este bloco constrói cada tabela a partir do objeto mais próximo disponível, em vez de criar silenciosamente abas vazias.

`df_pares_suspeitos_31` recebe uma cópia de `df_revisao_manual_31`, preservando todos os pares originalmente apresentados ao usuário, inclusive aqueles depois resolvidos. `df_grupos_revisao_manual` seleciona em `df` apenas os registros cujo status final continua como `revisar_manual`. A tabela de decisões manuais é preservada quando existe e recebe esquema vazio somente quando a etapa correspondente não foi executada.

Essa distinção permite comparar o universo originalmente submetido à revisão com o conjunto que permanece pendente após as decisões.

**Contrato técnico com o código.** O bloco cria ou confirma `df_pares_suspeitos_31`, `df_grupos_revisao_manual` e `df_decisoes_manuais_31`, deixando os três objetos prontos para as abas da Etapa 9.6.


In [ ]:
if "df_pares_suspeitos_31" not in globals():
    df_pares_suspeitos_31 = (
        df_revisao_manual_31.copy()
        if "df_revisao_manual_31" in globals()
        else pd.DataFrame()
    )

if "df_grupos_revisao_manual" not in globals():
    df_grupos_revisao_manual = df.loc[
        df["status_tratamento"] == "revisar_manual"
    ].copy()

if "df_decisoes_manuais_31" not in globals():
    df_decisoes_manuais_31 = pd.DataFrame()


## 9.5 — Definição do caminho do relatório

O relatório de auditoria é direcionado para `02_outputs` com um nome fixo e descritivo.

Manter um caminho único facilita a reprodução do fluxo, mas também significa que uma nova execução substitui o relatório anterior. A base de entrada permanece preservada em `01_bases`.

**Contrato técnico com o código.** O bloco cria ou atualiza `caminho_relatorio`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
caminho_relatorio = (
    PASTA_SAIDA
    / "relatorio_auditoria_duplicidades_canie_com_revisao_manual.xlsx"
)


## 9.6 — Escrita das abas do relatório Excel

`pd.ExcelWriter` reúne em um único arquivo a base original, a base com status, a base consolidada, os pares com decisão, produtos de revisão e tabelas-resumo.

Cada aba representa um nível de auditoria diferente. A presença simultânea da base original e da consolidada permite rastrear qualquer remoção; as abas de pares e decisões explicam por que ela ocorreu.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 9.6.1 — Abertura controlada do escritor Excel

O `ExcelWriter` é aberto explicitamente para permitir que grupos pequenos de abas sejam gravados em células independentes. Ele será fechado somente após todas as tabelas serem adicionadas.

**Por que o fechamento é adiado.** Cada chamada `to_excel` posterior acrescenta uma planilha ao mesmo workbook em memória. Manter uma única instância evita sobrescritas entre abas e garante que todas pertençam ao mesmo arquivo indicado por `caminho_relatorio`.

**Responsabilidade.** A partir desta célula, qualquer erro antes de `close()` pode deixar o arquivo incompleto. A célula 9.6.7 realiza a finalização explícita após todas as escritas.

**Contrato técnico com o código.** O bloco cria ou atualiza `writer_relatorio`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
writer_relatorio = pd.ExcelWriter(caminho_relatorio, engine="openpyxl")


### 9.6.2 — Bases original, tratada e consolidada

As três primeiras abas permitem comparar entrada, situação após o tratamento e resultado consolidado. Essa sequência constitui o núcleo da rastreabilidade por registro.

**Conteúdo das visões.** `df_original` preserva os dados lidos sem colunas auxiliares; `df` inclui evidências, grupos e status; `df_consolidado` exclui somente os registros com decisão de remoção. A comparação entre as três permite reconstruir a passagem da entrada ao produto final.

**Efeito.** Esta célula apenas escreve cópias tabulares no relatório; não modifica nenhum dos DataFrames em memória.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
df_original.to_excel(writer_relatorio, sheet_name="base_original_preservada", index=False)
df.to_excel(writer_relatorio, sheet_name="base_com_status", index=False)
df_consolidado.to_excel(writer_relatorio, sheet_name="base_consolidada", index=False)


### 9.6.3 — Pares e grupos com decisão

Estas abas registram os pares completos, os pares suspeitos agrupados e os grupos destinados à revisão. Elas explicam como relações entre registros resultaram nos status finais.

**Níveis de auditoria.** `df_pares_tratamento` registra cada relação e sua decisão final. `df_pares_suspeitos_31` organiza os casos candidatos usados na etapa manual, e `df_grupos_revisao_manual` lista os registros cujo status final ainda exige revisão, mantendo seus códigos de grupo para contextualização.

**Rastreabilidade.** Em conjunto, essas abas permitem partir de um status atribuído a uma linha e localizar os pares e grupos que forneceram sua justificativa.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
df_pares_tratamento.to_excel(writer_relatorio, sheet_name="pares_com_decisao_final", index=False)
df_pares_suspeitos_31.to_excel(writer_relatorio, sheet_name="pares_suspeitos_grupos", index=False)
df_grupos_revisao_manual.to_excel(writer_relatorio, sheet_name="grupos_revisao_manual", index=False)


### 9.6.4 — Decisões e remoções

As abas deste bloco isolam decisões manuais, registros envolvidos em duplicidade, remoções manuais e pares que justificaram remoção. Elas formam a trilha de auditoria das exclusões.

**Separação das evidências.** As decisões explícitas ficam em aba própria; os registros pertencentes a grupos automáticos são apresentados separadamente; e as tabelas de removidos e pares preservam, respectivamente, o objeto excluído e a relação que fundamentou a exclusão.

**Objetivo de controle.** Essa divisão permite conferir se cada remoção manual foi autorizada e se cada remoção automática pode ser relacionada a ao menos um par classificado como duplicidade.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
df_decisoes_manuais_31.to_excel(writer_relatorio, sheet_name="decisoes_manuais", index=False)
df_duplicados_auto.to_excel(writer_relatorio, sheet_name="duplicados_auto_e_manuais", index=False)
df_removidos_manuais.to_excel(writer_relatorio, sheet_name="removidos_manuais", index=False)
df_pares_auto.to_excel(writer_relatorio, sheet_name="pares_auto_e_manuais", index=False)


### 9.6.5 — Produtos para revisão humana

As duas abas seguintes apresentam os casos pendentes em nível de linha e em nível de par. A primeira favorece inspeção individual; a segunda preserva a relação que originou a suspeita.

**Duas escalas de leitura.** `df_revisar_linhas` mostra cada cadastro envolvido, útil para consultar seus atributos completos. `df_revisar_pares` mantém A e B lado a lado, incluindo distância e demais evidências que motivaram a pendência.

**Consequência operacional.** Essas abas não alteram a base consolidada. Elas formam a fila de trabalho para validação posterior em campo ou por análise documental.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
df_revisar_linhas.to_excel(writer_relatorio, sheet_name="revisao_manual_linhas", index=False)
df_revisar_pares.to_excel(writer_relatorio, sheet_name="revisao_manual_pares", index=False)


### 9.6.6 — Tabelas-resumo

Os resumos agregam status, classes, faixas de distância e decisões finais. Eles permitem avaliar rapidamente a distribuição dos resultados sem percorrer as tabelas detalhadas.

**Granularidade.** `resumo_status` conta registros; os demais resumos contam pares. As grandezas não devem ser somadas entre si, porque um registro pode participar de vários pares e um grupo pode conter mais de dois membros.

**Uso na conferência.** As distribuições ajudam a detectar mudanças anormais entre execuções, como aumento de revisões, concentração em uma faixa de distância ou alteração da proporção de remoções.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
resumo_status.to_excel(writer_relatorio, sheet_name="resumo_status", index=False)
resumo_classes.to_excel(writer_relatorio, sheet_name="resumo_classes", index=False)
resumo_distancias.to_excel(writer_relatorio, sheet_name="resumo_distancias", index=False)
resumo_decisao_final.to_excel(writer_relatorio, sheet_name="resumo_decisao_final", index=False)


### 9.6.7 — Fechamento do arquivo Excel

`close()` finaliza as estruturas internas do workbook e garante que todas as abas sejam efetivamente gravadas. A confirmação da Etapa 9 só deve aparecer depois deste fechamento.

**Efeito técnico.** O fechamento grava índices, relações e demais estruturas do formato `.xlsx` que ainda estavam em memória. Somente depois dessa chamada o arquivo deve ser considerado pronto para abertura por outro programa.

**Sequência.** Esta célula depende de todas as escritas 9.4.2 a 9.4.6 e antecede a mensagem de sucesso. Se ocorrer erro aqui, a confirmação seguinte não deve ser executada.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
writer_relatorio.close()


## 9.7 — Confirmação da geração do relatório

Após o fechamento seguro do `ExcelWriter`, o bloco informa que a etapa terminou e mostra o caminho do arquivo produzido.

A mensagem só é executada depois que todas as abas foram gravadas, funcionando como confirmação de conclusão da exportação.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("Etapa 9 concluída.")
print(f"Relatório de auditoria gerado: {caminho_relatorio}")


## Síntese da Etapa 9

### Resultado produzido

O arquivo Excel de auditoria reúne bases, pares, grupos, decisões manuais, remoções, pendências e resumos em abas separadas, todas derivadas da mesma execução.

### Verificações esperadas

- o arquivo deve ser fechado antes da mensagem de sucesso;
- as três visões da base devem manter suas granularidades próprias;
- abas de pares contam relações, enquanto abas de base contam registros;
- objetos opcionais vazios devem preservar o esquema das respectivas abas.

### Ligação com a etapa seguinte

A Etapa 10 espacializa as mesmas decisões em pontos, linhas e centroides, mantendo o Excel como referência tabular de auditoria.

**Natureza desta célula.** Esta síntese não executa código.


# Etapa 10 — Geração do GeoPackage espacial

## Finalidade

Esta etapa converte os resultados tabulares em camadas para inspeção em software SIG. O objetivo é permitir que registros, relações e grupos sejam examinados espacialmente sem substituir as tabelas de auditoria.

## Entradas e referência espacial

São utilizados `df`, `df_consolidado`, os subconjuntos de status e a tabela de pares efetivamente usada no tratamento. As geometrias de saída usam SIRGAS 2000 (`EPSG:4674`); distâncias já foram calculadas em sistema métrico na Etapa 2.

## Produtos

O GeoPackage contém camadas pontuais das bases, linhas entre membros de pares e centroides sintéticos dos grupos de revisão. Datas e objetos são convertidos para tipos interoperáveis, nomes de campos são reduzidos e camadas vazias são ignoradas com mensagem explícita.

## Limites de interpretação

Registros sem coordenadas permanecem nas tabelas, mas não geram feições. Linhas representam relações entre cadastros, e centroides representam médias para localização geral; nenhum deles constitui nova coordenada oficial de cavidade.

## Como ler a etapa

Os tópicos 10.1 e 10.2 definem parâmetros; 10.3 concentra funções reutilizáveis; 10.4 a 10.6 constroem geometrias; 10.7 grava as camadas; 10.8 confere o resultado.

**Natureza desta célula.** Esta abertura apresenta o contrato cartográfico da exportação. A execução começa com as importações imediatamente seguintes.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point, LineString
from pathlib import Path


## 10.1 — Sistema de referência e caminho do GeoPackage

Todas as camadas finais usam SIRGAS 2000 geográfico (`EPSG:4674`), compatível com as longitudes e latitudes originais.

O caminho é construído em `02_outputs`. Se um GeoPackage anterior existir, ele é removido antes da escrita para impedir que camadas antigas permaneçam misturadas com a execução atual.

**Contrato técnico com o código.** O bloco cria ou atualiza `CRS_SAIDA_GPKG`, `caminho_geopackage`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
CRS_SAIDA_GPKG = "EPSG:4674"

caminho_geopackage = (
    PASTA_SAIDA
    / "resultado_espacial_tratamento_duplicidades_canie.gpkg"
)

if caminho_geopackage.exists():
    caminho_geopackage.unlink()


## 10.2 — Escolha da tabela de pares para espacialização

A etapa prioriza a versão mais completa das decisões disponíveis. Usa `df_pares_31` quando a revisão manual foi incorporada e, depois, prioriza `df_pares_tratamento`, que é a referência efetivamente usada pelas Etapas 4–9.

`COL_DECISAO_ESPACIAL` registra qual coluna contém a decisão válida. Essa escolha garante coerência entre o relatório Excel e as camadas espaciais.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_pares_espacial`, `COL_DECISAO_ESPACIAL`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if "df_pares_31" in globals() and "decisao_final_31" in df_pares_31.columns:
    df_pares_espacial = df_pares_31.copy()
    COL_DECISAO_ESPACIAL = "decisao_final_31"
else:
    df_pares_espacial = df_pares.copy()
    COL_DECISAO_ESPACIAL = "decisao_preliminar"

if "df_pares_tratamento" in globals() and not df_pares_tratamento.empty:
    df_pares_espacial = df_pares_tratamento.copy()

    if "decisao_final_31" in df_pares_espacial.columns:
        COL_DECISAO_ESPACIAL = "decisao_final_31"
    else:
        COL_DECISAO_ESPACIAL = "decisao_preliminar"


## 10.3 — Funções auxiliares de preparação e construção geométrica

Este bloco define funções para converter tipos incompatíveis, encurtar nomes de campos, criar pontos a partir de longitude/latitude, criar linhas entre pares e calcular centroides dos grupos de revisão.

Também define a função de escrita condicional das camadas. Camadas vazias são ignoradas, evitando arquivos ou tabelas sem feições.

As funções não gravam dados imediatamente; elas padronizam operações que serão chamadas nos blocos seguintes.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


### 10.3.1 — Conversão de tipos incompatíveis com GeoPackage

Datas são convertidas para texto ISO e objetos Python são transformados em strings. A geometria é excluída dessa conversão para preservar seu tipo espacial.

**Regras de conversão.** Colunas datetime usam o formato ISO `AAAA-MM-DD`, que é estável e ordenável. Colunas `object` convertem ausências em string vazia e demais objetos em texto. A coluna `geometry` é explicitamente ignorada para continuar sob controle do GeoPandas.

**Saída.** A função trabalha sobre uma cópia e devolve um GeoDataFrame compatível com a escrita no GeoPackage, sem alterar o objeto recebido pelas demais rotinas.

**Contrato técnico com o código.** Esta célula define a função `converter_tipos_para_gpkg(gdf)`. Ela recebe `gdf` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def converter_tipos_para_gpkg(gdf):
    gdf = gdf.copy()
    for coluna in gdf.columns:
        if coluna == "geometry":
            continue
        if pd.api.types.is_datetime64_any_dtype(gdf[coluna]):
            gdf[coluna] = gdf[coluna].dt.strftime("%Y-%m-%d")
        elif gdf[coluna].dtype == "object":
            gdf[coluna] = gdf[coluna].apply(
                lambda valor: "" if pd.isna(valor) else str(valor)
            )
    return gdf


### 10.3.2 — Padronização dos nomes de campos espaciais

Nomes extensos são substituídos por abreviações estáveis e limitados a 60 caracteres. Isso melhora a interoperabilidade com programas SIG e mantém os atributos reconhecíveis.

**Algoritmo.** Substituições conhecidas produzem abreviações legíveis; depois, qualquer nome é limitado a 60 caracteres. `geometry` não entra no mapa porque precisa conservar o nome esperado pelo GeoPandas.

**Contrato.** O retorno é um dicionário `nome_original: nome_exportado`, aplicado de uma só vez pela função seguinte. Centralizar o mapa mantém o mesmo vocabulário em todas as camadas.

**Contrato técnico com o código.** Esta célula define a função `criar_mapa_colunas_gpkg(colunas)`. Ela recebe `colunas` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_mapa_colunas_gpkg(colunas):
    substituicoes = {
        "decisao_preliminar": "dec_prelim", "decisao_final_31": "dec_final",
        "status_tratamento": "status_trat", "motivo_tratamento": "motivo_trat",
        "grupo_duplicidade_auto": "grp_dup_auto", "grupo_duplicidade_revisao": "grp_dup_rev",
        "remover_por_decisao_manual_31": "rem_manual", "observacao_manual_31": "obs_manual",
        "codigo_revisao_manual": "cod_rev_man", "distancia_metros": "dist_m",
        "classe_suspeita": "classe_susp", "classe_distancia": "classe_dist",
        "similaridade_nome": "sim_nome"
    }
    mapa = {}
    for coluna in colunas:
        if coluna == "geometry":
            continue
        novo_nome = str(coluna)
        for antigo, novo in substituicoes.items():
            novo_nome = novo_nome.replace(antigo, novo)
        mapa[coluna] = novo_nome[:60]
    return mapa


### 10.3.3 — Preparação final das colunas

A função coordenadora aplica a conversão de tipos e depois renomeia os campos. Todas as funções de construção geométrica chamam este mesmo ponto, garantindo esquema consistente entre camadas.

**Ordem obrigatória.** A conversão ocorre antes da renomeação para que as verificações de tipo trabalhem sobre as colunas originais. Em seguida, o mapa é calculado a partir do esquema já convertido e aplicado com `rename`.

**Ponto único de preparação.** Camadas de pontos, linhas e centroides passam por esta função, reduzindo diferenças de esquema e problemas de interoperabilidade na Etapa 10.7.

**Contrato técnico com o código.** Esta célula define a função `preparar_colunas_para_gpkg(gdf)`. Ela recebe `gdf` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def preparar_colunas_para_gpkg(gdf):
    gdf = converter_tipos_para_gpkg(gdf)
    return gdf.rename(columns=criar_mapa_colunas_gpkg(gdf.columns))


### 10.3.4 — Construção de camadas pontuais

Registros sem longitude ou latitude são descartados apenas da camada espacial. Para os demais, `Point(longitude, latitude)` cria geometrias em SIRGAS 2000 e `origem_camada` registra a visão que gerou cada feição.

**Entradas.** `df_base` pode ser a base completa ou um subconjunto, e `nome_origem` registra qual produto originou a camada. Sem linhas, a função devolve um GeoDataFrame vazio com CRS definido; com linhas, somente coordenadas completas seguem para a geometria.

**Importante.** A filtragem espacial não remove registros das tabelas. Ela afeta apenas a camada produzida, pois um ponto não pode ser criado sem longitude e latitude válidas.

**Contrato técnico com o código.** Esta célula define a função `criar_gdf_pontos(df_base, nome_origem)`. Ela recebe `df_base`, `nome_origem` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_gdf_pontos(df_base, nome_origem):
    if df_base.empty:
        return gpd.GeoDataFrame(df_base.copy(), geometry=[], crs=CRS_SAIDA_GPKG)
    df_temp = df_base[
        df_base["longitude_num"].notna() & df_base["latitude_num"].notna()
    ].copy()
    df_temp["origem_camada"] = nome_origem
    geometria = [Point(lon, lat) for lon, lat in zip(
        df_temp["longitude_num"], df_temp["latitude_num"]
    )]
    gdf_temp = gpd.GeoDataFrame(df_temp, geometry=geometria, crs=CRS_SAIDA_GPKG)
    return preparar_colunas_para_gpkg(gdf_temp)


### 10.3.5 — Construção de linhas entre pares

A função valida as quatro coordenadas necessárias, remove pares incompletos e cria uma `LineString` do registro A ao B. Essas linhas permitem visualizar distância e relações de duplicidade no mapa.

**Validações.** Um DataFrame vazio gera uma camada vazia. Se alguma coluna de coordenada nem sequer existir, a função lança `KeyError`, pois isso indica quebra de esquema. Quando as colunas existem, pares com qualquer coordenada ausente são excluídos apenas da representação espacial.

**Geometria e saída.** Cada linha liga A a B no CRS geográfico de saída e preserva os atributos do par. A preparação final converte tipos e nomes antes da escrita.

**Contrato técnico com o código.** Esta célula define a função `criar_linhas_pares(df_pares_base, nome_origem)`. Ela recebe `df_pares_base`, `nome_origem` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_linhas_pares(df_pares_base, nome_origem):
    if df_pares_base.empty:
        return gpd.GeoDataFrame(df_pares_base.copy(), geometry=[], crs=CRS_SAIDA_GPKG)
    colunas_coord = ["longitude_num_a", "latitude_num_a", "longitude_num_b", "latitude_num_b"]
    ausentes = [coluna for coluna in colunas_coord if coluna not in df_pares_base.columns]
    if ausentes:
        raise KeyError(f"Não foi possível criar linhas dos pares. Colunas ausentes: {ausentes}")
    df_temp = df_pares_base.dropna(subset=colunas_coord).copy()
    df_temp["origem_camada"] = nome_origem
    geometrias = [LineString([
        (row["longitude_num_a"], row["latitude_num_a"]),
        (row["longitude_num_b"], row["latitude_num_b"])
    ]) for _, row in df_temp.iterrows()]
    gdf_linhas = gpd.GeoDataFrame(df_temp, geometry=geometrias, crs=CRS_SAIDA_GPKG)
    return preparar_colunas_para_gpkg(gdf_linhas)


### 10.3.6 — Estrutura vazia para centroides

Esta função centraliza o esquema devolvido quando não existem grupos ou quando a coluna de agrupamento está ausente. Isso mantém retornos previsíveis para o restante da etapa.

**Contrato de esquema.** Mesmo sem feições, o retorno declara identificador do grupo, quantidade, listas de CANIEs, nomes, status e geometria, além do CRS. As rotinas chamadoras não precisam criar tratamentos especiais para `None`.

**Uso.** `criar_centroides_grupos` utiliza esta estrutura tanto quando a coluna de grupos não existe quanto quando nenhum grupo possui coordenadas utilizáveis.

**Contrato técnico com o código.** Esta célula define a função `criar_gdf_centroides_vazio()`. Ela recebe nenhum argumento posicional e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_gdf_centroides_vazio():
    return gpd.GeoDataFrame(
        columns=["grupo_duplicidade_revisao", "qtd_registros", "canies_grupo",
                 "nomes_grupo", "status_grupo", "geometry"],
        geometry="geometry", crs=CRS_SAIDA_GPKG
    )


### 10.3.7 — Síntese de um grupo de revisão

Para cada grupo, a função calcula a média das coordenadas e concatena CANIEs, nomes e status únicos. O dicionário retornado representa uma única feição de centroide com seus atributos de contexto.

**Agregação.** Longitude e latitude médias determinam a posição sintética. Valores únicos de CANIE, nome e status são ordenados e unidos com ` | `, tornando o atributo reprodutível e legível. `qtd_registros` mantém o tamanho original do grupo.

**Limite cartográfico.** O ponto médio é um recurso de localização, não uma nova coordenada oficial de cavidade. Os pontos individuais e as linhas devem ser consultados para análise detalhada.

**Contrato técnico com o código.** Esta célula define a função `resumir_grupo_revisao(grupo, membros)`. Ela recebe `grupo`, `membros` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def resumir_grupo_revisao(grupo, membros):
    lon_media = membros["longitude_num"].mean()
    lat_media = membros["latitude_num"].mean()
    canies = sorted(membros["canie_txt"].dropna().astype(str).str.strip().unique())
    nomes = sorted(membros[COL_NOME].dropna().astype(str).str.strip().unique())
    status = sorted(membros["status_tratamento"].dropna().astype(str).str.strip().unique())
    return {
        "grupo_duplicidade_revisao": grupo, "qtd_registros": len(membros),
        "canies_grupo": " | ".join(canies), "nomes_grupo": " | ".join(nomes),
        "status_grupo": " | ".join(status), "longitude_media": lon_media,
        "latitude_media": lat_media, "geometry": Point(lon_media, lat_media)
    }


### 10.3.8 — Construção da camada de centroides

A função filtra grupos e coordenadas válidas, aplica a síntese anterior a cada agrupamento e converte os resultados em GeoDataFrame. Grupos ausentes ou vazios retornam o esquema vazio padronizado.

**Seleção.** Somente códigos de grupo não vazios e linhas com ambas as coordenadas entram no agrupamento. Cada conjunto é enviado a `resumir_grupo_revisao`, e os dicionários retornados formam as feições do GeoDataFrame.

**Saída.** A camada usa o mesmo CRS das demais exportações e passa pela preparação padronizada. A ausência de dados produz o esquema vazio definido anteriormente, sem interromper a Etapa 10.

**Contrato técnico com o código.** Esta célula define a função `criar_centroides_grupos(df_base)`. Ela recebe `df_base` e devolve um resultado ao chamador. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def criar_centroides_grupos(df_base):
    if "grupo_duplicidade_revisao" not in df_base.columns:
        return criar_gdf_centroides_vazio()
    df_temp = df_base[
        (df_base["grupo_duplicidade_revisao"] != "")
        & df_base["longitude_num"].notna()
        & df_base["latitude_num"].notna()
    ].copy()
    if df_temp.empty:
        return criar_gdf_centroides_vazio()
    registros = [
        resumir_grupo_revisao(grupo, membros)
        for grupo, membros in df_temp.groupby("grupo_duplicidade_revisao")
    ]
    gdf = gpd.GeoDataFrame(registros, geometry="geometry", crs=CRS_SAIDA_GPKG)
    return preparar_colunas_para_gpkg(gdf)


### 10.3.9 — Escrita condicional de uma camada

Camadas vazias são anunciadas e ignoradas. Camadas com feições são anexadas ao GeoPackage e têm sua quantidade registrada no output, permitindo conferir a exportação.

**Comportamento.** O teste de vazio evita criar camadas sem feições e registra essa decisão no output do notebook. Para uma camada válida, `to_file` escreve no caminho informado com o driver `GPKG`; chamadas sucessivas acrescentam camadas de nomes diferentes ao mesmo arquivo.

**Auditoria.** A mensagem final informa o nome e a contagem gravada, permitindo comparar a exportação espacial com as tabelas correspondentes.

**Contrato técnico com o código.** Esta célula define a função `escrever_camada_gpkg(gdf, caminho, nome_camada)`. Ela recebe `gdf`, `caminho`, `nome_camada` e atua por efeito sobre os objetos manipulados. A definição apenas registra a lógica na memória; o processamento efetivo ocorre quando a função é chamada nas células posteriores.


In [ ]:
def escrever_camada_gpkg(gdf, caminho, nome_camada):
    if gdf.empty:
        print(f"Camada ignorada, sem registros: {nome_camada}")
        return
    gdf.to_file(caminho, layer=nome_camada, driver="GPKG")
    print(f"Camada gravada: {nome_camada} — {len(gdf):,} registros")


## 10.4 — Criação das camadas pontuais

São construídas quatro visões pontuais: base completa com status, base consolidada, registros removidos e registros ainda pendentes de revisão.

Cada camada é filtrada para coordenadas válidas e recebe uma coluna que identifica sua origem. Isso permite comparar espacialmente o universo original, o resultado consolidado e os diferentes motivos de tratamento.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf_base_com_status`, `gdf_base_consolidada`, `df_registros_removidos`, `gdf_registros_removidos`, `df_registros_revisao_manual`, `gdf_registros_revisao_manual`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf_base_com_status = criar_gdf_pontos(
    df_base=df,
    nome_origem="base_com_status"
)

gdf_base_consolidada = criar_gdf_pontos(
    df_base=df_consolidado,
    nome_origem="base_consolidada"
)

df_registros_removidos = df[
    df["status_tratamento"].isin([
        "duplicado_remover",
        "duplicado_remover_manual"
    ])
].copy()

gdf_registros_removidos = criar_gdf_pontos(
    df_base=df_registros_removidos,
    nome_origem="registros_removidos"
)

df_registros_revisao_manual = df[
    df["status_tratamento"] == "revisar_manual"
].copy()

gdf_registros_revisao_manual = criar_gdf_pontos(
    df_base=df_registros_revisao_manual,
    nome_origem="registros_revisao_manual"
)


## 10.5 — Criação das camadas lineares de pares

As linhas conectam as coordenadas dos registros A e B de cada par. O script cria uma camada geral de pares suspeitos e subconjuntos para pares removidos, pendentes de revisão e preservados manualmente.

A seleção usa a coluna de decisão definida em 10.2, assegurando que as geometrias reflitam a mesma decisão empregada na consolidação.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


## 10.5.1 — Linha para cada par espacializável

A camada geral representa todos os pares, antes de separá-los por decisão.

**Entrada.** `df_pares_espacial` é a versão consolidada dos pares escolhida na Etapa 10.2. A função exige coordenadas válidas dos dois registros e preserva os atributos disponíveis, inclusive classe, distância e decisão.

**Finalidade.** Esta camada geral representa o universo espacializável de relações. Ela serve como referência antes dos filtros temáticos aplicados nas duas células seguintes.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf_pares_suspeitos_linhas`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf_pares_suspeitos_linhas = criar_linhas_pares(
    df_pares_base=df_pares_espacial,
    nome_origem="pares_suspeitos_linhas"
)


## 10.5.2 — Separação por decisão final

Pares são divididos em removidos, revisão e preservação manual; ausência da coluna gera subconjuntos vazios.

**Categorias.** Removidos incluem duplicidade automática e confirmação manual de mesma cavidade; revisão contém decisões ainda pendentes; e preservação manual contém pares declarados como cavidades diferentes. Cada filtro usa `.copy()` para manter os subconjuntos independentes.

**Tolerância de esquema.** Se a coluna de decisão não existir, são criados DataFrames vazios. Isso evita classificar pares por suposição e permite que a escrita condicional ignore as camadas sem dados.

**Contrato técnico com o código.** O bloco cria ou atualiza `df_pares_removidos`, `df_pares_revisao_manual`, `df_pares_nao_remover_manual`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
if COL_DECISAO_ESPACIAL in df_pares_espacial.columns:
    df_pares_removidos = df_pares_espacial[
        df_pares_espacial[COL_DECISAO_ESPACIAL].isin([
            "duplicidade_automatica",
            "duplicidade_manual_mesma_cavidade"
        ])
    ].copy()

    df_pares_revisao_manual = df_pares_espacial[
        df_pares_espacial[COL_DECISAO_ESPACIAL] == "revisar_manual"
    ].copy()

    df_pares_nao_remover_manual = df_pares_espacial[
        df_pares_espacial[COL_DECISAO_ESPACIAL] == "nao_remover_manual"
    ].copy()
else:
    df_pares_removidos = pd.DataFrame()
    df_pares_revisao_manual = pd.DataFrame()
    df_pares_nao_remover_manual = pd.DataFrame()


## 10.5.3 — Camadas lineares temáticas

Cada subconjunto é convertido em uma camada própria para inspeção cartográfica.

**Conversão.** Cada subconjunto da célula anterior é passado separadamente a `criar_linhas_pares`, recebendo um valor próprio de `origem_camada`. A geometria mantém a direção A–B apenas como representação; ela não implica precedência entre os registros.

**Resultado.** São produzidos três GeoDataFrames temáticos que, junto à camada geral, permitem ligar visualmente cada relação à decisão que recebeu.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf_pares_removidos_linhas`, `gdf_pares_revisao_manual_linhas`, `gdf_pares_nao_remover_manual_linhas`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf_pares_removidos_linhas = criar_linhas_pares(
    df_pares_base=df_pares_removidos,
    nome_origem="pares_removidos_linhas"
)

gdf_pares_revisao_manual_linhas = criar_linhas_pares(
    df_pares_base=df_pares_revisao_manual,
    nome_origem="pares_revisao_manual_linhas"
)

gdf_pares_nao_remover_manual_linhas = criar_linhas_pares(
    df_pares_base=df_pares_nao_remover_manual,
    nome_origem="pares_nao_remover_manual_linhas"
)


## 10.6 — Centroides dos grupos de revisão

Para cada `grupo_duplicidade_revisao`, é criado um ponto médio acompanhado da quantidade de registros, lista de CANIEs, nomes e status.

Essa camada é uma visão sintética para localizar rapidamente concentrações de casos relacionados, sem substituir os pontos e linhas detalhados.

**Dependências.** A função usa `df`, que já contém grupos e status atribuídos nas Etapas 4 e 6. Somente grupos não vazios com coordenadas válidas geram pontos; os atributos agregados preservam o contexto necessário para localizar seus membros.

**Interpretação correta.** Como o centroide é calculado pela média simples das coordenadas, ele resume a distribuição dos cadastros e não substitui uma localização validada da cavidade.

**Contrato técnico com o código.** O bloco cria ou atualiza `gdf_centroides_grupos_revisao`. Ele utiliza o estado preparado nas células anteriores e deixa esses objetos disponíveis em memória para as verificações, decisões ou exportações subsequentes.


In [ ]:
gdf_centroides_grupos_revisao = criar_centroides_grupos(
    df_base=df
)


## 10.7 — Escrita das camadas no arquivo GeoPackage

Cada GeoDataFrame preparado é enviado à função `escrever_camada_gpkg`. A função grava somente camadas com registros e informa no output se uma camada foi criada ou ignorada.

Todas as camadas ficam dentro do mesmo arquivo `.gpkg`, facilitando distribuição, auditoria e abertura conjunta em um sistema de informações geográficas.

**Natureza desta célula.** Esta é uma seção de orientação e não possui código próprio de forma intencional. Ela organiza o raciocínio, apresenta o contexto ou sintetiza resultados; a execução começa no primeiro subtópico que contém uma célula de código.


## 10.7.1 — Bases e registros pontuais

São gravadas as visões completa, consolidada, removida e pendente.

**Camadas gravadas.** `base_com_status` representa todos os registros espacializáveis; `base_consolidada`, os preservados; `registros_removidos`, as exclusões; e `registros_revisao_manual`, as pendências. Cada chamada usa o mesmo GeoPackage e um nome de camada exclusivo.

**Conferência.** A função de escrita informa se a camada foi gravada e quantas feições contém. Diferenças em relação às tabelas podem ocorrer apenas por ausência de coordenadas válidas.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
escrever_camada_gpkg(
    gdf_base_com_status,
    caminho_geopackage,
    "base_com_status"
)

escrever_camada_gpkg(
    gdf_base_consolidada,
    caminho_geopackage,
    "base_consolidada"
)

escrever_camada_gpkg(
    gdf_registros_removidos,
    caminho_geopackage,
    "registros_removidos"
)

escrever_camada_gpkg(
    gdf_registros_revisao_manual,
    caminho_geopackage,
    "registros_revisao_manual"
)


## 10.7.2 — Linhas dos pares

As quatro camadas lineares preservam relações e decisões distintas.

**Camadas gravadas.** A primeira contém todos os pares espacializáveis; as demais isolam relações que levaram a remoção, permaneceram em revisão ou foram preservadas manualmente. Os atributos do par acompanham cada linha para consulta no SIG.

**Relação com a auditoria tabular.** Essas camadas representam espacialmente as mesmas decisões exportadas no Excel da Etapa 9, mas podem conter menos linhas quando algum membro do par não possui coordenadas válidas.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
escrever_camada_gpkg(
    gdf_pares_suspeitos_linhas,
    caminho_geopackage,
    "pares_suspeitos_linhas"
)

escrever_camada_gpkg(
    gdf_pares_removidos_linhas,
    caminho_geopackage,
    "pares_removidos_linhas"
)

escrever_camada_gpkg(
    gdf_pares_revisao_manual_linhas,
    caminho_geopackage,
    "pares_revisao_manual_linhas"
)

escrever_camada_gpkg(
    gdf_pares_nao_remover_manual_linhas,
    caminho_geopackage,
    "pares_nao_remover_manual_linhas"
)


## 10.7.3 — Centroides dos grupos

A última chamada grava a síntese espacial dos grupos ampliados.

**Conteúdo.** `gdf_centroides_grupos_revisao` possui uma feição por grupo espacializável e atributos agregados de quantidade, CANIE, nome e status. A escrita condicional ignora o produto se nenhum grupo puder ser representado.

**Fechamento do conjunto espacial.** Depois desta chamada, o GeoPackage reúne visões de registros, relações e agrupamentos. A célula de resumo seguinte apresenta o caminho do arquivo final.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
escrever_camada_gpkg(
    gdf_centroides_grupos_revisao,
    caminho_geopackage,
    "centroides_grupos_revisao"
)


## 10.8 — Resumo final das feições geradas

O último bloco confirma a conclusão da etapa, mostra o caminho do GeoPackage e informa a quantidade de feições em cada camada.

Essas contagens devem ser confrontadas com as tabelas das Etapas 8 e 9. Por exemplo, `base_consolidada` deve ter a mesma quantidade de registros espaciais válidos correspondente à base consolidada tabular.

**Contrato técnico com o código.** O bloco executa uma ação sobre objetos já preparados — como validação, apresentação ou gravação — sem definir uma nova interface pública. Seu resultado deve ser interpretado pelo efeito observado no estado, na mensagem exibida ou no arquivo produzido.


In [ ]:
print("\nEtapa 10 concluída.")
print(f"GeoPackage gerado em: {caminho_geopackage}")

print("\nResumo das camadas espaciais:")
print(f"- base_com_status: {len(gdf_base_com_status):,} feições")
print(f"- base_consolidada: {len(gdf_base_consolidada):,} feições")
print(f"- registros_removidos: {len(gdf_registros_removidos):,} feições")
print(f"- registros_revisao_manual: {len(gdf_registros_revisao_manual):,} feições")
print(f"- pares_suspeitos_linhas: {len(gdf_pares_suspeitos_linhas):,} feições")
print(f"- pares_removidos_linhas: {len(gdf_pares_removidos_linhas):,} feições")
print(f"- pares_revisao_manual_linhas: {len(gdf_pares_revisao_manual_linhas):,} feições")
print(f"- pares_nao_remover_manual_linhas: {len(gdf_pares_nao_remover_manual_linhas):,} feições")
print(f"- centroides_grupos_revisao: {len(gdf_centroides_grupos_revisao):,} feições")


## Síntese da Etapa 10

### Resultado produzido

O GeoPackage reúne camadas pontuais das bases e dos status, linhas que representam pares e uma camada de centroides dos grupos de revisão. Todas as camadas gravadas compartilham o CRS de saída e um esquema preparado para interoperabilidade.

### Verificações esperadas

- a mensagem final deve apresentar o caminho do GeoPackage e a quantidade de feições por camada;
- camadas vazias devem ser ignoradas explicitamente, sem interromper a exportação;
- diferenças entre contagens tabulares e espaciais devem ser explicáveis por coordenadas ausentes;
- centroides devem ser interpretados como síntese, não como coordenadas oficiais.

### Encerramento do fluxo

Ao final, `02_outputs` contém três famílias complementares de produtos: base consolidada, relatório tabular de auditoria e GeoPackage espacial. Juntas, elas permitem uso operacional, rastreabilidade das decisões e inspeção cartográfica.

**Natureza desta célula.** Esta síntese encerra o notebook e não executa código.
